# Day 27 — Out-of-Time Lexical-Instability Hypothesis Test

Test the frozen lexical-instability hypothesis on the future population.

In [ ]:
# Imports
import json
from pathlib import Path

In [ ]:
# Project paths
def find_project_root():
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "README.md").exists():
            return path
    raise RuntimeError(
        "Project root not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

def resolve_artifact_paths(value):
    """Rebase legacy project paths in loaded metadata; never rewrite artifacts.

    Keep all non-path values, model identifiers, and scientific settings intact.
    Support the original project name and the public repository name.
    """
    if isinstance(value, dict):
        return {key: resolve_artifact_paths(item) for key, item in value.items()}
    if isinstance(value, list):
        return [resolve_artifact_paths(item) for item in value]
    if isinstance(value, str):
        normalized = value.replace("\\", "/")
        for name in ("FinGPT_Project", "FinGPT-Temporal-Generalization"):
            marker = "/" + name + "/"
            if marker in normalized and "\n" not in normalized:
                # Only standalone filesystem references, never prose containing a path.
                prefix, relative = normalized.split(marker, 1)
                if prefix.startswith("/") or (len(prefix) > 1 and prefix[1] == ":"):
                    if not any(char.isspace() for char in relative):
                        return str(PROJECT_ROOT / relative)
    return value

def load_artifact_json(stream, *args, **kwargs):
    return resolve_artifact_paths(json.load(stream, *args, **kwargs))


In [ ]:
# Reproducibility settings
# Original seeds and deterministic settings remain at their original workflow
# positions below, including stage-specific RNG resets. No new seed is introduced.

**ColabFinGPT | Pre-Registered Out-of-Time Evaluation**

---

## Scientific Objective

Day 27 evaluates whether the **historical-only lexical-instability predictor frozen on Day 26** predicts classification error on the **untouched Day 24 future population (F1–F6)**.

No Day 27 result may be used to redefine the Day 26 vocabulary, token-instability scores, support threshold, tokenizer, or primary document-level exposure rule.

---

## Frozen Primary Hypothesis

\[
H_1:
\quad
\mathbb{E}[E_i \mid \mathrm{error}_i=1]
>
\mathbb{E}[E_i \mid \mathrm{error}_i=0]
\]

where \(E_i\) is the frozen historical lexical-instability exposure of future observation \(i\).

The primary estimand is

\[
\Delta_{\mathrm{primary}}
=
\operatorname{mean}(E_i\mid \mathrm{error}_i=1)
-
\operatorname{mean}(E_i\mid \mathrm{error}_i=0).
\]

### Null hypothesis

\[
H_0:
\quad
\text{There is no systematic positive association between historical lexical-instability exposure and future classification error.}
\]

---

## Frozen Primary Exposure

For future document \(i\), let \(V_i\) denote the eligible Day 26 unigram/bigram vocabulary features present in the document.

\[
E_i
=
\frac{1}{|V_i|}
\sum_{v\in V_i} I(v),
\]

where

\[
I(v)=\max_c SD_t[A(v,c,t)].
\]

Primary rules:

- use the frozen Day 26 **df ≥ 20** vocabulary;
- use **binary feature presence**;
- repeated occurrences do not increase exposure;
- unknown or Day 26-ineligible features are ignored;
- if no eligible feature is present, exposure is missing;
- **df ≥ 50 is secondary sensitivity analysis only**.

---

## Pre-Registered Primary Inference

Primary bootstrap:

- observation-level resampling;
- **1,000 bootstrap replicates**;
- **seed = 42**;
- two-sided **95% percentile bootstrap confidence interval**.

### Scientific decision gate

**Gate A — H1 supported**

\[
\Delta_{\mathrm{primary}}>0
\]

and the entire 95% bootstrap CI lies above zero.

**Gate B — Directionally consistent but inconclusive**

\[
\Delta_{\mathrm{primary}}>0
\]

but the 95% CI crosses zero.

**Gate C — Primary H1 fails**

\[
\Delta_{\mathrm{primary}}\leq 0
\]

or there is no meaningful positive effect under the frozen primary definition.

Secondary F3, period, source, class-conditioned, df≥50, or regression analyses **cannot override the primary all-future gate**.

---

## Scientific Firewall

Before the primary result is frozen, Day 27 must not:

1. retrain or fine-tune the model;
2. perform new full-model inference;
3. change the Day 24 predictions;
4. refit vocabulary on future text;
5. change Day 26 token instability \(I(v)\);
6. change the historical windows or support threshold;
7. try alternative primary exposure definitions;
8. redefine the primary analysis based on F3;
9. inspect secondary analyses before saving the primary estimate and CI;
10. use secondary results to rescue a failed primary hypothesis.

Day 27 is therefore a **genuine out-of-time falsification test** of the predictor frozen on Day 26.

---

**H1 status entering Day 27: UNTESTED**

**Primary outcome access: LOCKED until upstream identity, future population, and exposure artifacts are verified and frozen.**

In [3]:
from pathlib import Path
import json
import hashlib
import pandas as pd
import numpy as np
from datetime import datetime

# =============================================================================
# Day 27 — Canonical project paths
# =============================================================================

# PROJECT_ROOT is discovered in the project-path setup cell.

RESULTS_ROOT = PROJECT_ROOT / "results"
NOTEBOOKS_ROOT = PROJECT_ROOT / "notebooks"

DAY24_DIR = RESULTS_ROOT / "day24"
DAY26_DIR = RESULTS_ROOT / "day26"
DAY27_DIR = RESULTS_ROOT / "day27"

DAY27_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOKS_ROOT.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# Frozen Day 24 inputs
# -----------------------------------------------------------------------------

DAY24_FUTURE_PREDICTIONS = (
    DAY24_DIR / "optimized_future_predictions.csv"
)

DAY24_PREDICTION_MANIFEST = (
    DAY24_DIR / "optimized_future_prediction_manifest.json"
)

# Day 24 completion checkpoint may be used if present in the frozen pipeline.
DAY24_COMPLETION_CHECKPOINT = (
    DAY24_DIR / "day24_completion_checkpoint.json"
)

# -----------------------------------------------------------------------------
# Frozen Day 26 inputs
# -----------------------------------------------------------------------------

DAY26_TOKEN_RANKING = (
    DAY26_DIR / "token_instability_rank.csv"
)

DAY26_HANDOFF_MANIFEST = (
    DAY26_DIR / "day26_day27_handoff_manifest.json"
)

DAY26_COMPLETION_CHECKPOINT = (
    DAY26_DIR / "day26_completion_checkpoint.json"
)

# -----------------------------------------------------------------------------
# Day 27 canonical outputs
# -----------------------------------------------------------------------------

DAY27_ANALYSIS_MANIFEST = (
    DAY27_DIR / "day27_analysis_manifest.json"
)

DAY27_FUTURE_POPULATION_MANIFEST = (
    DAY27_DIR / "day27_future_population_manifest.json"
)

DAY27_EXPOSURE_CSV = (
    DAY27_DIR / "future_lexical_instability_exposure.csv"
)

DAY27_EXPOSURE_QC_CSV = (
    DAY27_DIR / "future_exposure_qc.csv"
)

DAY27_PRIMARY_TEST_JSON = (
    DAY27_DIR / "day27_primary_hypothesis_test.json"
)

DAY27_BOOTSTRAP_CSV = (
    DAY27_DIR / "day27_primary_bootstrap_replicates.csv"
)

DAY27_PERIOD_CONTRASTS_CSV = (
    DAY27_DIR / "future_period_exposure_contrasts.csv"
)

DAY27_F3_CONTRAST_CSV = (
    DAY27_DIR / "f3_exposure_contrast.csv"
)

DAY27_F3_POSITIVE_CSV = (
    DAY27_DIR / "f3_positive_posneg_vs_pospos_exposure.csv"
)

DAY27_WITHIN_SOURCE_CSV = (
    DAY27_DIR / "within_source_exposure_contrasts.csv"
)

DAY27_CLASS_CONDITIONED_CSV = (
    DAY27_DIR / "class_conditioned_exposure_contrasts.csv"
)

DAY27_DF50_SENSITIVITY_JSON = (
    DAY27_DIR / "df50_support_sensitivity_contrast.json"
)

DAY27_LOGISTIC_SECONDARY_JSON = (
    DAY27_DIR / "day27_logistic_secondary.json"
)

DAY27_EVIDENCE_TABLE_CSV = (
    DAY27_DIR / "day27_evidence_table.csv"
)

DAY27_FINAL_SUMMARY_JSON = (
    DAY27_DIR / "day27_final_scientific_summary.json"
)

DAY27_CONCLUSION_TXT = (
    DAY27_DIR / "day27_scientific_conclusion.txt"
)

DAY27_FINAL_AUDIT_CSV = (
    DAY27_DIR / "day27_final_artifact_audit.csv"
)

DAY27_COMPLETION_CHECKPOINT = (
    DAY27_DIR / "day27_completion_checkpoint.json"
)

DAY27_CANONICAL_NOTEBOOK = (
    NOTEBOOKS_ROOT / "day27_out_of_time_lexical_instability_hypothesis_test.ipynb"
)

# =============================================================================
# Helper functions
# =============================================================================

def sha256_file(path, chunk_size=1024 * 1024):
    """
    Return SHA256 hex digest for a file.
    """
    path = Path(path)

    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


def load_json(path):
    """
    Load JSON from disk.
    """
    path = Path(path)

    with path.open("r", encoding="utf-8") as f:
        return load_artifact_json(f)


def save_json(obj, path):
    """
    Save JSON in deterministic human-readable form.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        json.dump(
            obj,
            f,
            indent=2,
            ensure_ascii=False,
            sort_keys=True,
            default=str
        )


def file_info(path):
    """
    Return basic immutable file identity information.
    """
    path = Path(path)

    return {
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else None,
        "sha256": sha256_file(path) if path.exists() else None,
    }


def print_check(name, condition):
    """
    Standardized PASS / FAIL display.
    """
    status = "PASS" if bool(condition) else "FAIL"
    print(f"{status:<6} — {name}")
    return bool(condition)


# =============================================================================
# Day 27 frozen protocol constants
# =============================================================================

DAY27_PROTOCOL = {
    "day": 27,
    "analysis_name": "out_of_time_lexical_instability_hypothesis_test",

    "expected_future_n": 20002,

    "future_periods": [
        "F1",
        "F2",
        "F3",
        "F4",
        "F5",
        "F6",
    ],

    "primary_support_threshold": 20,
    "secondary_support_threshold": 50,

    "bootstrap_replicates": 1000,
    "bootstrap_seed": 42,

    "primary_exposure_rule": "mean_token_instability_binary_presence",

    "primary_estimand": (
        "mean(exposure | error=1) - mean(exposure | error=0)"
    ),

    "primary_gate": (
        "H1 supported iff delta_primary > 0 "
        "and the full 95% percentile bootstrap CI is > 0"
    ),
}

print("=" * 120)
print("DAY 27 — CANONICAL PATHS AND HELPERS INITIALIZED")
print("=" * 120)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DAY24_DIR    : {DAY24_DIR}")
print(f"DAY26_DIR    : {DAY26_DIR}")
print(f"DAY27_DIR    : {DAY27_DIR}")
print(f"NOTEBOOK     : {DAY27_CANONICAL_NOTEBOOK}")

print("-" * 120)

print_check("project_root_exists", PROJECT_ROOT.exists())
print_check("results_root_exists", RESULTS_ROOT.exists())
print_check("notebooks_root_exists", NOTEBOOKS_ROOT.exists())
print_check("day27_output_dir_exists", DAY27_DIR.exists())

print("-" * 120)
print("DAY 27 PROTOCOL CONSTANTS")
print("-" * 120)

for key, value in DAY27_PROTOCOL.items():
    print(f"{key:<30}: {value}")

print("=" * 120)

DAY 27 — CANONICAL PATHS AND HELPERS INITIALIZED
PROJECT_ROOT : /content/drive/MyDrive/FinGPT_Project
DAY24_DIR    : /content/drive/MyDrive/FinGPT_Project/results/day24
DAY26_DIR    : /content/drive/MyDrive/FinGPT_Project/results/day26
DAY27_DIR    : /content/drive/MyDrive/FinGPT_Project/results/day27
NOTEBOOK     : /content/drive/MyDrive/FinGPT_Project/notebooks/day27_out_of_time_lexical_instability_hypothesis_test.ipynb
------------------------------------------------------------------------------------------------------------------------
PASS   — project_root_exists
PASS   — results_root_exists
PASS   — notebooks_root_exists
PASS   — day27_output_dir_exists
------------------------------------------------------------------------------------------------------------------------
DAY 27 PROTOCOL CONSTANTS
------------------------------------------------------------------------------------------------------------------------
day                           : 27
analysis_name               

In [5]:
# =============================================================================
# Day 27 — Frozen Day24 / Day26 upstream identity audit
# Canonical retained cell
# =============================================================================

print("=" * 120)
print("DAY 27 — FROZEN UPSTREAM IDENTITY AUDIT")
print("=" * 120)

# -----------------------------------------------------------------------------
# Load frozen upstream metadata
# -----------------------------------------------------------------------------

day24_prediction_manifest = load_json(DAY24_PREDICTION_MANIFEST)
day24_completion = load_json(DAY24_COMPLETION_CHECKPOINT)

day26_handoff = load_json(DAY26_HANDOFF_MANIFEST)
day26_completion = load_json(DAY26_COMPLETION_CHECKPOINT)

# -----------------------------------------------------------------------------
# Current immutable file identities
# -----------------------------------------------------------------------------

current_day24_prediction_sha256 = sha256_file(DAY24_FUTURE_PREDICTIONS)
current_day26_token_rank_sha256 = sha256_file(DAY26_TOKEN_RANKING)
current_day26_handoff_sha256 = sha256_file(DAY26_HANDOFF_MANIFEST)

# -----------------------------------------------------------------------------
# Frozen expected identities
# -----------------------------------------------------------------------------

expected_day24_prediction_sha256_manifest = (
    day24_prediction_manifest["prediction_artifact"]["sha256"]
)

expected_day24_prediction_sha256_completion = (
    day24_completion["future_prediction_artifact"]["sha256"]
)

expected_day24_prediction_sha256_day26_handoff = (
    day26_handoff["day24_future_prediction_identity"]["sha256"]
)

expected_day24_prediction_sha256_day26_completion = (
    day26_completion["upstream_identity"]["day24_future_prediction_sha256"]
)

expected_day26_token_rank_sha256_handoff = (
    day26_handoff["instability_lookup"]["primary_ranking_sha256"]
)

expected_day26_token_rank_sha256_completion = (
    day26_completion["core_artifact_hashes"]["token_instability_rank"]
)

expected_day26_handoff_sha256_completion = (
    day26_completion["core_artifact_hashes"]["day27_handoff"]
)

# -----------------------------------------------------------------------------
# Frozen protocol identities
# -----------------------------------------------------------------------------

frozen_future_n_day24 = (
    day24_prediction_manifest["future_population"]["n"]
)

frozen_future_n_day26 = (
    day26_handoff["day24_future_prediction_identity"]["n"]
)

frozen_token_n_handoff = (
    day26_handoff["instability_lookup"]["n_tokens"]
)

frozen_token_n_completion = (
    day26_completion["primary_scientific_endpoint"]["eligible_vocabulary_n"]
)

frozen_bootstrap_replicates = (
    day26_handoff["day27_primary_test"]["bootstrap_replicates"]
)

frozen_bootstrap_seed = (
    day26_handoff["day27_primary_test"]["bootstrap_seed"]
)

frozen_support_threshold = (
    day26_handoff["vocabulary"]["minimum_historical_df"]
)

# -----------------------------------------------------------------------------
# Integrity checks
# -----------------------------------------------------------------------------

checks = {}

print("-" * 120)
print("FILE EXISTENCE")
print("-" * 120)

checks["day24_prediction_manifest_exists"] = print_check(
    "day24_prediction_manifest_exists",
    DAY24_PREDICTION_MANIFEST.exists()
)

checks["day24_completion_exists"] = print_check(
    "day24_completion_exists",
    DAY24_COMPLETION_CHECKPOINT.exists()
)

checks["day24_future_predictions_exists"] = print_check(
    "day24_future_predictions_exists",
    DAY24_FUTURE_PREDICTIONS.exists()
)

checks["day26_handoff_exists"] = print_check(
    "day26_handoff_exists",
    DAY26_HANDOFF_MANIFEST.exists()
)

checks["day26_completion_exists"] = print_check(
    "day26_completion_exists",
    DAY26_COMPLETION_CHECKPOINT.exists()
)

checks["day26_token_ranking_exists"] = print_check(
    "day26_token_ranking_exists",
    DAY26_TOKEN_RANKING.exists()
)

print("-" * 120)
print("DAY 24 FROZEN PREDICTION IDENTITY")
print("-" * 120)

checks["day24_hash_matches_manifest"] = print_check(
    "day24_hash_matches_manifest",
    current_day24_prediction_sha256
    == expected_day24_prediction_sha256_manifest
)

checks["day24_hash_matches_day24_completion"] = print_check(
    "day24_hash_matches_day24_completion",
    current_day24_prediction_sha256
    == expected_day24_prediction_sha256_completion
)

checks["day24_hash_matches_day26_handoff"] = print_check(
    "day24_hash_matches_day26_handoff",
    current_day24_prediction_sha256
    == expected_day24_prediction_sha256_day26_handoff
)

checks["day24_hash_matches_day26_completion"] = print_check(
    "day24_hash_matches_day26_completion",
    current_day24_prediction_sha256
    == expected_day24_prediction_sha256_day26_completion
)

checks["day24_manifest_status_frozen"] = print_check(
    "day24_manifest_status_frozen",
    day24_prediction_manifest["status"] == "COMPLETE_AND_FROZEN"
)

checks["day24_future_predictions_immutable"] = print_check(
    "day24_future_predictions_immutable",
    day24_prediction_manifest["future_predictions_immutable"] is True
)

checks["day24_prediction_integrity_pass"] = print_check(
    "day24_prediction_integrity_pass",
    day24_prediction_manifest["prediction_integrity"]["all_checks_pass"] is True
)

checks["day24_zero_invalid_predictions"] = print_check(
    "day24_zero_invalid_predictions",
    day24_prediction_manifest["prediction_integrity"]["invalid_predictions"] == 0
)

checks["day24_complete"] = print_check(
    "day24_complete",
    day24_completion["day24_complete"] is True
)

checks["day24_completion_status_pass"] = print_check(
    "day24_completion_status_pass",
    day24_completion["status"] == "PASS"
)

print("-" * 120)
print("DAY 26 FROZEN PREDICTOR IDENTITY")
print("-" * 120)

checks["day26_token_rank_hash_matches_handoff"] = print_check(
    "day26_token_rank_hash_matches_handoff",
    current_day26_token_rank_sha256
    == expected_day26_token_rank_sha256_handoff
)

checks["day26_token_rank_hash_matches_completion"] = print_check(
    "day26_token_rank_hash_matches_completion",
    current_day26_token_rank_sha256
    == expected_day26_token_rank_sha256_completion
)

checks["day26_handoff_hash_matches_completion"] = print_check(
    "day26_handoff_hash_matches_completion",
    current_day26_handoff_sha256
    == expected_day26_handoff_sha256_completion
)

checks["day26_handoff_frozen_for_day27"] = print_check(
    "day26_handoff_frozen_for_day27",
    day26_handoff["status"] == "FROZEN_FOR_DAY27"
)

checks["day26_complete"] = print_check(
    "day26_complete",
    day26_completion["day26_complete"] is True
)

checks["day26_completion_status_pass"] = print_check(
    "day26_completion_status_pass",
    day26_completion["status"] == "PASS"
)

checks["day26_H1_untested"] = print_check(
    "day26_H1_untested",
    day26_completion["hypothesis_status"]["H1_tested"] is False
)

checks["day26_no_future_error_predictor_leakage"] = print_check(
    "day26_no_future_error_predictor_leakage",
    day26_completion["scientific_firewall"][
        "future_errors_used_to_construct_predictor"
    ] is False
)

checks["day26_no_future_label_predictor_leakage"] = print_check(
    "day26_no_future_label_predictor_leakage",
    day26_completion["scientific_firewall"][
        "future_labels_used_to_construct_predictor"
    ] is False
)

checks["day26_no_new_model_training"] = print_check(
    "day26_no_new_model_training",
    day26_handoff["scientific_firewall"][
        "new_model_training_performed"
    ] is False
)

checks["day26_no_new_full_model_inference"] = print_check(
    "day26_no_new_full_model_inference",
    day26_handoff["scientific_firewall"][
        "new_full_model_inference_performed"
    ] is False
)

print("-" * 120)
print("FROZEN DAY 27 PROTOCOL CONSISTENCY")
print("-" * 120)

checks["future_n_consistent"] = print_check(
    "future_n_consistent",
    frozen_future_n_day24
    == frozen_future_n_day26
    == DAY27_PROTOCOL["expected_future_n"]
    == 20002
)

checks["eligible_token_n_consistent"] = print_check(
    "eligible_token_n_consistent",
    frozen_token_n_handoff
    == frozen_token_n_completion
    == 3240
)

checks["support_threshold_consistent"] = print_check(
    "support_threshold_consistent",
    frozen_support_threshold
    == DAY27_PROTOCOL["primary_support_threshold"]
    == 20
)

checks["bootstrap_replicates_consistent"] = print_check(
    "bootstrap_replicates_consistent",
    frozen_bootstrap_replicates
    == DAY27_PROTOCOL["bootstrap_replicates"]
    == 1000
)

checks["bootstrap_seed_consistent"] = print_check(
    "bootstrap_seed_consistent",
    frozen_bootstrap_seed
    == DAY27_PROTOCOL["bootstrap_seed"]
    == 42
)

checks["binary_presence_frozen"] = print_check(
    "binary_presence_frozen",
    day26_handoff["tokenizer"]["binary_document_presence"] is True
)

checks["zero_eligible_exposure_missing"] = print_check(
    "zero_eligible_exposure_missing",
    day26_handoff["future_observation_exposure"][
        "zero_eligible_tokens"
    ].startswith("Exposure is missing")
)

# -----------------------------------------------------------------------------
# Final gate
# -----------------------------------------------------------------------------

all_checks_pass = all(checks.values())

print("-" * 120)
print("FROZEN IDENTITIES")
print("-" * 120)

print(f"Day24 future prediction SHA256 : {current_day24_prediction_sha256}")
print(f"Day26 token ranking SHA256     : {current_day26_token_rank_sha256}")
print(f"Day26 handoff SHA256           : {current_day26_handoff_sha256}")
print(f"Frozen future n                : {frozen_future_n_day24:,}")
print(f"Frozen eligible vocabulary n   : {frozen_token_n_handoff:,}")
print(f"Primary df threshold           : {frozen_support_threshold}")
print(f"Bootstrap replicates           : {frozen_bootstrap_replicates}")
print(f"Bootstrap seed                 : {frozen_bootstrap_seed}")
print(f"H1 tested before Day27         : {day26_completion['hypothesis_status']['H1_tested']}")

print("-" * 120)

print_check(
    "UPSTREAM_IDENTITY_AUDIT_ALL_PASS",
    all_checks_pass
)

if not all_checks_pass:
    raise RuntimeError(
        "DAY 27 INTEGRITY STOP: frozen Day24/Day26 upstream identity audit failed."
    )

print("=" * 120)
print("DAY 27 STAGE 0 UPSTREAM IDENTITY: PASS")
print("SAFE TO FREEZE THE DAY 27 ANALYSIS MANIFEST")
print("=" * 120)

DAY 27 — FROZEN UPSTREAM IDENTITY AUDIT
------------------------------------------------------------------------------------------------------------------------
FILE EXISTENCE
------------------------------------------------------------------------------------------------------------------------
PASS   — day24_prediction_manifest_exists
PASS   — day24_completion_exists
PASS   — day24_future_predictions_exists
PASS   — day26_handoff_exists
PASS   — day26_completion_exists
PASS   — day26_token_ranking_exists
------------------------------------------------------------------------------------------------------------------------
DAY 24 FROZEN PREDICTION IDENTITY
------------------------------------------------------------------------------------------------------------------------
PASS   — day24_hash_matches_manifest
PASS   — day24_hash_matches_day24_completion
PASS   — day24_hash_matches_day26_handoff
PASS   — day24_hash_matches_day26_completion
PASS   — day24_manifest_status_frozen
PASS 

In [6]:
# =============================================================================
# Day 27 — Analysis manifest freeze
# Canonical retained cell
# =============================================================================

from datetime import datetime, timezone

print("=" * 120)
print("DAY 27 — ANALYSIS MANIFEST FREEZE")
print("=" * 120)

# -----------------------------------------------------------------------------
# Freeze the exact Day 27 analysis protocol before future outcome analysis
# -----------------------------------------------------------------------------

day27_analysis_manifest = {
    "day": 27,
    "artifact": "day27_analysis_manifest",
    "status": "FROZEN_BEFORE_PRIMARY_OUTCOME_ANALYSIS",

    "analysis_name": DAY27_PROTOCOL["analysis_name"],

    "created_utc": datetime.now(timezone.utc).isoformat(),

    # -------------------------------------------------------------------------
    # Scientific question
    # -------------------------------------------------------------------------
    "scientific_question": (
        "Does higher exposure to historically unstable lexical-label "
        "associations predict higher classification error in the untouched "
        "Day24 future F1-F6 population?"
    ),

    "hypotheses": {
        "H1": (
            "Mean historical lexical-instability exposure is higher among "
            "misclassified future observations than among correctly "
            "classified future observations."
        ),
        "H0": (
            "There is no systematic positive association between historical "
            "lexical-instability exposure and future classification error."
        ),
        "H1_status_entering_day27": "UNTESTED",
    },

    # -------------------------------------------------------------------------
    # Frozen primary estimand
    # -------------------------------------------------------------------------
    "primary_estimand": {
        "name": "delta_primary",
        "definition": (
            "mean(exposure | error=1) - mean(exposure | error=0)"
        ),
        "direction_of_support": "positive",
    },

    # -------------------------------------------------------------------------
    # Frozen primary exposure
    # -------------------------------------------------------------------------
    "primary_exposure": {
        "token_score_field": "token_instability",
        "token_score_definition": (
            "I(v) = max_c sample SD across historical windows of "
            "smoothed one-vs-rest lexical-label log-odds"
        ),
        "document_exposure_definition": (
            "Mean token_instability across eligible frozen historical "
            "vocabulary features present in the future observation"
        ),
        "binary_presence": True,
        "primary_minimum_historical_df": 20,
        "eligible_vocabulary_n": 3240,
        "unknown_or_ineligible_tokens": "ignored",
        "zero_eligible_tokens": "missing exposure; no imputation",
        "alternative_primary_aggregation_allowed": False,
    },

    # -------------------------------------------------------------------------
    # Frozen inference rule
    # -------------------------------------------------------------------------
    "primary_inference": {
        "bootstrap_level": "observation",
        "bootstrap_replicates": 1000,
        "bootstrap_seed": 42,
        "confidence_interval": "two-sided 95% percentile bootstrap CI",
        "support_rule": (
            "Gate A iff delta_primary > 0 and both CI bounds are > 0"
        ),
        "suggestive_rule": (
            "Gate B iff delta_primary > 0 but CI crosses 0"
        ),
        "failure_rule": (
            "Gate C iff delta_primary <= 0 or there is no meaningful "
            "positive effect"
        ),
    },

    # -------------------------------------------------------------------------
    # Frozen future population identity
    # -------------------------------------------------------------------------
    "future_population": {
        "source_artifact": str(DAY24_FUTURE_PREDICTIONS),
        "expected_n": 20002,
        "expected_periods": ["F1", "F2", "F3", "F4", "F5", "F6"],
        "prediction_sha256": current_day24_prediction_sha256,
        "predictions_immutable": True,
        "new_model_inference_allowed": False,
    },

    # -------------------------------------------------------------------------
    # Frozen Day26 predictor identity
    # -------------------------------------------------------------------------
    "day26_predictor": {
        "token_rank_artifact": str(DAY26_TOKEN_RANKING),
        "token_rank_sha256": current_day26_token_rank_sha256,
        "handoff_artifact": str(DAY26_HANDOFF_MANIFEST),
        "handoff_sha256": current_day26_handoff_sha256,
        "eligible_token_n": 3240,
        "minimum_historical_df": 20,
        "frozen_for_day27": True,
    },

    # -------------------------------------------------------------------------
    # Tokenizer / vectorizer interface frozen from Day26
    # -------------------------------------------------------------------------
    "tokenizer": {
        "implementation": day26_handoff["tokenizer"]["implementation"],
        "lowercase": day26_handoff["tokenizer"]["lowercase"],
        "ngram_range": day26_handoff["tokenizer"]["ngram_range"],
        "token_pattern": day26_handoff["tokenizer"]["token_pattern"],
        "stop_words": day26_handoff["tokenizer"]["stop_words"],
        "max_df_historical_fit": day26_handoff["tokenizer"][
            "max_df_historical_fit"
        ],
        "binary_document_presence": day26_handoff["tokenizer"][
            "binary_document_presence"
        ],
        "future_vocabulary_refit_allowed": False,
    },

    # -------------------------------------------------------------------------
    # Secondary analyses are frozen but cannot alter primary conclusion
    # -------------------------------------------------------------------------
    "secondary_analyses": {
        "future_period_F1_F6": True,
        "F3_specific": True,
        "F3_positive_P_to_N_vs_P_to_P": True,
        "within_source": True,
        "class_conditioned": True,
        "df50_sensitivity": True,
        "logistic_unadjusted": True,
        "logistic_adjusted": True,
        "may_override_primary_gate": False,
    },

    # -------------------------------------------------------------------------
    # Scientific firewall
    # -------------------------------------------------------------------------
    "scientific_firewall": {
        "model_training_allowed": False,
        "model_finetuning_allowed": False,
        "new_full_model_inference_allowed": False,
        "checkpoint_reselection_allowed": False,
        "prompt_modification_allowed": False,
        "prediction_modification_allowed": False,
        "future_vocabulary_refit_allowed": False,
        "token_score_redefinition_allowed": False,
        "support_threshold_redefinition_allowed": False,
        "primary_exposure_redefinition_allowed": False,
        "F3_based_primary_redefinition_allowed": False,
        "secondary_analysis_before_primary_freeze_allowed": False,
        "secondary_result_may_override_primary_gate": False,
    },

    # -------------------------------------------------------------------------
    # Stage ordering
    # -------------------------------------------------------------------------
    "execution_order": [
        "stage0_upstream_identity_audit",
        "stage1_future_population_freeze",
        "stage2_frozen_exposure_construction",
        "stage3_exposure_qc",
        "stage4_primary_hypothesis_test",
        "stage5_pre_specified_secondary_analyses",
        "stage6_evidence_integration",
        "stage7_scientific_closure",
    ],
}

# -----------------------------------------------------------------------------
# Save canonical manifest
# -----------------------------------------------------------------------------

save_json(
    day27_analysis_manifest,
    DAY27_ANALYSIS_MANIFEST
)

# -----------------------------------------------------------------------------
# Deterministic round-trip audit
# -----------------------------------------------------------------------------

reloaded_day27_manifest = load_json(
    DAY27_ANALYSIS_MANIFEST
)

manifest_roundtrip_exact = (
    day27_analysis_manifest == reloaded_day27_manifest
)

day27_analysis_manifest_sha256 = sha256_file(
    DAY27_ANALYSIS_MANIFEST
)

# -----------------------------------------------------------------------------
# Integrity checks
# -----------------------------------------------------------------------------

checks_manifest = {}

checks_manifest["manifest_exists"] = print_check(
    "manifest_exists",
    DAY27_ANALYSIS_MANIFEST.exists()
)

checks_manifest["manifest_nonempty"] = print_check(
    "manifest_nonempty",
    DAY27_ANALYSIS_MANIFEST.stat().st_size > 0
)

checks_manifest["manifest_roundtrip_exact"] = print_check(
    "manifest_roundtrip_exact",
    manifest_roundtrip_exact
)

checks_manifest["status_frozen"] = print_check(
    "status_frozen",
    reloaded_day27_manifest["status"]
    == "FROZEN_BEFORE_PRIMARY_OUTCOME_ANALYSIS"
)

checks_manifest["H1_still_untested"] = print_check(
    "H1_still_untested",
    reloaded_day27_manifest["hypotheses"][
        "H1_status_entering_day27"
    ] == "UNTESTED"
)

checks_manifest["future_n_frozen_20002"] = print_check(
    "future_n_frozen_20002",
    reloaded_day27_manifest["future_population"][
        "expected_n"
    ] == 20002
)

checks_manifest["future_hash_frozen"] = print_check(
    "future_hash_frozen",
    reloaded_day27_manifest["future_population"][
        "prediction_sha256"
    ] == current_day24_prediction_sha256
)

checks_manifest["day26_rank_hash_frozen"] = print_check(
    "day26_rank_hash_frozen",
    reloaded_day27_manifest["day26_predictor"][
        "token_rank_sha256"
    ] == current_day26_token_rank_sha256
)

checks_manifest["bootstrap_1000"] = print_check(
    "bootstrap_1000",
    reloaded_day27_manifest["primary_inference"][
        "bootstrap_replicates"
    ] == 1000
)

checks_manifest["bootstrap_seed_42"] = print_check(
    "bootstrap_seed_42",
    reloaded_day27_manifest["primary_inference"][
        "bootstrap_seed"
    ] == 42
)

checks_manifest["primary_df20"] = print_check(
    "primary_df20",
    reloaded_day27_manifest["primary_exposure"][
        "primary_minimum_historical_df"
    ] == 20
)

checks_manifest["secondary_cannot_override_primary"] = print_check(
    "secondary_cannot_override_primary",
    reloaded_day27_manifest["secondary_analyses"][
        "may_override_primary_gate"
    ] is False
)

all_manifest_checks_pass = all(
    checks_manifest.values()
)

print("-" * 120)
print("FROZEN ANALYSIS MANIFEST")
print("-" * 120)

print(f"Path   : {DAY27_ANALYSIS_MANIFEST}")
print(f"Size   : {DAY27_ANALYSIS_MANIFEST.stat().st_size:,} bytes")
print(f"SHA256 : {day27_analysis_manifest_sha256}")

print("-" * 120)

print_check(
    "DAY27_ANALYSIS_MANIFEST_ALL_PASS",
    all_manifest_checks_pass
)

if not all_manifest_checks_pass:
    raise RuntimeError(
        "DAY 27 INTEGRITY STOP: analysis manifest freeze failed."
    )

print("=" * 120)
print("DAY 27 ANALYSIS MANIFEST: FROZEN")
print("PRIMARY OUTCOME ANALYSIS HAS NOT STARTED")
print("READY — STAGE 1 FUTURE POPULATION FREEZE")
print("=" * 120)

DAY 27 — ANALYSIS MANIFEST FREEZE
PASS   — manifest_exists
PASS   — manifest_nonempty
PASS   — manifest_roundtrip_exact
PASS   — status_frozen
PASS   — H1_still_untested
PASS   — future_n_frozen_20002
PASS   — future_hash_frozen
PASS   — day26_rank_hash_frozen
PASS   — bootstrap_1000
PASS   — bootstrap_seed_42
PASS   — primary_df20
PASS   — secondary_cannot_override_primary
------------------------------------------------------------------------------------------------------------------------
FROZEN ANALYSIS MANIFEST
------------------------------------------------------------------------------------------------------------------------
Path   : /content/drive/MyDrive/FinGPT_Project/results/day27/day27_analysis_manifest.json
Size   : 4,632 bytes
SHA256 : 6da41aad254c5c713624234677993b853131f49e6b1073934d9c8640dd3ac97e
------------------------------------------------------------------------------------------------------------------------
PASS   — DAY27_ANALYSIS_MANIFEST_ALL_PASS
DAY 27 A

## Stage 1 — Future Population Freeze

The Day 27 protocol and upstream identities are now frozen.

This stage loads the immutable Day 24 future prediction artifact and reconstructs the future population **without changing any prediction**.

Required checks:

- exactly **20,002** observations;
- unique observation IDs;
- future periods restricted to **F1–F6**;
- frozen period counts must match the Day 24 manifest;
- valid true labels and Day 24 predictions;
- zero invalid predictions;
- no new model inference.

Correctness is defined deterministically as

\[
\mathrm{correct}_i
=
\mathbf{1}
\left[
y^{\mathrm{Day24}}_{\mathrm{pred},i}
=
y_{\mathrm{true},i}
\right],
\]

and

\[
\mathrm{error}_i
=
1-\mathrm{correct}_i.
\]

At this stage, correctness is reconstructed only to freeze the future population.

**No lexical-instability exposure is computed or compared with correctness yet.**

In [7]:
# =============================================================================
# Day 27 — Stage 1
# Future population load, validation, and freeze
# Canonical retained cell
# =============================================================================

print("=" * 120)
print("DAY 27 — STAGE 1 FUTURE POPULATION FREEZE")
print("=" * 120)

# -----------------------------------------------------------------------------
# Re-verify immutable Day24 prediction artifact before reading
# -----------------------------------------------------------------------------

current_future_sha256 = sha256_file(DAY24_FUTURE_PREDICTIONS)

print_check(
    "future_prediction_hash_still_matches_frozen_identity",
    current_future_sha256
    == day27_analysis_manifest["future_population"]["prediction_sha256"]
)

if (
    current_future_sha256
    != day27_analysis_manifest["future_population"]["prediction_sha256"]
):
    raise RuntimeError(
        "DAY 27 INTEGRITY STOP: Day24 future prediction artifact changed "
        "after the Day27 analysis manifest was frozen."
    )

# -----------------------------------------------------------------------------
# Load immutable Day24 future predictions
# -----------------------------------------------------------------------------

future_df = pd.read_csv(DAY24_FUTURE_PREDICTIONS)

print("-" * 120)
print("LOADED FUTURE ARTIFACT")
print("-" * 120)

print(f"Rows    : {len(future_df):,}")
print(f"Columns : {future_df.shape[1]}")
print()
print("COLUMN NAMES")
print(list(future_df.columns))

# -----------------------------------------------------------------------------
# Expected canonical schema
# -----------------------------------------------------------------------------

required_columns = [
    "observation_id",
    "period",
    "source",
    "text",
    "y_true",
    "y_pred_day24",
    "prediction_valid_day24",
]

missing_required_columns = [
    col for col in required_columns
    if col not in future_df.columns
]

print("-" * 120)
print("SCHEMA CHECK")
print("-" * 120)

schema_ok = print_check(
    "required_future_columns_present",
    len(missing_required_columns) == 0
)

if not schema_ok:
    print(f"Missing required columns: {missing_required_columns}")
    raise RuntimeError(
        "DAY 27 STOP: frozen Day24 future prediction schema does not match "
        "the expected Day27 interface."
    )

# -----------------------------------------------------------------------------
# Normalize only representation-level fields
# No predictions or labels are modified
# -----------------------------------------------------------------------------

future_df["observation_id"] = future_df["observation_id"].astype(str)

future_df["period"] = (
    future_df["period"]
    .astype(str)
    .str.strip()
)

future_df["source"] = future_df["source"].astype(str)

future_df["y_true"] = (
    future_df["y_true"]
    .astype(str)
    .str.strip()
    .str.lower()
)

future_df["y_pred_day24"] = (
    future_df["y_pred_day24"]
    .astype(str)
    .str.strip()
    .str.lower()
)

valid_labels = {
    "negative",
    "neutral",
    "positive",
}

expected_periods = set(
    day27_analysis_manifest["future_population"]["expected_periods"]
)

expected_window_counts = (
    day24_prediction_manifest["future_population"]["window_counts"]
)

# -----------------------------------------------------------------------------
# Deterministic correctness reconstruction
# -----------------------------------------------------------------------------

future_df["correct_day24"] = (
    future_df["y_pred_day24"] == future_df["y_true"]
)

future_df["error_day24"] = (
    ~future_df["correct_day24"]
).astype(int)

# -----------------------------------------------------------------------------
# Integrity checks
# -----------------------------------------------------------------------------

checks_stage1 = {}

print("-" * 120)
print("POPULATION INTEGRITY")
print("-" * 120)

checks_stage1["n_exact_20002"] = print_check(
    "n_exact_20002",
    len(future_df) == 20002
)

checks_stage1["observation_ids_nonmissing"] = print_check(
    "observation_ids_nonmissing",
    future_df["observation_id"].notna().all()
)

checks_stage1["observation_ids_unique"] = print_check(
    "observation_ids_unique",
    future_df["observation_id"].is_unique
)

checks_stage1["text_nonmissing"] = print_check(
    "text_nonmissing",
    future_df["text"].notna().all()
)

checks_stage1["period_nonmissing"] = print_check(
    "period_nonmissing",
    future_df["period"].notna().all()
)

checks_stage1["periods_exact_F1_F6"] = print_check(
    "periods_exact_F1_F6",
    set(future_df["period"].unique()) == expected_periods
)

checks_stage1["true_labels_valid"] = print_check(
    "true_labels_valid",
    set(future_df["y_true"].unique()).issubset(valid_labels)
)

checks_stage1["predicted_labels_valid"] = print_check(
    "predicted_labels_valid",
    set(future_df["y_pred_day24"].unique()).issubset(valid_labels)
)

# Handle bool / 0-1 representations robustly without altering artifact
prediction_valid_series = future_df["prediction_valid_day24"]

if prediction_valid_series.dtype == bool:
    prediction_valid_mask = prediction_valid_series
else:
    prediction_valid_mask = (
        prediction_valid_series
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1"])
    )

checks_stage1["all_predictions_valid"] = print_check(
    "all_predictions_valid",
    prediction_valid_mask.all()
)

checks_stage1["zero_invalid_predictions"] = print_check(
    "zero_invalid_predictions",
    (~prediction_valid_mask).sum() == 0
)

checks_stage1["error_binary"] = print_check(
    "error_binary",
    set(future_df["error_day24"].unique()).issubset({0, 1})
)

# -----------------------------------------------------------------------------
# Period-count audit against Day24 frozen manifest
# -----------------------------------------------------------------------------

actual_window_counts = (
    future_df["period"]
    .value_counts()
    .sort_index()
    .to_dict()
)

print("-" * 120)
print("FROZEN PERIOD COUNT AUDIT")
print("-" * 120)

period_count_checks = {}

for period in DAY27_PROTOCOL["future_periods"]:
    expected_n = int(expected_window_counts[period])
    actual_n = int(actual_window_counts.get(period, 0))

    period_count_checks[period] = (
        actual_n == expected_n
    )

    print_check(
        f"{period}_count_{actual_n}_matches_expected_{expected_n}",
        period_count_checks[period]
    )

checks_stage1["all_period_counts_match"] = print_check(
    "all_period_counts_match",
    all(period_count_checks.values())
)

# -----------------------------------------------------------------------------
# Ordered observation-ID hash audit
# -----------------------------------------------------------------------------

ordered_id_payload = "\n".join(
    future_df["observation_id"].tolist()
).encode("utf-8")

ordered_observation_id_sha256 = hashlib.sha256(
    ordered_id_payload
).hexdigest()

expected_ordered_id_sha256 = (
    day24_prediction_manifest["future_population"][
        "ordered_observation_id_sha256"
    ]
)

checks_stage1["ordered_observation_ids_match_day24"] = print_check(
    "ordered_observation_ids_match_day24",
    ordered_observation_id_sha256
    == expected_ordered_id_sha256
)

# -----------------------------------------------------------------------------
# Population descriptive counts
# These are descriptive only — no exposure exists yet
# -----------------------------------------------------------------------------

future_n_correct = int(
    future_df["correct_day24"].sum()
)

future_n_error = int(
    future_df["error_day24"].sum()
)

future_accuracy = float(
    future_df["correct_day24"].mean()
)

print("-" * 120)
print("FROZEN FUTURE POPULATION SUMMARY")
print("-" * 120)

print(f"Total observations : {len(future_df):,}")
print(f"Correct             : {future_n_correct:,}")
print(f"Error               : {future_n_error:,}")
print(f"Overall accuracy    : {future_accuracy:.6f}")

print()
print("Period counts:")
print(
    future_df["period"]
    .value_counts()
    .sort_index()
    .to_string()
)

print()
print("True-label counts:")
print(
    future_df["y_true"]
    .value_counts()
    .sort_index()
    .to_string()
)

print()
print("Source counts:")
print(
    future_df["source"]
    .value_counts()
    .sort_values(ascending=False)
    .to_string()
)

# -----------------------------------------------------------------------------
# Freeze future population manifest
# -----------------------------------------------------------------------------

day27_future_population_manifest = {
    "day": 27,
    "artifact": "day27_future_population_manifest",
    "status": "FUTURE_POPULATION_FROZEN",

    "source_prediction_artifact": {
        "path": str(DAY24_FUTURE_PREDICTIONS),
        "sha256": current_future_sha256,
        "immutable": True,
    },

    "population": {
        "n": int(len(future_df)),
        "n_correct": future_n_correct,
        "n_error": future_n_error,
        "accuracy": future_accuracy,
        "ordered_observation_id_sha256": ordered_observation_id_sha256,
        "period_counts": {
            k: int(v)
            for k, v in actual_window_counts.items()
        },
        "true_label_counts": {
            str(k): int(v)
            for k, v in future_df["y_true"].value_counts().to_dict().items()
        },
        "source_counts": {
            str(k): int(v)
            for k, v in future_df["source"].value_counts().to_dict().items()
        },
    },

    "correctness_rule": {
        "correct_day24": "y_pred_day24 == y_true",
        "error_day24": "1[y_pred_day24 != y_true]",
    },

    "scientific_firewall": {
        "new_model_inference_performed": False,
        "prediction_values_modified": False,
        "lexical_exposure_computed": False,
        "error_exposure_comparison_performed": False,
    },

    "created_utc": datetime.now(timezone.utc).isoformat(),
}

save_json(
    day27_future_population_manifest,
    DAY27_FUTURE_POPULATION_MANIFEST
)

# -----------------------------------------------------------------------------
# Manifest round-trip audit
# -----------------------------------------------------------------------------

future_population_manifest_reload = load_json(
    DAY27_FUTURE_POPULATION_MANIFEST
)

checks_stage1["population_manifest_exists"] = print_check(
    "population_manifest_exists",
    DAY27_FUTURE_POPULATION_MANIFEST.exists()
)

checks_stage1["population_manifest_nonempty"] = print_check(
    "population_manifest_nonempty",
    DAY27_FUTURE_POPULATION_MANIFEST.stat().st_size > 0
)

checks_stage1["population_manifest_roundtrip_exact"] = print_check(
    "population_manifest_roundtrip_exact",
    future_population_manifest_reload
    == day27_future_population_manifest
)

# -----------------------------------------------------------------------------
# Final Stage 1 gate
# -----------------------------------------------------------------------------

all_stage1_checks_pass = all(
    checks_stage1.values()
)

day27_future_population_manifest_sha256 = sha256_file(
    DAY27_FUTURE_POPULATION_MANIFEST
)

print("-" * 120)
print("FROZEN FUTURE POPULATION MANIFEST")
print("-" * 120)

print(f"Path   : {DAY27_FUTURE_POPULATION_MANIFEST}")
print(
    f"Size   : "
    f"{DAY27_FUTURE_POPULATION_MANIFEST.stat().st_size:,} bytes"
)
print(f"SHA256 : {day27_future_population_manifest_sha256}")

print("-" * 120)

print_check(
    "DAY27_STAGE1_ALL_PASS",
    all_stage1_checks_pass
)

if not all_stage1_checks_pass:
    raise RuntimeError(
        "DAY 27 INTEGRITY STOP: future population freeze failed."
    )

print("=" * 120)
print("DAY 27 STAGE 1: PASS — FUTURE POPULATION FROZEN")
print("NO LEXICAL EXPOSURE HAS BEEN COMPUTED")
print("NO ERROR-vs-EXPOSURE COMPARISON HAS BEEN PERFORMED")
print("READY — STAGE 2 FROZEN LEXICAL-INSTABILITY EXPOSURE CONSTRUCTION")
print("=" * 120)

DAY 27 — STAGE 1 FUTURE POPULATION FREEZE
PASS   — future_prediction_hash_still_matches_frozen_identity
------------------------------------------------------------------------------------------------------------------------
LOADED FUTURE ARTIFACT
------------------------------------------------------------------------------------------------------------------------
Rows    : 20,002
Columns : 12

COLUMN NAMES
['observation_id', 'timestamp_utc', 'period', 'period_type', 'source', 'message_id', 'text', 'y_true', 'y_pred_day24', 'raw_generation_day24', 'prediction_valid_day24', 'inference_status_day24']
------------------------------------------------------------------------------------------------------------------------
SCHEMA CHECK
------------------------------------------------------------------------------------------------------------------------
PASS   — required_future_columns_present
------------------------------------------------------------------------------------------------

## Stage 2 — Frozen Lexical-Instability Exposure Construction

The future population has been frozen at

\[
n = 20{,}002
\]

with the original Day 24 predictions unchanged.

This stage applies the **historical-only lexical-instability predictor frozen on Day 26** to the future text.

### Frozen token score

For vocabulary feature \(v\),

\[
I(v)
=
\max_c
SD_t[A(v,c,t)].
\]

The Day 26 primary vocabulary contains

\[
3{,}240
\]

eligible unigram/bigram features satisfying the frozen historical support rule

\[
df \ge 20.
\]

### Frozen document-level exposure

For future observation \(i\), let \(V_i\) denote the eligible Day 26 vocabulary features present in the document. The primary exposure is

\[
E_i
=
\frac{1}{|V_i|}
\sum_{v\in V_i} I(v).
\]

Rules:

- use **binary feature presence**;
- repeated occurrences do not increase exposure;
- do **not** fit or expand the vocabulary using future text;
- unknown or Day 26-ineligible features are ignored;
- if \(|V_i|=0\), primary exposure is recorded as missing;
- the primary analysis uses the frozen **df ≥ 20** vocabulary;
- **df ≥ 50** exposure is retained only as a pre-specified secondary sensitivity analysis.

The exposure artifact must be constructed and frozen **before any error-versus-exposure comparison is performed**.

**Primary hypothesis status remains UNTESTED during Stage 2.**

In [9]:
# =============================================================================
# Day 27 — Stage 2
# Rebuild frozen Day26 vocabulary / vectorizer interface
# Canonical retained cell
# =============================================================================

from sklearn.feature_extraction.text import CountVectorizer
import sklearn

print("=" * 120)
print("DAY 27 — STAGE 2A FROZEN VOCABULARY / VECTORIZER RECONSTRUCTION")
print("=" * 120)

# -----------------------------------------------------------------------------
# Re-verify immutable Day26 ranking identity
# -----------------------------------------------------------------------------

current_rank_sha256 = sha256_file(DAY26_TOKEN_RANKING)

if (
    current_rank_sha256
    != day27_analysis_manifest["day26_predictor"]["token_rank_sha256"]
):
    raise RuntimeError(
        "DAY 27 INTEGRITY STOP: Day26 token ranking changed before exposure construction."
    )

print_check(
    "token_rank_hash_matches_frozen_identity",
    current_rank_sha256
    == day27_analysis_manifest["day26_predictor"]["token_rank_sha256"]
)

# -----------------------------------------------------------------------------
# Load frozen Day26 token ranking
# -----------------------------------------------------------------------------

token_rank_df = pd.read_csv(DAY26_TOKEN_RANKING)

required_rank_columns = [
    "eligible_feature_index",
    "token",
    "historical_df",
    "token_instability",
]

missing_rank_columns = [
    col for col in required_rank_columns
    if col not in token_rank_df.columns
]

print("-" * 120)
print("TOKEN-RANKING SCHEMA")
print("-" * 120)

rank_schema_ok = print_check(
    "required_rank_columns_present",
    len(missing_rank_columns) == 0
)

if not rank_schema_ok:
    print(f"Missing columns: {missing_rank_columns}")
    raise RuntimeError(
        "DAY 27 STOP: frozen Day26 token-ranking schema is incomplete."
    )

# -----------------------------------------------------------------------------
# Canonical type normalization
# Representation only; no scientific values are changed
# -----------------------------------------------------------------------------

token_rank_df["eligible_feature_index"] = (
    token_rank_df["eligible_feature_index"].astype(int)
)

token_rank_df["token"] = (
    token_rank_df["token"].astype(str)
)

token_rank_df["historical_df"] = (
    token_rank_df["historical_df"].astype(int)
)

token_rank_df["token_instability"] = (
    token_rank_df["token_instability"].astype(float)
)

# -----------------------------------------------------------------------------
# Validate vocabulary structure
# -----------------------------------------------------------------------------

expected_vocab_n = int(
    day27_analysis_manifest["day26_predictor"]["eligible_token_n"]
)

expected_indices = np.arange(expected_vocab_n)

actual_indices_sorted = np.sort(
    token_rank_df["eligible_feature_index"].to_numpy()
)

checks_stage2a = {}

print("-" * 120)
print("FROZEN VOCABULARY INTEGRITY")
print("-" * 120)

checks_stage2a["token_count_3240"] = print_check(
    "token_count_3240",
    len(token_rank_df) == expected_vocab_n == 3240
)

checks_stage2a["token_nonmissing"] = print_check(
    "token_nonmissing",
    token_rank_df["token"].notna().all()
)

checks_stage2a["tokens_unique"] = print_check(
    "tokens_unique",
    token_rank_df["token"].is_unique
)

checks_stage2a["feature_indices_unique"] = print_check(
    "feature_indices_unique",
    token_rank_df["eligible_feature_index"].is_unique
)

checks_stage2a["feature_indices_contiguous_0_to_3239"] = print_check(
    "feature_indices_contiguous_0_to_3239",
    np.array_equal(actual_indices_sorted, expected_indices)
)

checks_stage2a["historical_df_ge_20"] = print_check(
    "historical_df_ge_20",
    (token_rank_df["historical_df"] >= 20).all()
)

checks_stage2a["token_instability_nonmissing"] = print_check(
    "token_instability_nonmissing",
    token_rank_df["token_instability"].notna().all()
)

checks_stage2a["token_instability_finite"] = print_check(
    "token_instability_finite",
    np.isfinite(
        token_rank_df["token_instability"].to_numpy()
    ).all()
)

checks_stage2a["token_instability_nonnegative"] = print_check(
    "token_instability_nonnegative",
    (token_rank_df["token_instability"] >= 0).all()
)

# -----------------------------------------------------------------------------
# Verify frozen sklearn identity
# -----------------------------------------------------------------------------

expected_sklearn_version = str(
    day26_handoff["tokenizer"]["sklearn_version"]
)

current_sklearn_version = str(sklearn.__version__)

checks_stage2a["sklearn_version_matches_day26"] = print_check(
    f"sklearn_version_matches_day26_{expected_sklearn_version}",
    current_sklearn_version == expected_sklearn_version
)

# -----------------------------------------------------------------------------
# Reconstruct exact frozen vocabulary mapping
#
# IMPORTANT:
# We use the original Day26 eligible_feature_index as the CountVectorizer
# column index. No vocabulary is fitted on future text.
# -----------------------------------------------------------------------------

frozen_vocabulary = {
    row.token: int(row.eligible_feature_index)
    for row in token_rank_df[
        ["token", "eligible_feature_index"]
    ].itertuples(index=False)
}

checks_stage2a["frozen_vocabulary_size_3240"] = print_check(
    "frozen_vocabulary_size_3240",
    len(frozen_vocabulary) == 3240
)

checks_stage2a["frozen_vocabulary_indices_exact"] = print_check(
    "frozen_vocabulary_indices_exact",
    sorted(frozen_vocabulary.values())
    == list(range(3240))
)

# -----------------------------------------------------------------------------
# Rebuild frozen CountVectorizer interface
#
# vocabulary=... means transform-only interface.
# No fit() or fit_transform() is permitted on future text.
# -----------------------------------------------------------------------------

ngram_range_frozen = tuple(
    day26_handoff["tokenizer"]["ngram_range"]
)

frozen_vectorizer = CountVectorizer(
    vocabulary=frozen_vocabulary,
    lowercase=bool(
        day26_handoff["tokenizer"]["lowercase"]
    ),
    ngram_range=ngram_range_frozen,
    token_pattern=day26_handoff["tokenizer"]["token_pattern"],
    stop_words=day26_handoff["tokenizer"]["stop_words"],
    binary=True,
)

# -----------------------------------------------------------------------------
# Verify reconstructed vectorizer identity
# -----------------------------------------------------------------------------

checks_stage2a["vectorizer_binary_true"] = print_check(
    "vectorizer_binary_true",
    frozen_vectorizer.binary is True
)

checks_stage2a["vectorizer_lowercase_matches"] = print_check(
    "vectorizer_lowercase_matches",
    frozen_vectorizer.lowercase
    == day26_handoff["tokenizer"]["lowercase"]
)

checks_stage2a["vectorizer_ngram_range_matches"] = print_check(
    "vectorizer_ngram_range_matches",
    tuple(frozen_vectorizer.ngram_range)
    == tuple(day26_handoff["tokenizer"]["ngram_range"])
)

checks_stage2a["vectorizer_token_pattern_matches"] = print_check(
    "vectorizer_token_pattern_matches",
    frozen_vectorizer.token_pattern
    == day26_handoff["tokenizer"]["token_pattern"]
)

checks_stage2a["vectorizer_stop_words_matches"] = print_check(
    "vectorizer_stop_words_matches",
    frozen_vectorizer.stop_words
    == day26_handoff["tokenizer"]["stop_words"]
)

# -----------------------------------------------------------------------------
# Token instability lookup aligned exactly to vectorizer columns
# -----------------------------------------------------------------------------

token_rank_by_index = (
    token_rank_df
    .sort_values("eligible_feature_index")
    .reset_index(drop=True)
)

instability_vector = (
    token_rank_by_index["token_instability"]
    .to_numpy(dtype=float)
)

historical_df_vector = (
    token_rank_by_index["historical_df"]
    .to_numpy(dtype=int)
)

token_vector = (
    token_rank_by_index["token"]
    .astype(str)
    .to_numpy()
)

checks_stage2a["instability_vector_length_3240"] = print_check(
    "instability_vector_length_3240",
    len(instability_vector) == 3240
)

checks_stage2a["df_vector_length_3240"] = print_check(
    "df_vector_length_3240",
    len(historical_df_vector) == 3240
)

# -----------------------------------------------------------------------------
# Pre-specified secondary df>=50 subset
# This is defined now but remains SECONDARY.
# -----------------------------------------------------------------------------

df50_mask = historical_df_vector >= 50
df50_feature_indices = np.where(df50_mask)[0]

n_df50_tokens = int(df50_mask.sum())

checks_stage2a["df50_subset_nonempty"] = print_check(
    "df50_subset_nonempty",
    n_df50_tokens > 0
)

checks_stage2a["df50_subset_strictly_secondary"] = print_check(
    "df50_subset_strictly_secondary",
    day27_analysis_manifest["secondary_analyses"][
        "df50_sensitivity"
    ] is True
    and day27_analysis_manifest["primary_exposure"][
        "primary_minimum_historical_df"
    ] == 20
)

# -----------------------------------------------------------------------------
# Frozen token-order hash reconstruction
# -----------------------------------------------------------------------------

token_order_payload = "\n".join(
    token_vector.tolist()
).encode("utf-8")

reconstructed_token_order_sha256 = hashlib.sha256(
    token_order_payload
).hexdigest()

expected_token_order_sha256 = (
    day26_handoff["vocabulary"]["token_order_sha256"]
)

checks_stage2a["token_order_hash_matches_day26"] = print_check(
    "token_order_hash_matches_day26",
    reconstructed_token_order_sha256
    == expected_token_order_sha256
)

# -----------------------------------------------------------------------------
# Summary
# -----------------------------------------------------------------------------

print("-" * 120)
print("FROZEN VECTORIZER SUMMARY")
print("-" * 120)

print(f"sklearn version             : {current_sklearn_version}")
print(f"Vocabulary size             : {len(frozen_vocabulary):,}")
print(f"Primary support threshold   : df >= 20")
print(f"Secondary df>=50 token n    : {n_df50_tokens:,}")
print(f"Binary presence             : {frozen_vectorizer.binary}")
print(f"Lowercase                   : {frozen_vectorizer.lowercase}")
print(f"N-gram range                : {frozen_vectorizer.ngram_range}")
print(f"Token pattern               : {frozen_vectorizer.token_pattern}")
print(f"Stop words                  : {frozen_vectorizer.stop_words}")
print(f"Token-order SHA256          : {reconstructed_token_order_sha256}")

# -----------------------------------------------------------------------------
# Final Stage 2A gate
# -----------------------------------------------------------------------------

all_stage2a_checks_pass = all(
    checks_stage2a.values()
)

print("-" * 120)

print_check(
    "DAY27_STAGE2A_ALL_PASS",
    all_stage2a_checks_pass
)

if not all_stage2a_checks_pass:
    raise RuntimeError(
        "DAY 27 INTEGRITY STOP: frozen vocabulary/vectorizer reconstruction failed."
    )

print("=" * 120)
print("DAY 27 STAGE 2A: PASS — FROZEN VECTORIZER RECONSTRUCTED")
print("NO FUTURE VOCABULARY FIT WAS PERFORMED")
print("NO DOCUMENT EXPOSURE HAS YET BEEN COMPUTED")
print("NO ERROR-vs-EXPOSURE COMPARISON HAS BEEN PERFORMED")
print("READY — STAGE 2B FUTURE DOCUMENT EXPOSURE CONSTRUCTION")
print("=" * 120)

DAY 27 — STAGE 2A FROZEN VOCABULARY / VECTORIZER RECONSTRUCTION
PASS   — token_rank_hash_matches_frozen_identity
------------------------------------------------------------------------------------------------------------------------
TOKEN-RANKING SCHEMA
------------------------------------------------------------------------------------------------------------------------
PASS   — required_rank_columns_present
------------------------------------------------------------------------------------------------------------------------
FROZEN VOCABULARY INTEGRITY
------------------------------------------------------------------------------------------------------------------------
PASS   — token_count_3240
PASS   — token_nonmissing
PASS   — tokens_unique
PASS   — feature_indices_unique
PASS   — feature_indices_contiguous_0_to_3239
PASS   — historical_df_ge_20
PASS   — token_instability_nonmissing
PASS   — token_instability_finite
PASS   — token_instability_nonnegative
PASS   — sklearn_versi

In [10]:
# =============================================================================
# Day 27 — Stage 2B
# Future lexical-instability exposure construction and freeze
# Canonical retained cell
# =============================================================================

print("=" * 120)
print("DAY 27 — STAGE 2B FUTURE LEXICAL-INSTABILITY EXPOSURE CONSTRUCTION")
print("=" * 120)

# -----------------------------------------------------------------------------
# Re-verify frozen identities immediately before transform
# -----------------------------------------------------------------------------

print_check(
    "future_prediction_hash_still_frozen",
    sha256_file(DAY24_FUTURE_PREDICTIONS)
    == day27_analysis_manifest["future_population"]["prediction_sha256"]
)

print_check(
    "token_rank_hash_still_frozen",
    sha256_file(DAY26_TOKEN_RANKING)
    == day27_analysis_manifest["day26_predictor"]["token_rank_sha256"]
)

if (
    sha256_file(DAY24_FUTURE_PREDICTIONS)
    != day27_analysis_manifest["future_population"]["prediction_sha256"]
    or
    sha256_file(DAY26_TOKEN_RANKING)
    != day27_analysis_manifest["day26_predictor"]["token_rank_sha256"]
):
    raise RuntimeError(
        "DAY 27 INTEGRITY STOP: upstream frozen artifact changed before exposure construction."
    )

# -----------------------------------------------------------------------------
# Transform future text using frozen Day26 vocabulary only
#
# IMPORTANT:
# transform() only.
# No fit() / fit_transform() / vocabulary update is performed.
# -----------------------------------------------------------------------------

future_text = future_df["text"].astype(str).tolist()

X_future = frozen_vectorizer.transform(future_text)

print("-" * 120)
print("FROZEN FUTURE TRANSFORM")
print("-" * 120)

print(f"Rows                    : {X_future.shape[0]:,}")
print(f"Frozen feature columns  : {X_future.shape[1]:,}")
print(f"Matrix format           : {type(X_future).__name__}")
print(f"Nonzero entries         : {X_future.nnz:,}")

checks_stage2b = {}

checks_stage2b["future_matrix_rows_20002"] = print_check(
    "future_matrix_rows_20002",
    X_future.shape[0] == 20002
)

checks_stage2b["future_matrix_columns_3240"] = print_check(
    "future_matrix_columns_3240",
    X_future.shape[1] == 3240
)

checks_stage2b["future_matrix_binary"] = print_check(
    "future_matrix_binary",
    (
        X_future.nnz == 0
        or np.all(X_future.data == 1)
    )
)

# -----------------------------------------------------------------------------
# Primary df>=20 exposure
#
# For each document:
# eligible_token_count = number of frozen vocabulary features present
# instability_sum      = sum I(v) across present eligible features
# primary exposure     = instability_sum / eligible_token_count
#
# zero eligible tokens -> NaN
# -----------------------------------------------------------------------------

eligible_token_count = np.asarray(
    X_future.sum(axis=1)
).ravel().astype(int)

instability_sum = np.asarray(
    X_future @ instability_vector
).ravel().astype(float)

primary_exposure = np.full(
    len(future_df),
    np.nan,
    dtype=float
)

primary_nonmissing_mask = eligible_token_count > 0

primary_exposure[primary_nonmissing_mask] = (
    instability_sum[primary_nonmissing_mask]
    / eligible_token_count[primary_nonmissing_mask]
)

# -----------------------------------------------------------------------------
# Secondary df>=50 exposure
#
# Pre-specified sensitivity only.
# -----------------------------------------------------------------------------

X_future_df50 = X_future[:, df50_feature_indices]

df50_token_count = np.asarray(
    X_future_df50.sum(axis=1)
).ravel().astype(int)

df50_instability_vector = instability_vector[df50_feature_indices]

df50_instability_sum = np.asarray(
    X_future_df50 @ df50_instability_vector
).ravel().astype(float)

df50_exposure = np.full(
    len(future_df),
    np.nan,
    dtype=float
)

df50_nonmissing_mask = df50_token_count > 0

df50_exposure[df50_nonmissing_mask] = (
    df50_instability_sum[df50_nonmissing_mask]
    / df50_token_count[df50_nonmissing_mask]
)

# -----------------------------------------------------------------------------
# Coverage diagnostics
#
# Primary vocabulary coverage here is defined relative to the full frozen
# 3,240-feature vocabulary.
#
# Secondary df50 coverage is relative to the frozen df>=50 subset.
# -----------------------------------------------------------------------------

primary_vocab_n = len(instability_vector)
df50_vocab_n = len(df50_feature_indices)

primary_vocab_fraction_present = (
    eligible_token_count / primary_vocab_n
)

df50_vocab_fraction_present = (
    df50_token_count / df50_vocab_n
)

# -----------------------------------------------------------------------------
# Build exposure artifact
#
# IMPORTANT:
# Do NOT include correctness/error columns in this artifact.
# Exposure is frozen independently before hypothesis testing.
# -----------------------------------------------------------------------------

exposure_df = pd.DataFrame({
    "observation_id": future_df["observation_id"].astype(str).values,
    "period": future_df["period"].astype(str).values,
    "source": future_df["source"].astype(str).values,
    "y_true": future_df["y_true"].astype(str).values,

    "eligible_token_count_df20": eligible_token_count,
    "instability_sum_df20": instability_sum,
    "lexical_instability_exposure_df20": primary_exposure,
    "vocab_fraction_present_df20": primary_vocab_fraction_present,

    "eligible_token_count_df50": df50_token_count,
    "instability_sum_df50": df50_instability_sum,
    "lexical_instability_exposure_df50": df50_exposure,
    "vocab_fraction_present_df50": df50_vocab_fraction_present,
})

# -----------------------------------------------------------------------------
# Artifact integrity checks before save
# -----------------------------------------------------------------------------

print("-" * 120)
print("EXPOSURE CONSTRUCTION INTEGRITY")
print("-" * 120)

checks_stage2b["exposure_rows_20002"] = print_check(
    "exposure_rows_20002",
    len(exposure_df) == 20002
)

checks_stage2b["observation_ids_match_future_population"] = print_check(
    "observation_ids_match_future_population",
    exposure_df["observation_id"].tolist()
    == future_df["observation_id"].astype(str).tolist()
)

checks_stage2b["no_duplicate_observation_ids"] = print_check(
    "no_duplicate_observation_ids",
    exposure_df["observation_id"].is_unique
)

checks_stage2b["primary_count_nonnegative"] = print_check(
    "primary_count_nonnegative",
    (exposure_df["eligible_token_count_df20"] >= 0).all()
)

checks_stage2b["df50_count_nonnegative"] = print_check(
    "df50_count_nonnegative",
    (exposure_df["eligible_token_count_df50"] >= 0).all()
)

checks_stage2b["df50_count_never_exceeds_df20"] = print_check(
    "df50_count_never_exceeds_df20",
    (
        exposure_df["eligible_token_count_df50"]
        <= exposure_df["eligible_token_count_df20"]
    ).all()
)

checks_stage2b["primary_missing_iff_zero_eligible"] = print_check(
    "primary_missing_iff_zero_eligible",
    np.array_equal(
        exposure_df[
            "lexical_instability_exposure_df20"
        ].isna().to_numpy(),
        (
            exposure_df["eligible_token_count_df20"].to_numpy()
            == 0
        )
    )
)

checks_stage2b["df50_missing_iff_zero_eligible"] = print_check(
    "df50_missing_iff_zero_eligible",
    np.array_equal(
        exposure_df[
            "lexical_instability_exposure_df50"
        ].isna().to_numpy(),
        (
            exposure_df["eligible_token_count_df50"].to_numpy()
            == 0
        )
    )
)

primary_exposure_nonmissing = exposure_df[
    "lexical_instability_exposure_df20"
].dropna()

df50_exposure_nonmissing = exposure_df[
    "lexical_instability_exposure_df50"
].dropna()

checks_stage2b["primary_exposure_finite_when_nonmissing"] = print_check(
    "primary_exposure_finite_when_nonmissing",
    np.isfinite(primary_exposure_nonmissing.to_numpy()).all()
)

checks_stage2b["df50_exposure_finite_when_nonmissing"] = print_check(
    "df50_exposure_finite_when_nonmissing",
    np.isfinite(df50_exposure_nonmissing.to_numpy()).all()
)

checks_stage2b["primary_exposure_nonnegative"] = print_check(
    "primary_exposure_nonnegative",
    (primary_exposure_nonmissing >= 0).all()
)

checks_stage2b["df50_exposure_nonnegative"] = print_check(
    "df50_exposure_nonnegative",
    (df50_exposure_nonmissing >= 0).all()
)

# -----------------------------------------------------------------------------
# Save frozen exposure artifact
# -----------------------------------------------------------------------------

exposure_df.to_csv(
    DAY27_EXPOSURE_CSV,
    index=False
)

# -----------------------------------------------------------------------------
# Deterministic reload audit
# -----------------------------------------------------------------------------

exposure_reload = pd.read_csv(
    DAY27_EXPOSURE_CSV,
    dtype={"observation_id": str}
)

checks_stage2b["exposure_artifact_exists"] = print_check(
    "exposure_artifact_exists",
    DAY27_EXPOSURE_CSV.exists()
)

checks_stage2b["exposure_artifact_nonempty"] = print_check(
    "exposure_artifact_nonempty",
    DAY27_EXPOSURE_CSV.stat().st_size > 0
)

checks_stage2b["exposure_reload_rows_20002"] = print_check(
    "exposure_reload_rows_20002",
    len(exposure_reload) == 20002
)

checks_stage2b["exposure_reload_ids_exact"] = print_check(
    "exposure_reload_ids_exact",
    exposure_reload["observation_id"].tolist()
    == exposure_df["observation_id"].tolist()
)

# Float-safe round-trip comparisons
numeric_exposure_columns = [
    "eligible_token_count_df20",
    "instability_sum_df20",
    "lexical_instability_exposure_df20",
    "vocab_fraction_present_df20",
    "eligible_token_count_df50",
    "instability_sum_df50",
    "lexical_instability_exposure_df50",
    "vocab_fraction_present_df50",
]

roundtrip_numeric_ok = True

for col in numeric_exposure_columns:
    a = exposure_df[col].to_numpy(dtype=float)
    b = exposure_reload[col].to_numpy(dtype=float)

    if not np.allclose(
        a,
        b,
        equal_nan=True,
        rtol=1e-12,
        atol=1e-12,
    ):
        roundtrip_numeric_ok = False
        break

checks_stage2b["exposure_numeric_roundtrip_exact_with_tolerance"] = print_check(
    "exposure_numeric_roundtrip_exact_with_tolerance",
    roundtrip_numeric_ok
)

# -----------------------------------------------------------------------------
# Exposure-only summary
#
# IMPORTANT:
# No error/correct grouping is shown here.
# -----------------------------------------------------------------------------

n_primary_available = int(
    exposure_df["lexical_instability_exposure_df20"].notna().sum()
)

n_primary_missing = int(
    exposure_df["lexical_instability_exposure_df20"].isna().sum()
)

n_df50_available = int(
    exposure_df["lexical_instability_exposure_df50"].notna().sum()
)

n_df50_missing = int(
    exposure_df["lexical_instability_exposure_df50"].isna().sum()
)

print("-" * 120)
print("EXPOSURE-ONLY SUMMARY")
print("-" * 120)

print(f"Primary df>=20 exposure available : {n_primary_available:,}")
print(f"Primary df>=20 exposure missing   : {n_primary_missing:,}")
print(
    f"Primary availability fraction     : "
    f"{n_primary_available / len(exposure_df):.6f}"
)

print()

print(f"Secondary df>=50 available        : {n_df50_available:,}")
print(f"Secondary df>=50 missing          : {n_df50_missing:,}")
print(
    f"Secondary availability fraction   : "
    f"{n_df50_available / len(exposure_df):.6f}"
)

print()

print("Primary eligible-token count summary:")
print(
    exposure_df["eligible_token_count_df20"]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
    .to_string()
)

print()

print("Primary exposure summary:")
print(
    exposure_df["lexical_instability_exposure_df20"]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
    .to_string()
)

# -----------------------------------------------------------------------------
# Freeze artifact identity
# -----------------------------------------------------------------------------

day27_exposure_sha256 = sha256_file(
    DAY27_EXPOSURE_CSV
)

print("-" * 120)
print("FROZEN EXPOSURE ARTIFACT")
print("-" * 120)

print(f"Path   : {DAY27_EXPOSURE_CSV}")
print(f"Size   : {DAY27_EXPOSURE_CSV.stat().st_size:,} bytes")
print(f"SHA256 : {day27_exposure_sha256}")

# -----------------------------------------------------------------------------
# Final Stage 2B gate
# -----------------------------------------------------------------------------

all_stage2b_checks_pass = all(
    checks_stage2b.values()
)

print("-" * 120)

print_check(
    "DAY27_STAGE2B_ALL_PASS",
    all_stage2b_checks_pass
)

if not all_stage2b_checks_pass:
    raise RuntimeError(
        "DAY 27 INTEGRITY STOP: future exposure construction/freeze failed."
    )

print("=" * 120)
print("DAY 27 STAGE 2B: PASS — FUTURE EXPOSURE ARTIFACT FROZEN")
print("PRIMARY EXPOSURE = MEAN I(v), df>=20, BINARY PRESENCE")
print("df>=50 EXPOSURE RETAINED AS SECONDARY SENSITIVITY ONLY")
print("NO ERROR-vs-EXPOSURE COMPARISON HAS BEEN PERFORMED")
print("PRIMARY H1 STATUS: STILL UNTESTED")
print("READY — STAGE 2C EXPOSURE ARTIFACT INTEGRITY / RELOAD AUDIT")
print("=" * 120)

DAY 27 — STAGE 2B FUTURE LEXICAL-INSTABILITY EXPOSURE CONSTRUCTION
PASS   — future_prediction_hash_still_frozen
PASS   — token_rank_hash_still_frozen
------------------------------------------------------------------------------------------------------------------------
FROZEN FUTURE TRANSFORM
------------------------------------------------------------------------------------------------------------------------
Rows                    : 20,002
Frozen feature columns  : 3,240
Matrix format           : csr_matrix
Nonzero entries         : 291,622
PASS   — future_matrix_rows_20002
PASS   — future_matrix_columns_3240
PASS   — future_matrix_binary
------------------------------------------------------------------------------------------------------------------------
EXPOSURE CONSTRUCTION INTEGRITY
------------------------------------------------------------------------------------------------------------------------
PASS   — exposure_rows_20002
PASS   — observation_ids_match_future_populat

In [12]:
# =============================================================================
# Day 27 — Stage 2C
# Frozen exposure artifact integrity + deterministic reload audit
# Canonical retained cell
# =============================================================================

print("=" * 120)
print("DAY 27 — STAGE 2C FROZEN EXPOSURE ARTIFACT INTEGRITY AUDIT")
print("=" * 120)

# -----------------------------------------------------------------------------
# Reload exposure artifact from disk only
# -----------------------------------------------------------------------------

exposure_frozen = pd.read_csv(
    DAY27_EXPOSURE_CSV,
    dtype={"observation_id": str}
)

current_exposure_sha256 = sha256_file(
    DAY27_EXPOSURE_CSV
)

expected_exposure_sha256 = day27_exposure_sha256

# -----------------------------------------------------------------------------
# Canonical schema
# -----------------------------------------------------------------------------

expected_exposure_columns = [
    "observation_id",
    "period",
    "source",
    "y_true",

    "eligible_token_count_df20",
    "instability_sum_df20",
    "lexical_instability_exposure_df20",
    "vocab_fraction_present_df20",

    "eligible_token_count_df50",
    "instability_sum_df50",
    "lexical_instability_exposure_df50",
    "vocab_fraction_present_df50",
]

forbidden_outcome_columns = [
    "correct_day24",
    "error_day24",
    "y_pred_day24",
    "prediction_valid_day24",
    "inference_status_day24",
]

# -----------------------------------------------------------------------------
# Integrity checks
# -----------------------------------------------------------------------------

checks_stage2c = {}

print("-" * 120)
print("ARTIFACT IDENTITY")
print("-" * 120)

checks_stage2c["exposure_artifact_exists"] = print_check(
    "exposure_artifact_exists",
    DAY27_EXPOSURE_CSV.exists()
)

checks_stage2c["exposure_artifact_nonempty"] = print_check(
    "exposure_artifact_nonempty",
    DAY27_EXPOSURE_CSV.stat().st_size > 0
)

checks_stage2c["exposure_hash_stable"] = print_check(
    "exposure_hash_stable",
    current_exposure_sha256 == expected_exposure_sha256
)

print("-" * 120)
print("SCHEMA / FIREWALL")
print("-" * 120)

checks_stage2c["exposure_schema_exact"] = print_check(
    "exposure_schema_exact",
    list(exposure_frozen.columns) == expected_exposure_columns
)

checks_stage2c["no_outcome_columns_in_exposure_artifact"] = print_check(
    "no_outcome_columns_in_exposure_artifact",
    not any(
        col in exposure_frozen.columns
        for col in forbidden_outcome_columns
    )
)

checks_stage2c["rows_exact_20002"] = print_check(
    "rows_exact_20002",
    len(exposure_frozen) == 20002
)

checks_stage2c["observation_ids_unique"] = print_check(
    "observation_ids_unique",
    exposure_frozen["observation_id"].is_unique
)

checks_stage2c["observation_ids_nonmissing"] = print_check(
    "observation_ids_nonmissing",
    exposure_frozen["observation_id"].notna().all()
)

# -----------------------------------------------------------------------------
# Ordered population identity
# -----------------------------------------------------------------------------

exposure_ordered_id_payload = "\n".join(
    exposure_frozen["observation_id"].tolist()
).encode("utf-8")

exposure_ordered_id_sha256 = hashlib.sha256(
    exposure_ordered_id_payload
).hexdigest()

checks_stage2c["ordered_ids_match_frozen_future_population"] = print_check(
    "ordered_ids_match_frozen_future_population",
    exposure_ordered_id_sha256
    == day27_future_population_manifest[
        "population"
    ]["ordered_observation_id_sha256"]
)

# -----------------------------------------------------------------------------
# Scientific definition checks
# -----------------------------------------------------------------------------

primary_missing_mask = exposure_frozen[
    "lexical_instability_exposure_df20"
].isna()

df50_missing_mask = exposure_frozen[
    "lexical_instability_exposure_df50"
].isna()

checks_stage2c["primary_missing_exactly_one"] = print_check(
    "primary_missing_exactly_one",
    int(primary_missing_mask.sum()) == 1
)

checks_stage2c["primary_missing_only_when_zero_eligible"] = print_check(
    "primary_missing_only_when_zero_eligible",
    np.array_equal(
        primary_missing_mask.to_numpy(),
        (
            exposure_frozen["eligible_token_count_df20"].to_numpy()
            == 0
        )
    )
)

checks_stage2c["df50_missing_exactly_six"] = print_check(
    "df50_missing_exactly_six",
    int(df50_missing_mask.sum()) == 6
)

checks_stage2c["df50_missing_only_when_zero_eligible"] = print_check(
    "df50_missing_only_when_zero_eligible",
    np.array_equal(
        df50_missing_mask.to_numpy(),
        (
            exposure_frozen["eligible_token_count_df50"].to_numpy()
            == 0
        )
    )
)

checks_stage2c["primary_values_finite_when_present"] = print_check(
    "primary_values_finite_when_present",
    np.isfinite(
        exposure_frozen.loc[
            ~primary_missing_mask,
            "lexical_instability_exposure_df20"
        ].to_numpy(dtype=float)
    ).all()
)

checks_stage2c["primary_values_nonnegative"] = print_check(
    "primary_values_nonnegative",
    (
        exposure_frozen.loc[
            ~primary_missing_mask,
            "lexical_instability_exposure_df20"
        ] >= 0
    ).all()
)

# -----------------------------------------------------------------------------
# Deterministic recomputation from saved sufficient statistics
#
# This validates the saved exposure formula without re-vectorizing text.
# -----------------------------------------------------------------------------

recomputed_primary = np.full(
    len(exposure_frozen),
    np.nan,
    dtype=float
)

primary_nonzero = (
    exposure_frozen["eligible_token_count_df20"].to_numpy(dtype=int)
    > 0
)

recomputed_primary[primary_nonzero] = (
    exposure_frozen.loc[
        primary_nonzero,
        "instability_sum_df20"
    ].to_numpy(dtype=float)
    /
    exposure_frozen.loc[
        primary_nonzero,
        "eligible_token_count_df20"
    ].to_numpy(dtype=float)
)

checks_stage2c["primary_formula_recomputes_exactly"] = print_check(
    "primary_formula_recomputes_exactly",
    np.allclose(
        recomputed_primary,
        exposure_frozen[
            "lexical_instability_exposure_df20"
        ].to_numpy(dtype=float),
        equal_nan=True,
        rtol=1e-12,
        atol=1e-12,
    )
)

recomputed_df50 = np.full(
    len(exposure_frozen),
    np.nan,
    dtype=float
)

df50_nonzero = (
    exposure_frozen["eligible_token_count_df50"].to_numpy(dtype=int)
    > 0
)

recomputed_df50[df50_nonzero] = (
    exposure_frozen.loc[
        df50_nonzero,
        "instability_sum_df50"
    ].to_numpy(dtype=float)
    /
    exposure_frozen.loc[
        df50_nonzero,
        "eligible_token_count_df50"
    ].to_numpy(dtype=float)
)

checks_stage2c["df50_formula_recomputes_exactly"] = print_check(
    "df50_formula_recomputes_exactly",
    np.allclose(
        recomputed_df50,
        exposure_frozen[
            "lexical_instability_exposure_df50"
        ].to_numpy(dtype=float),
        equal_nan=True,
        rtol=1e-12,
        atol=1e-12,
    )
)

# -----------------------------------------------------------------------------
# Freeze exposure identity for downstream stages
# -----------------------------------------------------------------------------

day27_frozen_exposure_identity = {
    "path": str(DAY27_EXPOSURE_CSV),
    "sha256": current_exposure_sha256,
    "n": int(len(exposure_frozen)),
    "primary_available_n": int(
        exposure_frozen[
            "lexical_instability_exposure_df20"
        ].notna().sum()
    ),
    "primary_missing_n": int(
        exposure_frozen[
            "lexical_instability_exposure_df20"
        ].isna().sum()
    ),
    "primary_rule": "mean I(v) across frozen df>=20 tokens present",
    "df50_role": "secondary_sensitivity_only",
    "contains_outcome_columns": False,
}

# -----------------------------------------------------------------------------
# Final gate
# -----------------------------------------------------------------------------

all_stage2c_checks_pass = all(
    checks_stage2c.values()
)

print("-" * 120)
print("FROZEN EXPOSURE IDENTITY")
print("-" * 120)

print(f"Path                    : {DAY27_EXPOSURE_CSV}")
print(f"SHA256                  : {current_exposure_sha256}")
print(f"Rows                    : {len(exposure_frozen):,}")
print(
    f"Primary exposure present: "
    f"{day27_frozen_exposure_identity['primary_available_n']:,}"
)
print(
    f"Primary exposure missing: "
    f"{day27_frozen_exposure_identity['primary_missing_n']:,}"
)
print(f"Contains outcome columns: False")

print("-" * 120)

print_check(
    "DAY27_STAGE2C_ALL_PASS",
    all_stage2c_checks_pass
)

if not all_stage2c_checks_pass:
    raise RuntimeError(
        "DAY 27 INTEGRITY STOP: frozen exposure artifact integrity audit failed."
    )

print("=" * 120)
print("DAY 27 STAGE 2: COMPLETE")
print("PASS — FROZEN EXPOSURE ARTIFACT IS STABLE AND OUTCOME-INDEPENDENT")
print("PRIMARY H1 STATUS: STILL UNTESTED")
print("READY — STAGE 3 EXPOSURE QC / COVERAGE AUDIT")
print("=" * 120)

DAY 27 — STAGE 2C FROZEN EXPOSURE ARTIFACT INTEGRITY AUDIT
------------------------------------------------------------------------------------------------------------------------
ARTIFACT IDENTITY
------------------------------------------------------------------------------------------------------------------------
PASS   — exposure_artifact_exists
PASS   — exposure_artifact_nonempty
PASS   — exposure_hash_stable
------------------------------------------------------------------------------------------------------------------------
SCHEMA / FIREWALL
------------------------------------------------------------------------------------------------------------------------
PASS   — exposure_schema_exact
PASS   — no_outcome_columns_in_exposure_artifact
PASS   — rows_exact_20002
PASS   — observation_ids_unique
PASS   — observation_ids_nonmissing
PASS   — ordered_ids_match_frozen_future_population
PASS   — primary_missing_exactly_one
PASS   — primary_missing_only_when_zero_eligible
PASS   — 

## Stage 3 — Exposure QC / Coverage Audit

The Day 26 predictor has now been applied to the untouched Day 24 future population, and the resulting exposure artifact has been frozen independently of classification outcomes.

Before testing the primary hypothesis, Day 27 performs a pre-specified exposure quality-control audit.

### QC objectives

We will characterize:

- availability of the frozen primary df ≥ 20 exposure;
- number and fraction of zero-eligible-token observations;
- eligible-token-count distribution;
- exposure coverage across F1–F6;
- exposure coverage across true labels;
- exposure coverage across sources;
- exposure coverage across correctness groups.

These analyses are **diagnostic only**.

No coverage pattern may be used to:

- redefine the Day 26 vocabulary;
- alter the df ≥ 20 threshold;
- change the document-level exposure rule;
- impute missing exposure using future labels or correctness;
- remove observations selectively based on outcome, source, class, or period.

Observations with zero eligible df ≥ 20 tokens remain missing under the frozen primary rule.

The primary hypothesis test remains locked until this QC stage is completed and saved.

In [13]:
# =============================================================================
# Day 27 — Stage 3
# Exposure QC / coverage / missingness audit
# Canonical retained cell
# =============================================================================

print("=" * 120)
print("DAY 27 — STAGE 3 EXPOSURE QC / COVERAGE AUDIT")
print("=" * 120)

# -----------------------------------------------------------------------------
# Re-verify frozen exposure identity before QC
# -----------------------------------------------------------------------------

current_exposure_sha256_stage3 = sha256_file(
    DAY27_EXPOSURE_CSV
)

print_check(
    "exposure_hash_still_frozen",
    current_exposure_sha256_stage3
    == day27_frozen_exposure_identity["sha256"]
)

if (
    current_exposure_sha256_stage3
    != day27_frozen_exposure_identity["sha256"]
):
    raise RuntimeError(
        "DAY 27 INTEGRITY STOP: exposure artifact changed before QC."
    )

# -----------------------------------------------------------------------------
# Reload frozen exposure artifact
# -----------------------------------------------------------------------------

exposure_qc_base = pd.read_csv(
    DAY27_EXPOSURE_CSV,
    dtype={"observation_id": str}
)

# -----------------------------------------------------------------------------
# Merge correctness ONLY for availability diagnostics
#
# IMPORTANT:
# We do NOT compute mean exposure by correctness here.
# We do NOT compute delta_primary here.
# -----------------------------------------------------------------------------

qc_outcome_columns = future_df[
    [
        "observation_id",
        "correct_day24",
        "error_day24",
    ]
].copy()

qc_df = exposure_qc_base.merge(
    qc_outcome_columns,
    on="observation_id",
    how="left",
    validate="one_to_one",
)

# -----------------------------------------------------------------------------
# Derived QC flags
# -----------------------------------------------------------------------------

qc_df["primary_exposure_available"] = (
    qc_df["lexical_instability_exposure_df20"].notna()
)

qc_df["primary_zero_eligible"] = (
    qc_df["eligible_token_count_df20"] == 0
)

qc_df["df50_exposure_available"] = (
    qc_df["lexical_instability_exposure_df50"].notna()
)

qc_df["df50_zero_eligible"] = (
    qc_df["eligible_token_count_df50"] == 0
)

# -----------------------------------------------------------------------------
# Basic QC integrity
# -----------------------------------------------------------------------------

checks_stage3 = {}

print("-" * 120)
print("QC MERGE INTEGRITY")
print("-" * 120)

checks_stage3["qc_rows_20002"] = print_check(
    "qc_rows_20002",
    len(qc_df) == 20002
)

checks_stage3["qc_ids_unique"] = print_check(
    "qc_ids_unique",
    qc_df["observation_id"].is_unique
)

checks_stage3["correctness_complete"] = print_check(
    "correctness_complete",
    qc_df["correct_day24"].notna().all()
    and qc_df["error_day24"].notna().all()
)

checks_stage3["correctness_binary"] = print_check(
    "correctness_binary",
    set(qc_df["error_day24"].astype(int).unique()).issubset({0, 1})
)

checks_stage3["primary_availability_matches_zero_rule"] = print_check(
    "primary_availability_matches_zero_rule",
    np.array_equal(
        qc_df["primary_exposure_available"].to_numpy(),
        (~qc_df["primary_zero_eligible"]).to_numpy(),
    )
)

checks_stage3["df50_availability_matches_zero_rule"] = print_check(
    "df50_availability_matches_zero_rule",
    np.array_equal(
        qc_df["df50_exposure_available"].to_numpy(),
        (~qc_df["df50_zero_eligible"]).to_numpy(),
    )
)

# -----------------------------------------------------------------------------
# Overall availability
# -----------------------------------------------------------------------------

n_total = len(qc_df)

n_primary_available = int(
    qc_df["primary_exposure_available"].sum()
)

n_primary_missing = int(
    (~qc_df["primary_exposure_available"]).sum()
)

n_df50_available = int(
    qc_df["df50_exposure_available"].sum()
)

n_df50_missing = int(
    (~qc_df["df50_exposure_available"]).sum()
)

print("-" * 120)
print("OVERALL EXPOSURE AVAILABILITY")
print("-" * 120)

print(f"Total future observations      : {n_total:,}")
print(f"Primary df>=20 available       : {n_primary_available:,}")
print(f"Primary df>=20 missing         : {n_primary_missing:,}")
print(f"Primary availability fraction  : {n_primary_available / n_total:.6f}")

print()

print(f"Secondary df>=50 available     : {n_df50_available:,}")
print(f"Secondary df>=50 missing       : {n_df50_missing:,}")
print(f"Secondary availability fraction: {n_df50_available / n_total:.6f}")

# -----------------------------------------------------------------------------
# Eligible-token count distribution
# -----------------------------------------------------------------------------

print("-" * 120)
print("PRIMARY ELIGIBLE-TOKEN COUNT DISTRIBUTION")
print("-" * 120)

token_count_summary = (
    qc_df["eligible_token_count_df20"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

print(token_count_summary.to_string())

# -----------------------------------------------------------------------------
# Helper to construct availability table
# -----------------------------------------------------------------------------

def availability_table(df, group_col):
    out = (
        df.groupby(group_col, dropna=False)
        .agg(
            n=("observation_id", "size"),
            primary_available_n=(
                "primary_exposure_available",
                "sum",
            ),
            primary_zero_eligible_n=(
                "primary_zero_eligible",
                "sum",
            ),
            df50_available_n=(
                "df50_exposure_available",
                "sum",
            ),
            df50_zero_eligible_n=(
                "df50_zero_eligible",
                "sum",
            ),
            median_primary_eligible_tokens=(
                "eligible_token_count_df20",
                "median",
            ),
            mean_primary_eligible_tokens=(
                "eligible_token_count_df20",
                "mean",
            ),
        )
        .reset_index()
    )

    out["primary_available_fraction"] = (
        out["primary_available_n"] / out["n"]
    )

    out["df50_available_fraction"] = (
        out["df50_available_n"] / out["n"]
    )

    return out


# -----------------------------------------------------------------------------
# Period QC
# -----------------------------------------------------------------------------

period_qc = availability_table(
    qc_df,
    "period",
)

period_qc = (
    period_qc
    .sort_values("period")
    .reset_index(drop=True)
)

print("-" * 120)
print("COVERAGE BY FUTURE PERIOD")
print("-" * 120)

print(period_qc.to_string(index=False))

# -----------------------------------------------------------------------------
# True-label QC
# -----------------------------------------------------------------------------

class_qc = availability_table(
    qc_df,
    "y_true",
)

class_qc = (
    class_qc
    .sort_values("y_true")
    .reset_index(drop=True)
)

print("-" * 120)
print("COVERAGE BY TRUE LABEL")
print("-" * 120)

print(class_qc.to_string(index=False))

# -----------------------------------------------------------------------------
# Source QC
# -----------------------------------------------------------------------------

source_qc = availability_table(
    qc_df,
    "source",
)

source_qc = (
    source_qc
    .sort_values(
        ["n", "source"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

print("-" * 120)
print("COVERAGE BY SOURCE")
print("-" * 120)

print(source_qc.to_string(index=False))

# -----------------------------------------------------------------------------
# Correctness QC
#
# Diagnostic availability only.
# DO NOT compute mean exposure by correctness.
# -----------------------------------------------------------------------------

correctness_qc = availability_table(
    qc_df,
    "error_day24",
)

correctness_qc["group"] = correctness_qc[
    "error_day24"
].map({
    0: "correct",
    1: "error",
})

correctness_qc = correctness_qc[
    [
        "group",
        "error_day24",
        "n",
        "primary_available_n",
        "primary_zero_eligible_n",
        "primary_available_fraction",
        "df50_available_n",
        "df50_zero_eligible_n",
        "df50_available_fraction",
        "median_primary_eligible_tokens",
        "mean_primary_eligible_tokens",
    ]
]

print("-" * 120)
print("COVERAGE BY CORRECTNESS — DIAGNOSTIC ONLY")
print("-" * 120)

print(correctness_qc.to_string(index=False))

# -----------------------------------------------------------------------------
# Build canonical QC artifact
#
# One row per diagnostic group.
# No mean exposure contrast is included.
# -----------------------------------------------------------------------------

def add_scope(df, scope):
    out = df.copy()
    out.insert(0, "scope", scope)
    return out


period_qc_artifact = period_qc.rename(
    columns={"period": "group"}
)

class_qc_artifact = class_qc.rename(
    columns={"y_true": "group"}
)

source_qc_artifact = source_qc.rename(
    columns={"source": "group"}
)

correctness_qc_artifact = correctness_qc[
    [
        "group",
        "n",
        "primary_available_n",
        "primary_zero_eligible_n",
        "primary_available_fraction",
        "df50_available_n",
        "df50_zero_eligible_n",
        "df50_available_fraction",
        "median_primary_eligible_tokens",
        "mean_primary_eligible_tokens",
    ]
]

qc_artifact = pd.concat(
    [
        add_scope(
            period_qc_artifact,
            "period",
        ),
        add_scope(
            class_qc_artifact,
            "true_label",
        ),
        add_scope(
            source_qc_artifact,
            "source",
        ),
        add_scope(
            correctness_qc_artifact,
            "correctness",
        ),
    ],
    ignore_index=True,
    sort=False,
)

# -----------------------------------------------------------------------------
# Ensure primary effect estimates are absent from QC artifact
# -----------------------------------------------------------------------------

forbidden_qc_columns = {
    "mean_exposure",
    "mean_exposure_error",
    "mean_exposure_correct",
    "delta_primary",
    "bootstrap_ci_low",
    "bootstrap_ci_high",
}

checks_stage3["qc_contains_no_primary_effect_columns"] = print_check(
    "qc_contains_no_primary_effect_columns",
    forbidden_qc_columns.isdisjoint(
        set(qc_artifact.columns)
    )
)

# -----------------------------------------------------------------------------
# Save QC artifact
# -----------------------------------------------------------------------------

qc_artifact.to_csv(
    DAY27_EXPOSURE_QC_CSV,
    index=False,
)

qc_reload = pd.read_csv(
    DAY27_EXPOSURE_QC_CSV
)

checks_stage3["qc_artifact_exists"] = print_check(
    "qc_artifact_exists",
    DAY27_EXPOSURE_QC_CSV.exists()
)

checks_stage3["qc_artifact_nonempty"] = print_check(
    "qc_artifact_nonempty",
    DAY27_EXPOSURE_QC_CSV.stat().st_size > 0
)

checks_stage3["qc_artifact_roundtrip_rows"] = print_check(
    "qc_artifact_roundtrip_rows",
    len(qc_reload) == len(qc_artifact)
)

# -----------------------------------------------------------------------------
# Missingness assessment
# -----------------------------------------------------------------------------

primary_missing_fraction = (
    n_primary_missing / n_total
)

checks_stage3["primary_missing_fraction_below_1_percent"] = print_check(
    "primary_missing_fraction_below_1_percent",
    primary_missing_fraction < 0.01
)

# -----------------------------------------------------------------------------
# Freeze QC identity
# -----------------------------------------------------------------------------

day27_exposure_qc_sha256 = sha256_file(
    DAY27_EXPOSURE_QC_CSV
)

print("-" * 120)
print("FROZEN EXPOSURE QC ARTIFACT")
print("-" * 120)

print(f"Path   : {DAY27_EXPOSURE_QC_CSV}")
print(f"Size   : {DAY27_EXPOSURE_QC_CSV.stat().st_size:,} bytes")
print(f"SHA256 : {day27_exposure_qc_sha256}")

# -----------------------------------------------------------------------------
# Final Stage 3 gate
# -----------------------------------------------------------------------------

all_stage3_checks_pass = all(
    checks_stage3.values()
)

print("-" * 120)

print_check(
    "DAY27_STAGE3_ALL_PASS",
    all_stage3_checks_pass
)

if not all_stage3_checks_pass:
    raise RuntimeError(
        "DAY 27 INTEGRITY STOP: exposure QC failed."
    )

print("=" * 120)
print("DAY 27 STAGE 3: PASS — EXPOSURE QC COMPLETE")
print(f"PRIMARY EXPOSURE MISSING: {n_primary_missing:,} / {n_total:,}")
print("NO EXPOSURE DEFINITION WAS MODIFIED")
print("NO PRIMARY ERROR-vs-EXPOSURE CONTRAST HAS BEEN COMPUTED")
print("PRIMARY H1 STATUS: STILL UNTESTED")
print("READY — STAGE 4 PRIMARY HYPOTHESIS TEST")
print("=" * 120)

DAY 27 — STAGE 3 EXPOSURE QC / COVERAGE AUDIT
PASS   — exposure_hash_still_frozen
------------------------------------------------------------------------------------------------------------------------
QC MERGE INTEGRITY
------------------------------------------------------------------------------------------------------------------------
PASS   — qc_rows_20002
PASS   — qc_ids_unique
PASS   — correctness_complete
PASS   — correctness_binary
PASS   — primary_availability_matches_zero_rule
PASS   — df50_availability_matches_zero_rule
------------------------------------------------------------------------------------------------------------------------
OVERALL EXPOSURE AVAILABILITY
------------------------------------------------------------------------------------------------------------------------
Total future observations      : 20,002
Primary df>=20 available       : 20,001
Primary df>=20 missing         : 1
Primary availability fraction  : 0.999950

Secondary df>=50 available    

## Stage 4 — Primary Out-of-Time Hypothesis Test

The Day 27 protocol, future population, frozen lexical-instability exposure, and exposure QC have all passed integrity checks.

The primary hypothesis is now unlocked.

### Primary estimand

\[
\Delta_{\mathrm{primary}}
=
\operatorname{mean}(E_i \mid \mathrm{error}_i=1)
-
\operatorname{mean}(E_i \mid \mathrm{error}_i=0),
\]

where \(E_i\) is the frozen Day 26 historical lexical-instability exposure.

The primary population consists of all future F1–F6 observations with non-missing primary df ≥ 20 exposure.

### Pre-registered decision rule

A 1,000-replicate observation-level bootstrap with seed 42 will be used to estimate a two-sided 95% percentile confidence interval.

- **Gate A — Supported:**  
  \(\Delta_{\mathrm{primary}} > 0\) and the entire 95% CI is above 0.

- **Gate B — Suggestive:**  
  \(\Delta_{\mathrm{primary}} > 0\), but the 95% CI crosses 0.

- **Gate C — Failed:**  
  \(\Delta_{\mathrm{primary}} \le 0\), or there is no meaningful positive association under the frozen primary definition.

Secondary analyses cannot override this primary gate.

In [15]:
# =============================================================================
# Day 27 — Stage 4A
# Primary error-vs-correct lexical-instability contrast
# Canonical retained cell
# =============================================================================

print("=" * 120)
print("DAY 27 — STAGE 4A PRIMARY ERROR-vs-CORRECT EXPOSURE CONTRAST")
print("=" * 120)

# -----------------------------------------------------------------------------
# Re-verify frozen exposure artifact before unlocking primary analysis
# -----------------------------------------------------------------------------

current_exposure_sha256_stage4 = sha256_file(
    DAY27_EXPOSURE_CSV
)

print_check(
    "exposure_hash_matches_frozen_identity",
    current_exposure_sha256_stage4
    == day27_frozen_exposure_identity["sha256"]
)

if (
    current_exposure_sha256_stage4
    != day27_frozen_exposure_identity["sha256"]
):
    raise RuntimeError(
        "DAY 27 INTEGRITY STOP: exposure artifact changed before primary test."
    )

# -----------------------------------------------------------------------------
# Reload frozen exposure artifact
# -----------------------------------------------------------------------------

primary_exposure_df = pd.read_csv(
    DAY27_EXPOSURE_CSV,
    dtype={"observation_id": str}
)

# -----------------------------------------------------------------------------
# Merge frozen exposure with frozen Day24 correctness
#
# This is the FIRST primary outcome/exposure merge for effect estimation.
# -----------------------------------------------------------------------------

primary_analysis_df = primary_exposure_df.merge(
    future_df[
        [
            "observation_id",
            "y_pred_day24",
            "correct_day24",
            "error_day24",
        ]
    ],
    on="observation_id",
    how="left",
    validate="one_to_one",
)

# -----------------------------------------------------------------------------
# Primary analysis population
# Frozen rule: all F1-F6 observations with non-missing df>=20 exposure
# -----------------------------------------------------------------------------

primary_complete = primary_analysis_df.loc[
    primary_analysis_df[
        "lexical_instability_exposure_df20"
    ].notna()
].copy()

primary_complete["error_day24"] = (
    primary_complete["error_day24"].astype(int)
)

# -----------------------------------------------------------------------------
# Integrity checks
# -----------------------------------------------------------------------------

checks_stage4a = {}

print("-" * 120)
print("PRIMARY ANALYSIS POPULATION")
print("-" * 120)

checks_stage4a["primary_rows_20001"] = print_check(
    "primary_rows_20001",
    len(primary_complete) == 20001
)

checks_stage4a["exactly_one_excluded_for_missing_exposure"] = print_check(
    "exactly_one_excluded_for_missing_exposure",
    len(primary_analysis_df) - len(primary_complete) == 1
)

checks_stage4a["primary_ids_unique"] = print_check(
    "primary_ids_unique",
    primary_complete["observation_id"].is_unique
)

checks_stage4a["error_complete"] = print_check(
    "error_complete",
    primary_complete["error_day24"].notna().all()
)

checks_stage4a["error_binary"] = print_check(
    "error_binary",
    set(primary_complete["error_day24"].unique()) == {0, 1}
)

# -----------------------------------------------------------------------------
# Split frozen primary population
# -----------------------------------------------------------------------------

correct_group = primary_complete.loc[
    primary_complete["error_day24"] == 0,
    "lexical_instability_exposure_df20",
].to_numpy(dtype=float)

error_group = primary_complete.loc[
    primary_complete["error_day24"] == 1,
    "lexical_instability_exposure_df20",
].to_numpy(dtype=float)

n_correct_primary = len(correct_group)
n_error_primary = len(error_group)

mean_correct = float(np.mean(correct_group))
mean_error = float(np.mean(error_group))

median_correct = float(np.median(correct_group))
median_error = float(np.median(error_group))

delta_primary = float(
    mean_error - mean_correct
)

# -----------------------------------------------------------------------------
# Secondary descriptive effect size:
# pooled-SD standardized mean difference
# -----------------------------------------------------------------------------

sd_correct = float(
    np.std(correct_group, ddof=1)
)

sd_error = float(
    np.std(error_group, ddof=1)
)

pooled_sd = np.sqrt(
    (
        (n_correct_primary - 1) * sd_correct**2
        + (n_error_primary - 1) * sd_error**2
    )
    /
    (
        n_correct_primary
        + n_error_primary
        - 2
    )
)

standardized_mean_difference = float(
    delta_primary / pooled_sd
)

# -----------------------------------------------------------------------------
# Save POINT ESTIMATE artifact before bootstrap
# -----------------------------------------------------------------------------

day27_primary_point_estimate = {
    "day": 27,
    "artifact": "day27_primary_point_estimate",
    "status": "POINT_ESTIMATE_FROZEN_BEFORE_BOOTSTRAP",

    "population": {
        "n_total_future": int(len(primary_analysis_df)),
        "n_primary_complete": int(len(primary_complete)),
        "n_missing_primary_exposure": int(
            len(primary_analysis_df) - len(primary_complete)
        ),
        "n_correct": int(n_correct_primary),
        "n_error": int(n_error_primary),
    },

    "exposure": {
        "field": "lexical_instability_exposure_df20",
        "rule": "mean I(v) across frozen df>=20 tokens present",
        "artifact_sha256": current_exposure_sha256_stage4,
    },

    "descriptive_statistics": {
        "mean_correct": mean_correct,
        "mean_error": mean_error,
        "median_correct": median_correct,
        "median_error": median_error,
        "sd_correct": sd_correct,
        "sd_error": sd_error,
    },

    "primary_estimand": {
        "name": "delta_primary",
        "definition": "mean(error exposure) - mean(correct exposure)",
        "value": delta_primary,
    },

    "secondary_effect_size": {
        "name": "standardized_mean_difference",
        "value": standardized_mean_difference,
    },

    "bootstrap_status": "NOT_YET_RUN",

    "secondary_analyses_viewed": False,

    "created_utc": datetime.now(timezone.utc).isoformat(),
}

DAY27_PRIMARY_POINT_ESTIMATE_JSON = (
    DAY27_DIR / "day27_primary_point_estimate.json"
)

save_json(
    day27_primary_point_estimate,
    DAY27_PRIMARY_POINT_ESTIMATE_JSON
)

primary_point_reload = load_json(
    DAY27_PRIMARY_POINT_ESTIMATE_JSON
)

checks_stage4a["point_estimate_artifact_exists"] = print_check(
    "point_estimate_artifact_exists",
    DAY27_PRIMARY_POINT_ESTIMATE_JSON.exists()
)

checks_stage4a["point_estimate_artifact_nonempty"] = print_check(
    "point_estimate_artifact_nonempty",
    DAY27_PRIMARY_POINT_ESTIMATE_JSON.stat().st_size > 0
)

checks_stage4a["point_estimate_roundtrip_exact"] = print_check(
    "point_estimate_roundtrip_exact",
    primary_point_reload == day27_primary_point_estimate
)

checks_stage4a["bootstrap_not_yet_run"] = print_check(
    "bootstrap_not_yet_run",
    primary_point_reload["bootstrap_status"] == "NOT_YET_RUN"
)

checks_stage4a["secondary_not_viewed"] = print_check(
    "secondary_not_viewed",
    primary_point_reload["secondary_analyses_viewed"] is False
)

# -----------------------------------------------------------------------------
# Primary point estimate output
# -----------------------------------------------------------------------------

print("-" * 120)
print("PRIMARY POINT ESTIMATE")
print("-" * 120)

print(f"Primary analysis n     : {len(primary_complete):,}")
print(f"Correct n              : {n_correct_primary:,}")
print(f"Error n                : {n_error_primary:,}")

print()

print(f"Mean exposure — correct: {mean_correct:.9f}")
print(f"Mean exposure — error  : {mean_error:.9f}")

print()

print(f"Median — correct       : {median_correct:.9f}")
print(f"Median — error         : {median_error:.9f}")

print()

print(f"Delta_primary          : {delta_primary:+.9f}")
print(
    f"Standardized mean diff : "
    f"{standardized_mean_difference:+.6f}"
)

# -----------------------------------------------------------------------------
# Freeze point-estimate identity
# -----------------------------------------------------------------------------

day27_primary_point_estimate_sha256 = sha256_file(
    DAY27_PRIMARY_POINT_ESTIMATE_JSON
)

print("-" * 120)
print("FROZEN PRIMARY POINT ESTIMATE")
print("-" * 120)

print(f"Path   : {DAY27_PRIMARY_POINT_ESTIMATE_JSON}")
print(
    f"Size   : "
    f"{DAY27_PRIMARY_POINT_ESTIMATE_JSON.stat().st_size:,} bytes"
)
print(f"SHA256 : {day27_primary_point_estimate_sha256}")

# -----------------------------------------------------------------------------
# Final Stage 4A gate
# -----------------------------------------------------------------------------

all_stage4a_checks_pass = all(
    checks_stage4a.values()
)

print("-" * 120)

print_check(
    "DAY27_STAGE4A_ALL_PASS",
    all_stage4a_checks_pass
)

if not all_stage4a_checks_pass:
    raise RuntimeError(
        "DAY 27 INTEGRITY STOP: primary point-estimate freeze failed."
    )

print("=" * 120)
print("DAY 27 STAGE 4A: PASS — PRIMARY POINT ESTIMATE FROZEN")
print("BOOTSTRAP CI HAS NOT YET BEEN COMPUTED")
print("SECONDARY ANALYSES HAVE NOT BEEN VIEWED")
print("READY — STAGE 4B PRE-REGISTERED 1,000-REPLICATE BOOTSTRAP")
print("=" * 120)

DAY 27 — STAGE 4A PRIMARY ERROR-vs-CORRECT EXPOSURE CONTRAST
PASS   — exposure_hash_matches_frozen_identity
------------------------------------------------------------------------------------------------------------------------
PRIMARY ANALYSIS POPULATION
------------------------------------------------------------------------------------------------------------------------
PASS   — primary_rows_20001
PASS   — exactly_one_excluded_for_missing_exposure
PASS   — primary_ids_unique
PASS   — error_complete
PASS   — error_binary
PASS   — point_estimate_artifact_exists
PASS   — point_estimate_artifact_nonempty
PASS   — point_estimate_roundtrip_exact
PASS   — bootstrap_not_yet_run
PASS   — secondary_not_viewed
------------------------------------------------------------------------------------------------------------------------
PRIMARY POINT ESTIMATE
------------------------------------------------------------------------------------------------------------------------
Primary analysis n   

In [16]:
# =============================================================================
# Day 27 — Stage 4B
# Pre-registered 1,000-replicate bootstrap + primary decision gate
# Canonical retained cell
# =============================================================================

print("=" * 120)
print("DAY 27 — STAGE 4B PRE-REGISTERED PRIMARY BOOTSTRAP TEST")
print("=" * 120)

# -----------------------------------------------------------------------------
# Re-verify frozen point estimate artifact
# -----------------------------------------------------------------------------

current_point_estimate_sha256 = sha256_file(
    DAY27_PRIMARY_POINT_ESTIMATE_JSON
)

print_check(
    "primary_point_estimate_hash_still_frozen",
    current_point_estimate_sha256
    == day27_primary_point_estimate_sha256
)

if (
    current_point_estimate_sha256
    != day27_primary_point_estimate_sha256
):
    raise RuntimeError(
        "DAY 27 INTEGRITY STOP: primary point-estimate artifact changed "
        "before bootstrap."
    )

# -----------------------------------------------------------------------------
# Frozen bootstrap protocol
# -----------------------------------------------------------------------------

BOOTSTRAP_B = int(
    day27_analysis_manifest["primary_inference"][
        "bootstrap_replicates"
    ]
)

BOOTSTRAP_SEED = int(
    day27_analysis_manifest["primary_inference"][
        "bootstrap_seed"
    ]
)

print("-" * 120)
print("FROZEN BOOTSTRAP PROTOCOL")
print("-" * 120)

print(f"Replicates : {BOOTSTRAP_B}")
print(f"Seed       : {BOOTSTRAP_SEED}")
print("Level      : observation")
print("CI         : two-sided 95% percentile")
print("Estimand   : mean(error exposure) - mean(correct exposure)")

# -----------------------------------------------------------------------------
# Bootstrap arrays
# -----------------------------------------------------------------------------

bootstrap_exposure = primary_complete[
    "lexical_instability_exposure_df20"
].to_numpy(dtype=float)

bootstrap_error = primary_complete[
    "error_day24"
].to_numpy(dtype=int)

n_bootstrap_population = len(
    bootstrap_exposure
)

# -----------------------------------------------------------------------------
# Integrity
# -----------------------------------------------------------------------------

checks_stage4b = {}

checks_stage4b["bootstrap_population_20001"] = print_check(
    "bootstrap_population_20001",
    n_bootstrap_population == 20001
)

checks_stage4b["bootstrap_replicates_1000"] = print_check(
    "bootstrap_replicates_1000",
    BOOTSTRAP_B == 1000
)

checks_stage4b["bootstrap_seed_42"] = print_check(
    "bootstrap_seed_42",
    BOOTSTRAP_SEED == 42
)

checks_stage4b["bootstrap_exposure_complete"] = print_check(
    "bootstrap_exposure_complete",
    np.isfinite(bootstrap_exposure).all()
)

checks_stage4b["bootstrap_error_binary"] = print_check(
    "bootstrap_error_binary",
    set(np.unique(bootstrap_error)) == {0, 1}
)

# -----------------------------------------------------------------------------
# Observation-level bootstrap
# -----------------------------------------------------------------------------

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

bootstrap_delta = np.empty(
    BOOTSTRAP_B,
    dtype=float
)

bootstrap_n_correct = np.empty(
    BOOTSTRAP_B,
    dtype=int
)

bootstrap_n_error = np.empty(
    BOOTSTRAP_B,
    dtype=int
)

for b in range(BOOTSTRAP_B):

    sampled_idx = rng.integers(
        low=0,
        high=n_bootstrap_population,
        size=n_bootstrap_population,
    )

    sampled_exposure = bootstrap_exposure[
        sampled_idx
    ]

    sampled_error = bootstrap_error[
        sampled_idx
    ]

    sampled_correct_mask = (
        sampled_error == 0
    )

    sampled_error_mask = (
        sampled_error == 1
    )

    n_correct_b = int(
        sampled_correct_mask.sum()
    )

    n_error_b = int(
        sampled_error_mask.sum()
    )

    # With this sample size, absence of either group is practically impossible.
    # Still enforce the estimand definition explicitly.
    if (
        n_correct_b == 0
        or n_error_b == 0
    ):
        bootstrap_delta[b] = np.nan
        bootstrap_n_correct[b] = n_correct_b
        bootstrap_n_error[b] = n_error_b
        continue

    mean_correct_b = float(
        sampled_exposure[
            sampled_correct_mask
        ].mean()
    )

    mean_error_b = float(
        sampled_exposure[
            sampled_error_mask
        ].mean()
    )

    bootstrap_delta[b] = (
        mean_error_b
        - mean_correct_b
    )

    bootstrap_n_correct[b] = (
        n_correct_b
    )

    bootstrap_n_error[b] = (
        n_error_b
    )

# -----------------------------------------------------------------------------
# Bootstrap integrity
# -----------------------------------------------------------------------------

valid_bootstrap_mask = np.isfinite(
    bootstrap_delta
)

n_valid_bootstrap = int(
    valid_bootstrap_mask.sum()
)

checks_stage4b["all_1000_bootstrap_replicates_valid"] = print_check(
    "all_1000_bootstrap_replicates_valid",
    n_valid_bootstrap == 1000
)

if n_valid_bootstrap != BOOTSTRAP_B:
    raise RuntimeError(
        "DAY 27 INTEGRITY STOP: one or more bootstrap replicates "
        "did not contain both outcome groups."
    )

# -----------------------------------------------------------------------------
# Pre-registered percentile CI
# -----------------------------------------------------------------------------

ci_low = float(
    np.percentile(
        bootstrap_delta,
        2.5,
    )
)

ci_high = float(
    np.percentile(
        bootstrap_delta,
        97.5,
    )
)

bootstrap_mean_delta = float(
    np.mean(bootstrap_delta)
)

bootstrap_median_delta = float(
    np.median(bootstrap_delta)
)

bootstrap_sd_delta = float(
    np.std(
        bootstrap_delta,
        ddof=1,
    )
)

# -----------------------------------------------------------------------------
# Frozen primary decision gate
# -----------------------------------------------------------------------------

if (
    delta_primary > 0
    and ci_low > 0
):
    primary_gate = "GATE_A_SUPPORTED"

    primary_interpretation = (
        "H1 supported out-of-time: higher frozen historical "
        "lexical-instability exposure is associated with higher "
        "future classification error."
    )

elif (
    delta_primary > 0
    and ci_low <= 0 <= ci_high
):
    primary_gate = "GATE_B_SUGGESTIVE"

    primary_interpretation = (
        "The primary effect is directionally positive but statistically "
        "inconclusive under the pre-registered bootstrap criterion."
    )

else:
    primary_gate = "GATE_C_FAILED"

    primary_interpretation = (
        "The frozen historical lexical-instability exposure does not "
        "explain higher future classification error under the "
        "pre-registered primary Day27 definition."
    )

# -----------------------------------------------------------------------------
# Save bootstrap replicate artifact
# -----------------------------------------------------------------------------

bootstrap_df = pd.DataFrame({
    "bootstrap_replicate": np.arange(
        1,
        BOOTSTRAP_B + 1,
        dtype=int,
    ),
    "delta_primary": bootstrap_delta,
    "n_correct": bootstrap_n_correct,
    "n_error": bootstrap_n_error,
})

bootstrap_df.to_csv(
    DAY27_BOOTSTRAP_CSV,
    index=False,
)

# -----------------------------------------------------------------------------
# Save formal primary hypothesis test artifact
# -----------------------------------------------------------------------------

day27_primary_hypothesis_test = {
    "day": 27,
    "artifact": "day27_primary_hypothesis_test",
    "status": "PRIMARY_TEST_COMPLETE_AND_FROZEN",

    "population": {
        "n_total_future": int(
            len(primary_analysis_df)
        ),
        "n_primary_complete": int(
            len(primary_complete)
        ),
        "n_missing_primary_exposure": int(
            len(primary_analysis_df)
            - len(primary_complete)
        ),
        "n_correct": int(
            n_correct_primary
        ),
        "n_error": int(
            n_error_primary
        ),
    },

    "primary_exposure": {
        "field": "lexical_instability_exposure_df20",
        "rule": (
            "mean I(v) across eligible frozen df>=20 "
            "historical-vocabulary tokens present"
        ),
        "artifact_sha256": (
            current_exposure_sha256_stage4
        ),
    },

    "primary_estimand": {
        "definition": (
            "mean(exposure | error=1) "
            "- mean(exposure | error=0)"
        ),
        "mean_correct": mean_correct,
        "mean_error": mean_error,
        "median_correct": median_correct,
        "median_error": median_error,
        "delta_primary": delta_primary,
        "standardized_mean_difference": (
            standardized_mean_difference
        ),
    },

    "bootstrap": {
        "level": "observation",
        "replicates": int(
            BOOTSTRAP_B
        ),
        "seed": int(
            BOOTSTRAP_SEED
        ),
        "valid_replicates": int(
            n_valid_bootstrap
        ),
        "confidence_interval": (
            "two-sided 95% percentile"
        ),
        "ci_low": ci_low,
        "ci_high": ci_high,
        "bootstrap_mean_delta": (
            bootstrap_mean_delta
        ),
        "bootstrap_median_delta": (
            bootstrap_median_delta
        ),
        "bootstrap_sd_delta": (
            bootstrap_sd_delta
        ),
    },

    "decision_gate": {
        "gate": primary_gate,
        "rule_A": (
            "delta_primary > 0 and 95% CI entirely > 0"
        ),
        "rule_B": (
            "delta_primary > 0 but 95% CI crosses 0"
        ),
        "rule_C": (
            "delta_primary <= 0 or no meaningful "
            "positive association"
        ),
        "interpretation": (
            primary_interpretation
        ),
    },

    "scientific_constraints": {
        "secondary_analyses_may_override_primary": False,
        "causal_claim_permitted": False,
        "exposure_redefinition_after_result_permitted": False,
    },

    "secondary_analyses_viewed_before_primary_freeze": False,

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

save_json(
    day27_primary_hypothesis_test,
    DAY27_PRIMARY_TEST_JSON,
)

# -----------------------------------------------------------------------------
# Deterministic reload audits
# -----------------------------------------------------------------------------

primary_test_reload = load_json(
    DAY27_PRIMARY_TEST_JSON
)

bootstrap_reload = pd.read_csv(
    DAY27_BOOTSTRAP_CSV
)

checks_stage4b["primary_test_artifact_exists"] = print_check(
    "primary_test_artifact_exists",
    DAY27_PRIMARY_TEST_JSON.exists()
)

checks_stage4b["primary_test_artifact_nonempty"] = print_check(
    "primary_test_artifact_nonempty",
    DAY27_PRIMARY_TEST_JSON.stat().st_size > 0
)

checks_stage4b["primary_test_roundtrip_exact"] = print_check(
    "primary_test_roundtrip_exact",
    primary_test_reload
    == day27_primary_hypothesis_test
)

checks_stage4b["bootstrap_artifact_exists"] = print_check(
    "bootstrap_artifact_exists",
    DAY27_BOOTSTRAP_CSV.exists()
)

checks_stage4b["bootstrap_artifact_nonempty"] = print_check(
    "bootstrap_artifact_nonempty",
    DAY27_BOOTSTRAP_CSV.stat().st_size > 0
)

checks_stage4b["bootstrap_rows_1000"] = print_check(
    "bootstrap_rows_1000",
    len(bootstrap_reload) == 1000
)

checks_stage4b["primary_result_saved_before_secondary"] = print_check(
    "primary_result_saved_before_secondary",
    primary_test_reload[
        "secondary_analyses_viewed_before_primary_freeze"
    ] is False
)

# -----------------------------------------------------------------------------
# Artifact hashes
# -----------------------------------------------------------------------------

day27_primary_test_sha256 = sha256_file(
    DAY27_PRIMARY_TEST_JSON
)

day27_bootstrap_sha256 = sha256_file(
    DAY27_BOOTSTRAP_CSV
)

# -----------------------------------------------------------------------------
# Final primary result
# -----------------------------------------------------------------------------

print("-" * 120)
print("PRIMARY OUT-OF-TIME HYPOTHESIS TEST")
print("-" * 120)

print(
    f"Mean exposure — correct       : "
    f"{mean_correct:.9f}"
)

print(
    f"Mean exposure — error         : "
    f"{mean_error:.9f}"
)

print(
    f"Delta_primary                 : "
    f"{delta_primary:+.9f}"
)

print(
    f"Standardized mean difference  : "
    f"{standardized_mean_difference:+.6f}"
)

print()

print(
    f"Bootstrap mean delta          : "
    f"{bootstrap_mean_delta:+.9f}"
)

print(
    f"Bootstrap median delta        : "
    f"{bootstrap_median_delta:+.9f}"
)

print(
    f"Bootstrap SD                  : "
    f"{bootstrap_sd_delta:.9f}"
)

print(
    f"95% percentile CI             : "
    f"[{ci_low:+.9f}, {ci_high:+.9f}]"
)

print("-" * 120)
print("PRE-REGISTERED PRIMARY DECISION")
print("-" * 120)

print(f"Gate : {primary_gate}")
print()
print(primary_interpretation)

# -----------------------------------------------------------------------------
# Artifact identity
# -----------------------------------------------------------------------------

print("-" * 120)
print("FROZEN PRIMARY ARTIFACTS")
print("-" * 120)

print(
    f"Primary test JSON SHA256 : "
    f"{day27_primary_test_sha256}"
)

print(
    f"Bootstrap CSV SHA256     : "
    f"{day27_bootstrap_sha256}"
)

# -----------------------------------------------------------------------------
# Final Stage 4B gate
# -----------------------------------------------------------------------------

all_stage4b_checks_pass = all(
    checks_stage4b.values()
)

print("-" * 120)

print_check(
    "DAY27_STAGE4B_ALL_PASS",
    all_stage4b_checks_pass
)

if not all_stage4b_checks_pass:
    raise RuntimeError(
        "DAY 27 INTEGRITY STOP: primary bootstrap / gate freeze failed."
    )

print("=" * 120)
print("DAY 27 STAGE 4: COMPLETE — PRIMARY RESULT FROZEN")
print(f"PRIMARY DECISION: {primary_gate}")
print("SECONDARY ANALYSES MAY NOW BEGIN")
print("SECONDARY RESULTS CANNOT OVERRIDE THE PRIMARY GATE")
print("=" * 120)

DAY 27 — STAGE 4B PRE-REGISTERED PRIMARY BOOTSTRAP TEST
PASS   — primary_point_estimate_hash_still_frozen
------------------------------------------------------------------------------------------------------------------------
FROZEN BOOTSTRAP PROTOCOL
------------------------------------------------------------------------------------------------------------------------
Replicates : 1000
Seed       : 42
Level      : observation
CI         : two-sided 95% percentile
Estimand   : mean(error exposure) - mean(correct exposure)
PASS   — bootstrap_population_20001
PASS   — bootstrap_replicates_1000
PASS   — bootstrap_seed_42
PASS   — bootstrap_exposure_complete
PASS   — bootstrap_error_binary
PASS   — all_1000_bootstrap_replicates_valid
PASS   — primary_test_artifact_exists
PASS   — primary_test_artifact_nonempty
PASS   — primary_test_roundtrip_exact
PASS   — bootstrap_artifact_exists
PASS   — bootstrap_artifact_nonempty
PASS   — bootstrap_rows_1000
PASS   — primary_result_saved_before_seco

## Stage 5 — Pre-Specified Secondary Analyses

The primary Day 27 hypothesis test is now frozen.

### Primary result

\[
\Delta_{\mathrm{primary}}
=
-0.005388
\]

with a 95% percentile bootstrap confidence interval

\[
[-0.010077,\,-0.000992].
\]

Therefore,

\[
\boxed{\text{Primary Gate = GATE C — FAILED}}
\]

under the frozen Day 27 definition.

The historical-only lexical-instability exposure does **not** explain higher all-future classification error. The observed primary association is slightly negative rather than positive.

---

### Role of Stage 5

All analyses below were pre-specified before the primary result was observed.

They are used only to characterize heterogeneity and possible mechanisms.

They **cannot override the failed primary gate**.

The secondary analyses are:

1. F1–F6 period-specific exposure contrasts;
2. F3-specific exposure contrast;
3. F3 true-positive \(P\rightarrow N\) versus \(P\rightarrow P\);
4. within-source contrasts;
5. class-conditioned contrasts;
6. df ≥ 50 sensitivity analysis;
7. secondary logistic-regression corroboration.

The interpretation boundary remains:

\[
\text{secondary heterogeneity}
\neq
\text{primary hypothesis rescue}.
\]

In [17]:
# =============================================================================
# Day 27 — Stage 5A
# F1-F6 period-specific and F3-specific exposure contrasts
# Canonical retained cell
# =============================================================================

print("=" * 120)
print("DAY 27 — STAGE 5A PERIOD-SPECIFIC AND F3 EXPOSURE CONTRASTS")
print("=" * 120)

# -----------------------------------------------------------------------------
# Primary result must already be frozen
# -----------------------------------------------------------------------------

primary_test_frozen = load_json(
    DAY27_PRIMARY_TEST_JSON
)

checks_stage5a = {}

checks_stage5a["primary_test_complete_and_frozen"] = print_check(
    "primary_test_complete_and_frozen",
    primary_test_frozen["status"]
    == "PRIMARY_TEST_COMPLETE_AND_FROZEN"
)

checks_stage5a["primary_gate_is_gate_c_failed"] = print_check(
    "primary_gate_is_gate_c_failed",
    primary_test_frozen["decision_gate"]["gate"]
    == "GATE_C_FAILED"
)

if not all(checks_stage5a.values()):
    raise RuntimeError(
        "DAY 27 STOP: secondary analyses cannot begin before "
        "the primary result is frozen."
    )

# -----------------------------------------------------------------------------
# Use frozen primary-complete population only
# -----------------------------------------------------------------------------

secondary_base = primary_complete.copy()

exposure_col = "lexical_instability_exposure_df20"

# -----------------------------------------------------------------------------
# Helper for descriptive error-vs-correct contrast
# -----------------------------------------------------------------------------

def compute_exposure_contrast(df, scope_name, scope_value):

    correct = df.loc[
        df["error_day24"] == 0,
        exposure_col,
    ].to_numpy(dtype=float)

    error = df.loc[
        df["error_day24"] == 1,
        exposure_col,
    ].to_numpy(dtype=float)

    n_correct = len(correct)
    n_error = len(error)

    mean_correct = (
        float(np.mean(correct))
        if n_correct > 0
        else np.nan
    )

    mean_error = (
        float(np.mean(error))
        if n_error > 0
        else np.nan
    )

    median_correct = (
        float(np.median(correct))
        if n_correct > 0
        else np.nan
    )

    median_error = (
        float(np.median(error))
        if n_error > 0
        else np.nan
    )

    delta = (
        mean_error - mean_correct
        if n_correct > 0 and n_error > 0
        else np.nan
    )

    return {
        "scope": scope_name,
        "group": scope_value,
        "n": int(len(df)),
        "n_correct": int(n_correct),
        "n_error": int(n_error),
        "error_rate": float(n_error / len(df)) if len(df) > 0 else np.nan,
        "mean_exposure_correct": mean_correct,
        "mean_exposure_error": mean_error,
        "median_exposure_correct": median_correct,
        "median_exposure_error": median_error,
        "delta_error_minus_correct": float(delta) if np.isfinite(delta) else np.nan,
    }

# -----------------------------------------------------------------------------
# F1-F6 contrasts
# -----------------------------------------------------------------------------

period_rows = []

for period in DAY27_PROTOCOL["future_periods"]:

    period_df = secondary_base.loc[
        secondary_base["period"] == period
    ].copy()

    period_rows.append(
        compute_exposure_contrast(
            period_df,
            "future_period",
            period,
        )
    )

period_contrasts_df = pd.DataFrame(
    period_rows
)

# -----------------------------------------------------------------------------
# F3-specific contrast
# -----------------------------------------------------------------------------

f3_df = secondary_base.loc[
    secondary_base["period"] == "F3"
].copy()

f3_contrast = compute_exposure_contrast(
    f3_df,
    "f3_specific",
    "F3",
)

f3_contrast_df = pd.DataFrame([
    f3_contrast
])

# -----------------------------------------------------------------------------
# Integrity checks
# -----------------------------------------------------------------------------

print("-" * 120)
print("SECONDARY PERIOD ANALYSIS INTEGRITY")
print("-" * 120)

checks_stage5a["all_six_periods_present"] = print_check(
    "all_six_periods_present",
    period_contrasts_df["group"].tolist()
    == ["F1", "F2", "F3", "F4", "F5", "F6"]
)

checks_stage5a["period_n_sum_matches_primary_population"] = print_check(
    "period_n_sum_matches_primary_population",
    int(period_contrasts_df["n"].sum())
    == len(secondary_base)
)

checks_stage5a["all_periods_have_correct_and_error"] = print_check(
    "all_periods_have_correct_and_error",
    (
        (period_contrasts_df["n_correct"] > 0)
        & (period_contrasts_df["n_error"] > 0)
    ).all()
)

checks_stage5a["f3_population_matches"] = print_check(
    "f3_population_matches",
    int(f3_contrast_df.loc[0, "n"])
    == len(f3_df)
)

# -----------------------------------------------------------------------------
# Save secondary artifacts
# -----------------------------------------------------------------------------

period_contrasts_df.to_csv(
    DAY27_PERIOD_CONTRASTS_CSV,
    index=False,
)

f3_contrast_df.to_csv(
    DAY27_F3_CONTRAST_CSV,
    index=False,
)

period_reload = pd.read_csv(
    DAY27_PERIOD_CONTRASTS_CSV
)

f3_reload = pd.read_csv(
    DAY27_F3_CONTRAST_CSV
)

checks_stage5a["period_artifact_exists"] = print_check(
    "period_artifact_exists",
    DAY27_PERIOD_CONTRASTS_CSV.exists()
)

checks_stage5a["period_artifact_nonempty"] = print_check(
    "period_artifact_nonempty",
    DAY27_PERIOD_CONTRASTS_CSV.stat().st_size > 0
)

checks_stage5a["period_artifact_rows_6"] = print_check(
    "period_artifact_rows_6",
    len(period_reload) == 6
)

checks_stage5a["f3_artifact_exists"] = print_check(
    "f3_artifact_exists",
    DAY27_F3_CONTRAST_CSV.exists()
)

checks_stage5a["f3_artifact_nonempty"] = print_check(
    "f3_artifact_nonempty",
    DAY27_F3_CONTRAST_CSV.stat().st_size > 0
)

checks_stage5a["f3_artifact_rows_1"] = print_check(
    "f3_artifact_rows_1",
    len(f3_reload) == 1
)

# -----------------------------------------------------------------------------
# Output
# -----------------------------------------------------------------------------

print("-" * 120)
print("F1-F6 SECONDARY EXPOSURE CONTRASTS")
print("-" * 120)

display_cols = [
    "group",
    "n",
    "n_correct",
    "n_error",
    "error_rate",
    "mean_exposure_correct",
    "mean_exposure_error",
    "delta_error_minus_correct",
]

print(
    period_contrasts_df[
        display_cols
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print("-" * 120)
print("F3-SPECIFIC SECONDARY CONTRAST")
print("-" * 120)

print(
    f3_contrast_df[
        display_cols
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

# -----------------------------------------------------------------------------
# Artifact hashes
# -----------------------------------------------------------------------------

day27_period_contrasts_sha256 = sha256_file(
    DAY27_PERIOD_CONTRASTS_CSV
)

day27_f3_contrast_sha256 = sha256_file(
    DAY27_F3_CONTRAST_CSV
)

print("-" * 120)
print("SECONDARY ARTIFACT IDENTITIES")
print("-" * 120)

print(
    f"Period contrasts SHA256 : "
    f"{day27_period_contrasts_sha256}"
)

print(
    f"F3 contrast SHA256      : "
    f"{day27_f3_contrast_sha256}"
)

# -----------------------------------------------------------------------------
# Final Stage 5A gate
# -----------------------------------------------------------------------------

all_stage5a_checks_pass = all(
    checks_stage5a.values()
)

print("-" * 120)

print_check(
    "DAY27_STAGE5A_ALL_PASS",
    all_stage5a_checks_pass
)

if not all_stage5a_checks_pass:
    raise RuntimeError(
        "DAY 27 INTEGRITY STOP: period/F3 secondary analysis failed."
    )

print("=" * 120)
print("DAY 27 STAGE 5A: PASS — PERIOD AND F3 CONTRASTS COMPLETE")
print("THE PRIMARY GATE REMAINS GATE_C_FAILED")
print("SECONDARY RESULTS ARE DESCRIPTIVE AND CANNOT OVERRIDE THE PRIMARY RESULT")
print("READY — STAGE 5B F3 POSITIVE P->N vs P->P ANALYSIS")
print("=" * 120)

DAY 27 — STAGE 5A PERIOD-SPECIFIC AND F3 EXPOSURE CONTRASTS
PASS   — primary_test_complete_and_frozen
PASS   — primary_gate_is_gate_c_failed
------------------------------------------------------------------------------------------------------------------------
SECONDARY PERIOD ANALYSIS INTEGRITY
------------------------------------------------------------------------------------------------------------------------
PASS   — all_six_periods_present
PASS   — period_n_sum_matches_primary_population
PASS   — all_periods_have_correct_and_error
PASS   — f3_population_matches
PASS   — period_artifact_exists
PASS   — period_artifact_nonempty
PASS   — period_artifact_rows_6
PASS   — f3_artifact_exists
PASS   — f3_artifact_nonempty
PASS   — f3_artifact_rows_1
------------------------------------------------------------------------------------------------------------------------
F1-F6 SECONDARY EXPOSURE CONTRASTS
------------------------------------------------------------------------------------

In [18]:
# =============================================================================
# Day 27 — Stage 5B
# F3 positive-class P->N vs P->P exposure comparison
# Canonical retained cell
# =============================================================================

print("=" * 120)
print("DAY 27 — STAGE 5B F3 POSITIVE P->N vs P->P EXPOSURE ANALYSIS")
print("=" * 120)

# -----------------------------------------------------------------------------
# Primary gate must remain frozen
# -----------------------------------------------------------------------------

primary_test_frozen = load_json(
    DAY27_PRIMARY_TEST_JSON
)

checks_stage5b = {}

checks_stage5b["primary_gate_remains_gate_c_failed"] = print_check(
    "primary_gate_remains_gate_c_failed",
    primary_test_frozen["decision_gate"]["gate"]
    == "GATE_C_FAILED"
)

# -----------------------------------------------------------------------------
# Restrict to frozen primary-complete F3 true-positive observations
# -----------------------------------------------------------------------------

f3_positive_df = primary_complete.loc[
    (primary_complete["period"] == "F3")
    & (primary_complete["y_true"] == "positive")
].copy()

print("-" * 120)
print("F3 TRUE-POSITIVE POPULATION")
print("-" * 120)

print(f"F3 positive observations : {len(f3_positive_df):,}")

# -----------------------------------------------------------------------------
# Define pre-specified groups
#
# P->N = true positive predicted negative
# P->P = true positive predicted positive
#
# Other predictions (e.g. P->neutral) are retained in population accounting,
# but are not part of the primary secondary contrast.
# -----------------------------------------------------------------------------

f3_positive_df["transition"] = np.select(
    [
        f3_positive_df["y_pred_day24"] == "negative",
        f3_positive_df["y_pred_day24"] == "positive",
        f3_positive_df["y_pred_day24"] == "neutral",
    ],
    [
        "P_to_N",
        "P_to_P",
        "P_to_Neutral",
    ],
    default="other",
)

transition_counts = (
    f3_positive_df["transition"]
    .value_counts()
    .to_dict()
)

for key in [
    "P_to_N",
    "P_to_P",
    "P_to_Neutral",
    "other",
]:
    transition_counts.setdefault(key, 0)

checks_stage5b["f3_positive_nonempty"] = print_check(
    "f3_positive_nonempty",
    len(f3_positive_df) > 0
)

checks_stage5b["p_to_n_nonempty"] = print_check(
    "p_to_n_nonempty",
    transition_counts["P_to_N"] > 0
)

checks_stage5b["p_to_p_nonempty"] = print_check(
    "p_to_p_nonempty",
    transition_counts["P_to_P"] > 0
)

checks_stage5b["transition_counts_sum_to_f3_positive"] = print_check(
    "transition_counts_sum_to_f3_positive",
    sum(transition_counts.values())
    == len(f3_positive_df)
)

# -----------------------------------------------------------------------------
# Frozen pre-specified comparison: P->N versus P->P only
# -----------------------------------------------------------------------------

p_to_n = f3_positive_df.loc[
    f3_positive_df["transition"] == "P_to_N",
    "lexical_instability_exposure_df20",
].to_numpy(dtype=float)

p_to_p = f3_positive_df.loc[
    f3_positive_df["transition"] == "P_to_P",
    "lexical_instability_exposure_df20",
].to_numpy(dtype=float)

n_p_to_n = len(p_to_n)
n_p_to_p = len(p_to_p)

mean_p_to_n = float(np.mean(p_to_n))
mean_p_to_p = float(np.mean(p_to_p))

median_p_to_n = float(np.median(p_to_n))
median_p_to_p = float(np.median(p_to_p))

delta_pn_minus_pp = float(
    mean_p_to_n - mean_p_to_p
)

sd_p_to_n = float(
    np.std(p_to_n, ddof=1)
) if n_p_to_n > 1 else np.nan

sd_p_to_p = float(
    np.std(p_to_p, ddof=1)
) if n_p_to_p > 1 else np.nan

# -----------------------------------------------------------------------------
# Descriptive standardized mean difference
# -----------------------------------------------------------------------------

if (
    n_p_to_n > 1
    and n_p_to_p > 1
):

    pooled_sd_f3_positive = np.sqrt(
        (
            (n_p_to_n - 1) * sd_p_to_n**2
            + (n_p_to_p - 1) * sd_p_to_p**2
        )
        /
        (
            n_p_to_n + n_p_to_p - 2
        )
    )

    smd_pn_minus_pp = float(
        delta_pn_minus_pp
        / pooled_sd_f3_positive
    )

else:
    pooled_sd_f3_positive = np.nan
    smd_pn_minus_pp = np.nan

# -----------------------------------------------------------------------------
# Save canonical secondary artifact
# -----------------------------------------------------------------------------

f3_positive_result = pd.DataFrame([
    {
        "scope": "F3_true_positive",
        "comparison": "P_to_N_vs_P_to_P",
        "f3_positive_total_n": int(
            len(f3_positive_df)
        ),
        "p_to_n_n": int(
            n_p_to_n
        ),
        "p_to_p_n": int(
            n_p_to_p
        ),
        "p_to_neutral_n": int(
            transition_counts["P_to_Neutral"]
        ),
        "other_n": int(
            transition_counts["other"]
        ),
        "mean_exposure_p_to_n": mean_p_to_n,
        "mean_exposure_p_to_p": mean_p_to_p,
        "median_exposure_p_to_n": median_p_to_n,
        "median_exposure_p_to_p": median_p_to_p,
        "sd_exposure_p_to_n": sd_p_to_n,
        "sd_exposure_p_to_p": sd_p_to_p,
        "delta_p_to_n_minus_p_to_p": delta_pn_minus_pp,
        "standardized_mean_difference": smd_pn_minus_pp,
        "primary_gate": "GATE_C_FAILED",
        "analysis_role": "pre_specified_secondary",
    }
])

f3_positive_result.to_csv(
    DAY27_F3_POSITIVE_CSV,
    index=False,
)

# -----------------------------------------------------------------------------
# Reload audit
# -----------------------------------------------------------------------------

f3_positive_reload = pd.read_csv(
    DAY27_F3_POSITIVE_CSV
)

checks_stage5b["f3_positive_artifact_exists"] = print_check(
    "f3_positive_artifact_exists",
    DAY27_F3_POSITIVE_CSV.exists()
)

checks_stage5b["f3_positive_artifact_nonempty"] = print_check(
    "f3_positive_artifact_nonempty",
    DAY27_F3_POSITIVE_CSV.stat().st_size > 0
)

checks_stage5b["f3_positive_artifact_rows_1"] = print_check(
    "f3_positive_artifact_rows_1",
    len(f3_positive_reload) == 1
)

checks_stage5b["primary_gate_not_modified"] = print_check(
    "primary_gate_not_modified",
    f3_positive_reload.loc[
        0,
        "primary_gate"
    ] == "GATE_C_FAILED"
)

# -----------------------------------------------------------------------------
# Output
# -----------------------------------------------------------------------------

print("-" * 120)
print("F3 POSITIVE TRANSITION COUNTS")
print("-" * 120)

print(f"P -> N       : {n_p_to_n:,}")
print(f"P -> P       : {n_p_to_p:,}")
print(
    f"P -> Neutral : "
    f"{transition_counts['P_to_Neutral']:,}"
)
print(
    f"Other        : "
    f"{transition_counts['other']:,}"
)

print("-" * 120)
print("F3 POSITIVE P->N vs P->P EXPOSURE CONTRAST")
print("-" * 120)

print(
    f"Mean exposure — P->N     : "
    f"{mean_p_to_n:.9f}"
)

print(
    f"Mean exposure — P->P     : "
    f"{mean_p_to_p:.9f}"
)

print(
    f"Median exposure — P->N   : "
    f"{median_p_to_n:.9f}"
)

print(
    f"Median exposure — P->P   : "
    f"{median_p_to_p:.9f}"
)

print(
    f"Delta (P->N - P->P)      : "
    f"{delta_pn_minus_pp:+.9f}"
)

print(
    f"Standardized mean diff    : "
    f"{smd_pn_minus_pp:+.6f}"
)

# -----------------------------------------------------------------------------
# Artifact identity
# -----------------------------------------------------------------------------

day27_f3_positive_sha256 = sha256_file(
    DAY27_F3_POSITIVE_CSV
)

print("-" * 120)
print("SECONDARY ARTIFACT IDENTITY")
print("-" * 120)

print(
    f"Path   : "
    f"{DAY27_F3_POSITIVE_CSV}"
)

print(
    f"SHA256 : "
    f"{day27_f3_positive_sha256}"
)

# -----------------------------------------------------------------------------
# Final Stage 5B gate
# -----------------------------------------------------------------------------

all_stage5b_checks_pass = all(
    checks_stage5b.values()
)

print("-" * 120)

print_check(
    "DAY27_STAGE5B_ALL_PASS",
    all_stage5b_checks_pass
)

if not all_stage5b_checks_pass:
    raise RuntimeError(
        "DAY 27 INTEGRITY STOP: F3 positive P->N vs P->P "
        "secondary analysis failed."
    )

print("=" * 120)
print("DAY 27 STAGE 5B: PASS — F3 POSITIVE CONTRAST COMPLETE")
print("THE PRIMARY GATE REMAINS GATE_C_FAILED")
print("THIS RESULT IS SECONDARY AND CANNOT RESCUE THE PRIMARY HYPOTHESIS")
print("READY — STAGE 5C WITHIN-SOURCE + CLASS-CONDITIONED ANALYSES")
print("=" * 120)

DAY 27 — STAGE 5B F3 POSITIVE P->N vs P->P EXPOSURE ANALYSIS
PASS   — primary_gate_remains_gate_c_failed
------------------------------------------------------------------------------------------------------------------------
F3 TRUE-POSITIVE POPULATION
------------------------------------------------------------------------------------------------------------------------
F3 positive observations : 544
PASS   — f3_positive_nonempty
PASS   — p_to_n_nonempty
PASS   — p_to_p_nonempty
PASS   — transition_counts_sum_to_f3_positive
PASS   — f3_positive_artifact_exists
PASS   — f3_positive_artifact_nonempty
PASS   — f3_positive_artifact_rows_1
PASS   — primary_gate_not_modified
------------------------------------------------------------------------------------------------------------------------
F3 POSITIVE TRANSITION COUNTS
------------------------------------------------------------------------------------------------------------------------
P -> N       : 104
P -> P       : 346
P -> Neutr

In [19]:
# =============================================================================
# Day 27 — Stage 5C
# Within-source and class-conditioned exposure contrasts
# Canonical retained cell
# =============================================================================

print("=" * 120)
print("DAY 27 — STAGE 5C WITHIN-SOURCE + CLASS-CONDITIONED EXPOSURE ANALYSES")
print("=" * 120)

# -----------------------------------------------------------------------------
# Primary result must remain frozen
# -----------------------------------------------------------------------------

primary_test_frozen = load_json(
    DAY27_PRIMARY_TEST_JSON
)

checks_stage5c = {}

checks_stage5c["primary_gate_remains_gate_c_failed"] = print_check(
    "primary_gate_remains_gate_c_failed",
    primary_test_frozen["decision_gate"]["gate"]
    == "GATE_C_FAILED"
)

if not checks_stage5c["primary_gate_remains_gate_c_failed"]:
    raise RuntimeError(
        "DAY 27 STOP: primary gate identity changed before Stage 5C."
    )

# -----------------------------------------------------------------------------
# Frozen analysis base
# -----------------------------------------------------------------------------

stage5c_base = primary_complete.copy()

exposure_col = "lexical_instability_exposure_df20"

# -----------------------------------------------------------------------------
# Helper
# -----------------------------------------------------------------------------

def contrast_row(df, scope, group):

    correct = df.loc[
        df["error_day24"] == 0,
        exposure_col,
    ].to_numpy(dtype=float)

    error = df.loc[
        df["error_day24"] == 1,
        exposure_col,
    ].to_numpy(dtype=float)

    n_correct = len(correct)
    n_error = len(error)

    mean_correct = (
        float(np.mean(correct))
        if n_correct > 0 else np.nan
    )

    mean_error = (
        float(np.mean(error))
        if n_error > 0 else np.nan
    )

    median_correct = (
        float(np.median(correct))
        if n_correct > 0 else np.nan
    )

    median_error = (
        float(np.median(error))
        if n_error > 0 else np.nan
    )

    delta = (
        mean_error - mean_correct
        if n_correct > 0 and n_error > 0
        else np.nan
    )

    return {
        "scope": scope,
        "group": str(group),
        "n": int(len(df)),
        "n_correct": int(n_correct),
        "n_error": int(n_error),
        "error_rate": (
            float(n_error / len(df))
            if len(df) > 0 else np.nan
        ),
        "mean_exposure_correct": mean_correct,
        "mean_exposure_error": mean_error,
        "median_exposure_correct": median_correct,
        "median_exposure_error": median_error,
        "delta_error_minus_correct": (
            float(delta)
            if np.isfinite(delta)
            else np.nan
        ),
    }

# =============================================================================
# PART A — WITHIN-SOURCE ANALYSES
# =============================================================================

# -----------------------------------------------------------------------------
# Pre-specified support rule:
# report all sources, but only interpret sources with both outcome groups.
# -----------------------------------------------------------------------------

source_rows = []

for source_name, source_df in stage5c_base.groupby(
    "source",
    sort=False,
):

    source_rows.append(
        contrast_row(
            source_df,
            "within_source",
            source_name,
        )
    )

within_source_df = pd.DataFrame(
    source_rows
)

within_source_df["sufficient_for_contrast"] = (
    (within_source_df["n_correct"] > 0)
    & (within_source_df["n_error"] > 0)
)

within_source_df = (
    within_source_df
    .sort_values(
        ["n", "group"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

# =============================================================================
# PART B — CLASS-CONDITIONED ANALYSES
# =============================================================================

class_rows = []

for class_name in [
    "negative",
    "neutral",
    "positive",
]:

    class_df = stage5c_base.loc[
        stage5c_base["y_true"] == class_name
    ].copy()

    class_rows.append(
        contrast_row(
            class_df,
            "true_label",
            class_name,
        )
    )

class_conditioned_df = pd.DataFrame(
    class_rows
)

# -----------------------------------------------------------------------------
# Integrity checks
# -----------------------------------------------------------------------------

print("-" * 120)
print("WITHIN-SOURCE / CLASS INTEGRITY")
print("-" * 120)

checks_stage5c["all_primary_rows_accounted_for_by_source"] = print_check(
    "all_primary_rows_accounted_for_by_source",
    int(within_source_df["n"].sum())
    == len(stage5c_base)
)

checks_stage5c["class_rows_exact_3"] = print_check(
    "class_rows_exact_3",
    len(class_conditioned_df) == 3
)

checks_stage5c["all_primary_rows_accounted_for_by_class"] = print_check(
    "all_primary_rows_accounted_for_by_class",
    int(class_conditioned_df["n"].sum())
    == len(stage5c_base)
)

checks_stage5c["all_three_classes_have_both_outcomes"] = print_check(
    "all_three_classes_have_both_outcomes",
    (
        (class_conditioned_df["n_correct"] > 0)
        & (class_conditioned_df["n_error"] > 0)
    ).all()
)

checks_stage5c["at_least_one_source_supports_contrast"] = print_check(
    "at_least_one_source_supports_contrast",
    within_source_df["sufficient_for_contrast"].any()
)

# -----------------------------------------------------------------------------
# Save canonical artifacts
# -----------------------------------------------------------------------------

within_source_df.to_csv(
    DAY27_WITHIN_SOURCE_CSV,
    index=False,
)

class_conditioned_df.to_csv(
    DAY27_CLASS_CONDITIONED_CSV,
    index=False,
)

within_source_reload = pd.read_csv(
    DAY27_WITHIN_SOURCE_CSV
)

class_conditioned_reload = pd.read_csv(
    DAY27_CLASS_CONDITIONED_CSV
)

checks_stage5c["within_source_artifact_exists"] = print_check(
    "within_source_artifact_exists",
    DAY27_WITHIN_SOURCE_CSV.exists()
)

checks_stage5c["within_source_artifact_nonempty"] = print_check(
    "within_source_artifact_nonempty",
    DAY27_WITHIN_SOURCE_CSV.stat().st_size > 0
)

checks_stage5c["class_artifact_exists"] = print_check(
    "class_artifact_exists",
    DAY27_CLASS_CONDITIONED_CSV.exists()
)

checks_stage5c["class_artifact_nonempty"] = print_check(
    "class_artifact_nonempty",
    DAY27_CLASS_CONDITIONED_CSV.stat().st_size > 0
)

# -----------------------------------------------------------------------------
# Output — within-source
# -----------------------------------------------------------------------------

print("-" * 120)
print("WITHIN-SOURCE EXPOSURE CONTRASTS")
print("-" * 120)

within_source_display_cols = [
    "group",
    "n",
    "n_correct",
    "n_error",
    "error_rate",
    "mean_exposure_correct",
    "mean_exposure_error",
    "delta_error_minus_correct",
    "sufficient_for_contrast",
]

print(
    within_source_df[
        within_source_display_cols
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

# -----------------------------------------------------------------------------
# Output — class-conditioned
# -----------------------------------------------------------------------------

print("-" * 120)
print("CLASS-CONDITIONED EXPOSURE CONTRASTS")
print("-" * 120)

class_display_cols = [
    "group",
    "n",
    "n_correct",
    "n_error",
    "error_rate",
    "mean_exposure_correct",
    "mean_exposure_error",
    "delta_error_minus_correct",
]

print(
    class_conditioned_df[
        class_display_cols
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

# -----------------------------------------------------------------------------
# Direction summaries
# -----------------------------------------------------------------------------

supported_source_df = within_source_df.loc[
    within_source_df["sufficient_for_contrast"]
].copy()

source_positive_delta_n = int(
    (
        supported_source_df[
            "delta_error_minus_correct"
        ] > 0
    ).sum()
)

source_negative_delta_n = int(
    (
        supported_source_df[
            "delta_error_minus_correct"
        ] < 0
    ).sum()
)

class_positive_delta_n = int(
    (
        class_conditioned_df[
            "delta_error_minus_correct"
        ] > 0
    ).sum()
)

class_negative_delta_n = int(
    (
        class_conditioned_df[
            "delta_error_minus_correct"
        ] < 0
    ).sum()
)

print("-" * 120)
print("SECONDARY DIRECTION SUMMARY")
print("-" * 120)

print(
    f"Sources with valid contrasts : "
    f"{len(supported_source_df)}"
)

print(
    f"Positive source deltas        : "
    f"{source_positive_delta_n}"
)

print(
    f"Negative source deltas        : "
    f"{source_negative_delta_n}"
)

print()

print(
    f"Positive class deltas         : "
    f"{class_positive_delta_n}"
)

print(
    f"Negative class deltas         : "
    f"{class_negative_delta_n}"
)

# -----------------------------------------------------------------------------
# Artifact identities
# -----------------------------------------------------------------------------

day27_within_source_sha256 = sha256_file(
    DAY27_WITHIN_SOURCE_CSV
)

day27_class_conditioned_sha256 = sha256_file(
    DAY27_CLASS_CONDITIONED_CSV
)

print("-" * 120)
print("SECONDARY ARTIFACT IDENTITIES")
print("-" * 120)

print(
    f"Within-source SHA256      : "
    f"{day27_within_source_sha256}"
)

print(
    f"Class-conditioned SHA256  : "
    f"{day27_class_conditioned_sha256}"
)

# -----------------------------------------------------------------------------
# Final Stage 5C gate
# -----------------------------------------------------------------------------

all_stage5c_checks_pass = all(
    checks_stage5c.values()
)

print("-" * 120)

print_check(
    "DAY27_STAGE5C_ALL_PASS",
    all_stage5c_checks_pass
)

if not all_stage5c_checks_pass:
    raise RuntimeError(
        "DAY 27 INTEGRITY STOP: within-source/class-conditioned "
        "secondary analysis failed."
    )

print("=" * 120)
print("DAY 27 STAGE 5C: PASS — SOURCE AND CLASS CONTRASTS COMPLETE")
print("THE PRIMARY GATE REMAINS GATE_C_FAILED")
print("SECONDARY HETEROGENEITY CANNOT OVERRIDE THE PRIMARY RESULT")
print("READY — STAGE 5D df>=50 SENSITIVITY + SECONDARY LOGISTIC MODELS")
print("=" * 120)

DAY 27 — STAGE 5C WITHIN-SOURCE + CLASS-CONDITIONED EXPOSURE ANALYSES
PASS   — primary_gate_remains_gate_c_failed
------------------------------------------------------------------------------------------------------------------------
WITHIN-SOURCE / CLASS INTEGRITY
------------------------------------------------------------------------------------------------------------------------
PASS   — all_primary_rows_accounted_for_by_source
PASS   — class_rows_exact_3
PASS   — all_primary_rows_accounted_for_by_class
PASS   — all_three_classes_have_both_outcomes
PASS   — at_least_one_source_supports_contrast
PASS   — within_source_artifact_exists
PASS   — within_source_artifact_nonempty
PASS   — class_artifact_exists
PASS   — class_artifact_nonempty
------------------------------------------------------------------------------------------------------------------------
WITHIN-SOURCE EXPOSURE CONTRASTS
----------------------------------------------------------------------------------------------

In [20]:
# =============================================================================
# Day 27 — Stage 5D
# df>=50 sensitivity + secondary logistic-regression corroboration
# Canonical retained cell
# =============================================================================

import statsmodels.api as sm

print("=" * 120)
print("DAY 27 — STAGE 5D df>=50 SENSITIVITY + SECONDARY LOGISTIC MODELS")
print("=" * 120)

# -----------------------------------------------------------------------------
# Primary result must remain frozen
# -----------------------------------------------------------------------------

primary_test_frozen = load_json(
    DAY27_PRIMARY_TEST_JSON
)

checks_stage5d = {}

checks_stage5d["primary_gate_remains_gate_c_failed"] = print_check(
    "primary_gate_remains_gate_c_failed",
    primary_test_frozen["decision_gate"]["gate"]
    == "GATE_C_FAILED"
)

if not checks_stage5d["primary_gate_remains_gate_c_failed"]:
    raise RuntimeError(
        "DAY 27 STOP: primary gate identity changed before Stage 5D."
    )

# =============================================================================
# PART A — df>=50 PRE-SPECIFIED SENSITIVITY
# =============================================================================

# -----------------------------------------------------------------------------
# Use frozen exposure artifact + frozen correctness
# -----------------------------------------------------------------------------

df50_analysis_df = primary_analysis_df.loc[
    primary_analysis_df[
        "lexical_instability_exposure_df50"
    ].notna()
].copy()

df50_correct = df50_analysis_df.loc[
    df50_analysis_df["error_day24"] == 0,
    "lexical_instability_exposure_df50",
].to_numpy(dtype=float)

df50_error = df50_analysis_df.loc[
    df50_analysis_df["error_day24"] == 1,
    "lexical_instability_exposure_df50",
].to_numpy(dtype=float)

n_df50_correct = len(df50_correct)
n_df50_error = len(df50_error)

mean_df50_correct = float(
    np.mean(df50_correct)
)

mean_df50_error = float(
    np.mean(df50_error)
)

median_df50_correct = float(
    np.median(df50_correct)
)

median_df50_error = float(
    np.median(df50_error)
)

delta_df50 = float(
    mean_df50_error
    - mean_df50_correct
)

sd_df50_correct = float(
    np.std(
        df50_correct,
        ddof=1,
    )
)

sd_df50_error = float(
    np.std(
        df50_error,
        ddof=1,
    )
)

pooled_sd_df50 = np.sqrt(
    (
        (n_df50_correct - 1) * sd_df50_correct**2
        + (n_df50_error - 1) * sd_df50_error**2
    )
    /
    (
        n_df50_correct
        + n_df50_error
        - 2
    )
)

smd_df50 = float(
    delta_df50
    / pooled_sd_df50
)

# -----------------------------------------------------------------------------
# df>=50 integrity
# -----------------------------------------------------------------------------

print("-" * 120)
print("df>=50 SENSITIVITY INTEGRITY")
print("-" * 120)

checks_stage5d["df50_analysis_nonempty"] = print_check(
    "df50_analysis_nonempty",
    len(df50_analysis_df) > 0
)

checks_stage5d["df50_missing_exactly_six"] = print_check(
    "df50_missing_exactly_six",
    (
        len(primary_analysis_df)
        - len(df50_analysis_df)
    ) == 6
)

checks_stage5d["df50_has_correct_and_error"] = print_check(
    "df50_has_correct_and_error",
    n_df50_correct > 0
    and n_df50_error > 0
)

checks_stage5d["df50_role_is_secondary"] = print_check(
    "df50_role_is_secondary",
    day27_analysis_manifest[
        "secondary_analyses"
    ]["df50_sensitivity"] is True
    and day27_analysis_manifest[
        "primary_exposure"
    ]["primary_minimum_historical_df"] == 20
)

# -----------------------------------------------------------------------------
# Save df>=50 sensitivity artifact
# -----------------------------------------------------------------------------

df50_sensitivity_result = {
    "day": 27,
    "artifact": "df50_support_sensitivity_contrast",
    "status": "SECONDARY_COMPLETE",

    "analysis_role": "pre_specified_secondary_sensitivity",

    "support_threshold": 50,

    "population": {
        "n": int(
            len(df50_analysis_df)
        ),
        "n_missing": int(
            len(primary_analysis_df)
            - len(df50_analysis_df)
        ),
        "n_correct": int(
            n_df50_correct
        ),
        "n_error": int(
            n_df50_error
        ),
    },

    "contrast": {
        "mean_correct": mean_df50_correct,
        "mean_error": mean_df50_error,
        "median_correct": median_df50_correct,
        "median_error": median_df50_error,
        "delta_error_minus_correct": delta_df50,
        "standardized_mean_difference": smd_df50,
    },

    "primary_gate": "GATE_C_FAILED",

    "may_override_primary_gate": False,

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

save_json(
    df50_sensitivity_result,
    DAY27_DF50_SENSITIVITY_JSON,
)

df50_reload = load_json(
    DAY27_DF50_SENSITIVITY_JSON
)

checks_stage5d["df50_artifact_exists"] = print_check(
    "df50_artifact_exists",
    DAY27_DF50_SENSITIVITY_JSON.exists()
)

checks_stage5d["df50_artifact_nonempty"] = print_check(
    "df50_artifact_nonempty",
    DAY27_DF50_SENSITIVITY_JSON.stat().st_size > 0
)

checks_stage5d["df50_artifact_roundtrip_exact"] = print_check(
    "df50_artifact_roundtrip_exact",
    df50_reload
    == df50_sensitivity_result
)

# =============================================================================
# PART B — SECONDARY UNADJUSTED LOGISTIC REGRESSION
#
# error ~ standardized primary exposure
# =============================================================================

logistic_df = primary_complete[
    [
        "error_day24",
        "lexical_instability_exposure_df20",
        "period",
        "source",
        "y_true",
    ]
].copy()

exposure_mean = float(
    logistic_df[
        "lexical_instability_exposure_df20"
    ].mean()
)

exposure_sd = float(
    logistic_df[
        "lexical_instability_exposure_df20"
    ].std(ddof=1)
)

logistic_df["exposure_z"] = (
    (
        logistic_df[
            "lexical_instability_exposure_df20"
        ]
        - exposure_mean
    )
    / exposure_sd
)

# -----------------------------------------------------------------------------
# Unadjusted model
# -----------------------------------------------------------------------------

X_unadjusted = sm.add_constant(
    logistic_df[
        ["exposure_z"]
    ],
    has_constant="add",
)

y_logistic = logistic_df[
    "error_day24"
].astype(int)

unadjusted_model = sm.GLM(
    y_logistic,
    X_unadjusted,
    family=sm.families.Binomial(),
).fit()

beta_unadjusted = float(
    unadjusted_model.params[
        "exposure_z"
    ]
)

se_unadjusted = float(
    unadjusted_model.bse[
        "exposure_z"
    ]
)

ci_unadjusted = (
    unadjusted_model
    .conf_int()
    .loc["exposure_z"]
)

beta_unadjusted_ci_low = float(
    ci_unadjusted.iloc[0]
)

beta_unadjusted_ci_high = float(
    ci_unadjusted.iloc[1]
)

or_unadjusted = float(
    np.exp(beta_unadjusted)
)

or_unadjusted_ci_low = float(
    np.exp(
        beta_unadjusted_ci_low
    )
)

or_unadjusted_ci_high = float(
    np.exp(
        beta_unadjusted_ci_high
    )
)

p_unadjusted = float(
    unadjusted_model.pvalues[
        "exposure_z"
    ]
)

# =============================================================================
# PART C — SECONDARY ADJUSTED LOGISTIC REGRESSION
#
# error ~ exposure_z + period + source + y_true
# =============================================================================

# -----------------------------------------------------------------------------
# Dummy-code frozen observed categories
# -----------------------------------------------------------------------------

categorical_design = pd.get_dummies(
    logistic_df[
        [
            "period",
            "source",
            "y_true",
        ]
    ],
    drop_first=True,
    dtype=float,
)

X_adjusted = pd.concat(
    [
        logistic_df[
            ["exposure_z"]
        ].reset_index(drop=True),
        categorical_design.reset_index(drop=True),
    ],
    axis=1,
)

X_adjusted = sm.add_constant(
    X_adjusted,
    has_constant="add",
)

# Ensure numeric design matrix
X_adjusted = X_adjusted.astype(float)

adjusted_model = sm.GLM(
    y_logistic.reset_index(drop=True),
    X_adjusted.reset_index(drop=True),
    family=sm.families.Binomial(),
).fit()

beta_adjusted = float(
    adjusted_model.params[
        "exposure_z"
    ]
)

se_adjusted = float(
    adjusted_model.bse[
        "exposure_z"
    ]
)

ci_adjusted = (
    adjusted_model
    .conf_int()
    .loc["exposure_z"]
)

beta_adjusted_ci_low = float(
    ci_adjusted.iloc[0]
)

beta_adjusted_ci_high = float(
    ci_adjusted.iloc[1]
)

or_adjusted = float(
    np.exp(beta_adjusted)
)

or_adjusted_ci_low = float(
    np.exp(
        beta_adjusted_ci_low
    )
)

or_adjusted_ci_high = float(
    np.exp(
        beta_adjusted_ci_high
    )
)

p_adjusted = float(
    adjusted_model.pvalues[
        "exposure_z"
    ]
)

# -----------------------------------------------------------------------------
# Logistic integrity
# -----------------------------------------------------------------------------

print("-" * 120)
print("SECONDARY LOGISTIC MODEL INTEGRITY")
print("-" * 120)

checks_stage5d["logistic_n_matches_primary_complete"] = print_check(
    "logistic_n_matches_primary_complete",
    len(logistic_df)
    == len(primary_complete)
)

checks_stage5d["exposure_z_finite"] = print_check(
    "exposure_z_finite",
    np.isfinite(
        logistic_df[
            "exposure_z"
        ].to_numpy(dtype=float)
    ).all()
)

checks_stage5d["exposure_z_centered"] = print_check(
    "exposure_z_centered",
    abs(
        logistic_df[
            "exposure_z"
        ].mean()
    ) < 1e-12
)

checks_stage5d["exposure_z_sd_one"] = print_check(
    "exposure_z_sd_one",
    np.isclose(
        logistic_df[
            "exposure_z"
        ].std(ddof=1),
        1.0,
        rtol=1e-12,
        atol=1e-12,
    )
)

checks_stage5d["unadjusted_model_converged"] = print_check(
    "unadjusted_model_converged",
    bool(
        getattr(
            unadjusted_model,
            "converged",
            True,
        )
    )
)

checks_stage5d["adjusted_model_converged"] = print_check(
    "adjusted_model_converged",
    bool(
        getattr(
            adjusted_model,
            "converged",
            True,
        )
    )
)

checks_stage5d["unadjusted_or_finite"] = print_check(
    "unadjusted_or_finite",
    np.isfinite(or_unadjusted)
)

checks_stage5d["adjusted_or_finite"] = print_check(
    "adjusted_or_finite",
    np.isfinite(or_adjusted)
)

# -----------------------------------------------------------------------------
# Save logistic secondary artifact
# -----------------------------------------------------------------------------

day27_logistic_secondary = {
    "day": 27,
    "artifact": "day27_logistic_secondary",
    "status": "SECONDARY_COMPLETE",

    "analysis_role": (
        "secondary_corroboration_only"
    ),

    "population_n": int(
        len(logistic_df)
    ),

    "exposure_standardization": {
        "field": (
            "lexical_instability_exposure_df20"
        ),
        "mean": exposure_mean,
        "sd": exposure_sd,
        "unit": (
            "1 SD increase in frozen primary exposure"
        ),
    },

    "unadjusted_model": {
        "formula": (
            "error ~ standardized_exposure"
        ),
        "coefficient": beta_unadjusted,
        "standard_error": se_unadjusted,
        "coefficient_ci_low": (
            beta_unadjusted_ci_low
        ),
        "coefficient_ci_high": (
            beta_unadjusted_ci_high
        ),
        "odds_ratio_per_1sd": (
            or_unadjusted
        ),
        "odds_ratio_ci_low": (
            or_unadjusted_ci_low
        ),
        "odds_ratio_ci_high": (
            or_unadjusted_ci_high
        ),
        "p_value": p_unadjusted,
    },

    "adjusted_model": {
        "formula": (
            "error ~ standardized_exposure "
            "+ period + source + y_true"
        ),
        "coefficient": beta_adjusted,
        "standard_error": se_adjusted,
        "coefficient_ci_low": (
            beta_adjusted_ci_low
        ),
        "coefficient_ci_high": (
            beta_adjusted_ci_high
        ),
        "odds_ratio_per_1sd": (
            or_adjusted
        ),
        "odds_ratio_ci_low": (
            or_adjusted_ci_low
        ),
        "odds_ratio_ci_high": (
            or_adjusted_ci_high
        ),
        "p_value": p_adjusted,
        "n_design_columns": int(
            X_adjusted.shape[1]
        ),
    },

    "primary_gate": "GATE_C_FAILED",

    "may_override_primary_gate": False,

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

save_json(
    day27_logistic_secondary,
    DAY27_LOGISTIC_SECONDARY_JSON,
)

logistic_reload = load_json(
    DAY27_LOGISTIC_SECONDARY_JSON
)

checks_stage5d["logistic_artifact_exists"] = print_check(
    "logistic_artifact_exists",
    DAY27_LOGISTIC_SECONDARY_JSON.exists()
)

checks_stage5d["logistic_artifact_nonempty"] = print_check(
    "logistic_artifact_nonempty",
    DAY27_LOGISTIC_SECONDARY_JSON.stat().st_size > 0
)

checks_stage5d["logistic_artifact_roundtrip_exact"] = print_check(
    "logistic_artifact_roundtrip_exact",
    logistic_reload
    == day27_logistic_secondary
)

# =============================================================================
# OUTPUT
# =============================================================================

print("-" * 120)
print("df>=50 SECONDARY SENSITIVITY")
print("-" * 120)

print(
    f"Analysis n              : "
    f"{len(df50_analysis_df):,}"
)

print(
    f"Correct n               : "
    f"{n_df50_correct:,}"
)

print(
    f"Error n                 : "
    f"{n_df50_error:,}"
)

print()

print(
    f"Mean exposure — correct : "
    f"{mean_df50_correct:.9f}"
)

print(
    f"Mean exposure — error   : "
    f"{mean_df50_error:.9f}"
)

print(
    f"Delta df>=50            : "
    f"{delta_df50:+.9f}"
)

print(
    f"Standardized mean diff  : "
    f"{smd_df50:+.6f}"
)

print("-" * 120)
print("UNADJUSTED LOGISTIC — SECONDARY")
print("-" * 120)

print(
    f"Exposure coefficient    : "
    f"{beta_unadjusted:+.6f}"
)

print(
    f"OR per 1 SD exposure    : "
    f"{or_unadjusted:.6f}"
)

print(
    f"95% OR CI               : "
    f"[{or_unadjusted_ci_low:.6f}, "
    f"{or_unadjusted_ci_high:.6f}]"
)

print(
    f"p-value                 : "
    f"{p_unadjusted:.6g}"
)

print("-" * 120)
print("ADJUSTED LOGISTIC — SECONDARY")
print("-" * 120)

print(
    "Adjustment              : period + source + y_true"
)

print(
    f"Exposure coefficient    : "
    f"{beta_adjusted:+.6f}"
)

print(
    f"Adjusted OR per 1 SD     : "
    f"{or_adjusted:.6f}"
)

print(
    f"95% adjusted OR CI      : "
    f"[{or_adjusted_ci_low:.6f}, "
    f"{or_adjusted_ci_high:.6f}]"
)

print(
    f"p-value                 : "
    f"{p_adjusted:.6g}"
)

# -----------------------------------------------------------------------------
# Artifact identities
# -----------------------------------------------------------------------------

day27_df50_sha256 = sha256_file(
    DAY27_DF50_SENSITIVITY_JSON
)

day27_logistic_sha256 = sha256_file(
    DAY27_LOGISTIC_SECONDARY_JSON
)

print("-" * 120)
print("SECONDARY ARTIFACT IDENTITIES")
print("-" * 120)

print(
    f"df>=50 sensitivity SHA256 : "
    f"{day27_df50_sha256}"
)

print(
    f"Logistic secondary SHA256  : "
    f"{day27_logistic_sha256}"
)

# -----------------------------------------------------------------------------
# Final Stage 5D / Stage 5 gate
# -----------------------------------------------------------------------------

all_stage5d_checks_pass = all(
    checks_stage5d.values()
)

print("-" * 120)

print_check(
    "DAY27_STAGE5D_ALL_PASS",
    all_stage5d_checks_pass
)

if not all_stage5d_checks_pass:
    raise RuntimeError(
        "DAY 27 INTEGRITY STOP: df>=50/logistic secondary analysis failed."
    )

print("=" * 120)
print("DAY 27 STAGE 5: COMPLETE — ALL PRE-SPECIFIED SECONDARY ANALYSES FINISHED")
print("PRIMARY GATE REMAINS: GATE_C_FAILED")
print("NO SECONDARY RESULT MAY OVERRIDE THE PRIMARY CONCLUSION")
print("READY — STAGE 6 EVIDENCE INTEGRATION / INTERPRETATION GATE")
print("=" * 120)

DAY 27 — STAGE 5D df>=50 SENSITIVITY + SECONDARY LOGISTIC MODELS
PASS   — primary_gate_remains_gate_c_failed
------------------------------------------------------------------------------------------------------------------------
df>=50 SENSITIVITY INTEGRITY
------------------------------------------------------------------------------------------------------------------------
PASS   — df50_analysis_nonempty
PASS   — df50_missing_exactly_six
PASS   — df50_has_correct_and_error
PASS   — df50_role_is_secondary
PASS   — df50_artifact_exists
PASS   — df50_artifact_nonempty
PASS   — df50_artifact_roundtrip_exact
------------------------------------------------------------------------------------------------------------------------
SECONDARY LOGISTIC MODEL INTEGRITY
------------------------------------------------------------------------------------------------------------------------
PASS   — logistic_n_matches_primary_complete
PASS   — exposure_z_finite
PASS   — exposure_z_centered
PASS   

## Stage 6 — Evidence Integration / Interpretation Gate

All pre-specified Day 27 analyses are now complete.

The purpose of this stage is not to search for additional effects, but to integrate the frozen evidence and determine the scientific interpretation that is permitted by the Day 27 protocol.

### Primary result

The pre-registered all-future primary contrast was

\[
\Delta_{\mathrm{primary}}
=
-0.005388,
\]

with

\[
95\%\,CI
=
[-0.010077,\,-0.000992].
\]

Therefore,

\[
\boxed{\text{Primary Gate = GATE C — FAILED}}.
\]

The historical-only lexical-instability exposure frozen on Day 26 did not predict higher future classification error under the pre-registered Day 27 definition.

### Secondary consistency

The pre-specified secondary analyses showed:

- F1–F4 period-specific contrasts were negative;
- F5 and F6 were positive, indicating temporal heterogeneity;
- the F3-specific contrast remained negative;
- the F3 \(P\rightarrow N\) versus \(P\rightarrow P\) contrast was approximately zero and slightly negative;
- all three true-label-conditioned contrasts were negative;
- the three largest sources were negative;
- the df ≥ 50 sensitivity analysis was more strongly negative;
- unadjusted and adjusted logistic models both produced odds ratios below 1.

### Interpretation boundary

These results support the conclusion that the frozen Day 26 historical lexical-label instability measure is **not a valid positive predictor of future classification error under the current exposure definition**.

This does **not** imply that lexical or topic shift is unimportant.

It implies only that:

\[
\text{historical lexical-label instability}
\neq
\text{the mechanism captured by the Day 25 future failure signature}
\]

under the current frozen operationalization.

No causal claim is permitted.

No secondary subgroup may override the failed primary gate.

In [21]:
# =============================================================================
# Day 27 — Stage 6
# Evidence integration + final scientific summary
# Canonical retained cell
# =============================================================================

print("=" * 120)
print("DAY 27 — STAGE 6 EVIDENCE INTEGRATION / INTERPRETATION GATE")
print("=" * 120)

# -----------------------------------------------------------------------------
# Load frozen artifacts
# -----------------------------------------------------------------------------

primary_result = load_json(
    DAY27_PRIMARY_TEST_JSON
)

df50_result = load_json(
    DAY27_DF50_SENSITIVITY_JSON
)

logistic_result = load_json(
    DAY27_LOGISTIC_SECONDARY_JSON
)

period_result = pd.read_csv(
    DAY27_PERIOD_CONTRASTS_CSV
)

f3_result = pd.read_csv(
    DAY27_F3_CONTRAST_CSV
)

f3_positive_result = pd.read_csv(
    DAY27_F3_POSITIVE_CSV
)

within_source_result = pd.read_csv(
    DAY27_WITHIN_SOURCE_CSV
)

class_result = pd.read_csv(
    DAY27_CLASS_CONDITIONED_CSV
)

# -----------------------------------------------------------------------------
# Integrity checks
# -----------------------------------------------------------------------------

checks_stage6 = {}

checks_stage6["primary_gate_gate_c_failed"] = print_check(
    "primary_gate_gate_c_failed",
    primary_result["decision_gate"]["gate"]
    == "GATE_C_FAILED"
)

checks_stage6["primary_delta_negative"] = print_check(
    "primary_delta_negative",
    primary_result["primary_estimand"][
        "delta_primary"
    ] < 0
)

checks_stage6["primary_ci_entirely_below_zero"] = print_check(
    "primary_ci_entirely_below_zero",
    primary_result["bootstrap"]["ci_high"] < 0
)

checks_stage6["df50_secondary_negative"] = print_check(
    "df50_secondary_negative",
    df50_result["contrast"][
        "delta_error_minus_correct"
    ] < 0
)

checks_stage6["unadjusted_logistic_or_below_1"] = print_check(
    "unadjusted_logistic_or_below_1",
    logistic_result["unadjusted_model"][
        "odds_ratio_per_1sd"
    ] < 1
)

checks_stage6["adjusted_logistic_or_below_1"] = print_check(
    "adjusted_logistic_or_below_1",
    logistic_result["adjusted_model"][
        "odds_ratio_per_1sd"
    ] < 1
)

checks_stage6["all_three_class_deltas_negative"] = print_check(
    "all_three_class_deltas_negative",
    (
        class_result[
            "delta_error_minus_correct"
        ] < 0
    ).all()
)

# -----------------------------------------------------------------------------
# Major-source consistency
#
# Use the three largest sources only for structured summary.
# -----------------------------------------------------------------------------

major_sources = (
    within_source_result
    .sort_values(
        "n",
        ascending=False,
    )
    .head(3)
    .copy()
)

checks_stage6["three_largest_sources_negative"] = print_check(
    "three_largest_sources_negative",
    (
        major_sources[
            "delta_error_minus_correct"
        ] < 0
    ).all()
)

# -----------------------------------------------------------------------------
# Period-direction summary
# -----------------------------------------------------------------------------

negative_periods = (
    period_result.loc[
        period_result[
            "delta_error_minus_correct"
        ] < 0,
        "group",
    ]
    .astype(str)
    .tolist()
)

positive_periods = (
    period_result.loc[
        period_result[
            "delta_error_minus_correct"
        ] > 0,
        "group",
    ]
    .astype(str)
    .tolist()
)

# -----------------------------------------------------------------------------
# Evidence table
# -----------------------------------------------------------------------------

evidence_rows = [
    {
        "evidence_domain": "primary_all_future",
        "analysis": "df20_primary_mean_contrast",
        "estimate": float(
            primary_result[
                "primary_estimand"
            ]["delta_primary"]
        ),
        "ci_low": float(
            primary_result[
                "bootstrap"
            ]["ci_low"]
        ),
        "ci_high": float(
            primary_result[
                "bootstrap"
            ]["ci_high"]
        ),
        "direction": "negative",
        "role": "primary",
        "supports_H1": False,
        "interpretation": (
            "Primary H1 failed; exposure was slightly lower "
            "among errors than correct predictions."
        ),
    },
    {
        "evidence_domain": "f3_specific",
        "analysis": "F3_error_vs_correct",
        "estimate": float(
            f3_result.loc[
                0,
                "delta_error_minus_correct",
            ]
        ),
        "ci_low": np.nan,
        "ci_high": np.nan,
        "direction": (
            "negative"
            if f3_result.loc[
                0,
                "delta_error_minus_correct",
            ] < 0
            else "positive"
        ),
        "role": "secondary",
        "supports_H1": False,
        "interpretation": (
            "F3-specific contrast did not rescue the primary hypothesis."
        ),
    },
    {
        "evidence_domain": "f3_positive_failure_channel",
        "analysis": "P_to_N_vs_P_to_P",
        "estimate": float(
            f3_positive_result.loc[
                0,
                "delta_p_to_n_minus_p_to_p",
            ]
        ),
        "ci_low": np.nan,
        "ci_high": np.nan,
        "direction": (
            "negative"
            if f3_positive_result.loc[
                0,
                "delta_p_to_n_minus_p_to_p",
            ] < 0
            else "positive"
        ),
        "role": "secondary",
        "supports_H1": False,
        "interpretation": (
            "The specific F3 positive-to-negative failure channel "
            "showed essentially no positive instability signal."
        ),
    },
    {
        "evidence_domain": "df50_sensitivity",
        "analysis": "df50_error_vs_correct",
        "estimate": float(
            df50_result[
                "contrast"
            ]["delta_error_minus_correct"]
        ),
        "ci_low": np.nan,
        "ci_high": np.nan,
        "direction": "negative",
        "role": "secondary_sensitivity",
        "supports_H1": False,
        "interpretation": (
            "Restricting to better-supported tokens strengthened "
            "the negative direction."
        ),
    },
    {
        "evidence_domain": "logistic_unadjusted",
        "analysis": "OR_per_1SD_exposure",
        "estimate": float(
            logistic_result[
                "unadjusted_model"
            ]["odds_ratio_per_1sd"]
        ),
        "ci_low": float(
            logistic_result[
                "unadjusted_model"
            ]["odds_ratio_ci_low"]
        ),
        "ci_high": float(
            logistic_result[
                "unadjusted_model"
            ]["odds_ratio_ci_high"]
        ),
        "direction": "below_1",
        "role": "secondary",
        "supports_H1": False,
        "interpretation": (
            "Higher exposure was associated with lower error odds "
            "in the unadjusted secondary model."
        ),
    },
    {
        "evidence_domain": "logistic_adjusted",
        "analysis": "adjusted_OR_per_1SD_exposure",
        "estimate": float(
            logistic_result[
                "adjusted_model"
            ]["odds_ratio_per_1sd"]
        ),
        "ci_low": float(
            logistic_result[
                "adjusted_model"
            ]["odds_ratio_ci_low"]
        ),
        "ci_high": float(
            logistic_result[
                "adjusted_model"
            ]["odds_ratio_ci_high"]
        ),
        "direction": "below_1",
        "role": "secondary_adjusted",
        "supports_H1": False,
        "interpretation": (
            "The negative association persisted after adjustment "
            "for period, source, and true label."
        ),
    },
]

day27_evidence_table = pd.DataFrame(
    evidence_rows
)

day27_evidence_table.to_csv(
    DAY27_EVIDENCE_TABLE_CSV,
    index=False,
)

# -----------------------------------------------------------------------------
# Final scientific conclusion
# -----------------------------------------------------------------------------

final_conclusion = (
    "Day 27 falsified the pre-registered positive historical lexical-instability "
    "hypothesis under the frozen Day 26 exposure definition. The all-future "
    "primary contrast was negative, with its 95% bootstrap confidence interval "
    "entirely below zero. Pre-specified secondary analyses were broadly "
    "consistent with this result: F3 remained negative, the F3 positive-to-negative "
    "failure channel showed approximately zero/slightly negative contrast, all "
    "three class-conditioned contrasts were negative, the three largest sources "
    "were negative, df>=50 sensitivity was more strongly negative, and both "
    "unadjusted and adjusted logistic models produced odds ratios below one. "
    "Therefore, historical lexical-label instability as operationalized on Day 26 "
    "does not explain higher future classification error. This result does not "
    "rule out lexical/topic shift more broadly; rather, it separates the Day 26 "
    "historical instability construct from the future failure mechanism observed "
    "in earlier days."
)

# -----------------------------------------------------------------------------
# Final summary JSON
# -----------------------------------------------------------------------------

day27_final_scientific_summary = {
    "day": 27,
    "artifact": "day27_final_scientific_summary",
    "status": "SCIENTIFIC_INTERPRETATION_FROZEN",

    "primary_result": {
        "gate": primary_result[
            "decision_gate"
        ]["gate"],
        "delta_primary": float(
            primary_result[
                "primary_estimand"
            ]["delta_primary"]
        ),
        "ci_low": float(
            primary_result[
                "bootstrap"
            ]["ci_low"]
        ),
        "ci_high": float(
            primary_result[
                "bootstrap"
            ]["ci_high"]
        ),
        "standardized_mean_difference": float(
            primary_result[
                "primary_estimand"
            ]["standardized_mean_difference"]
        ),
    },

    "period_heterogeneity": {
        "negative_periods": negative_periods,
        "positive_periods": positive_periods,
        "f3_delta": float(
            f3_result.loc[
                0,
                "delta_error_minus_correct",
            ]
        ),
    },

    "f3_positive_failure_channel": {
        "p_to_n_n": int(
            f3_positive_result.loc[
                0,
                "p_to_n_n",
            ]
        ),
        "p_to_p_n": int(
            f3_positive_result.loc[
                0,
                "p_to_p_n",
            ]
        ),
        "delta_p_to_n_minus_p_to_p": float(
            f3_positive_result.loc[
                0,
                "delta_p_to_n_minus_p_to_p",
            ]
        ),
    },

    "class_conditioned": {
        row["group"]: float(
            row["delta_error_minus_correct"]
        )
        for _, row in class_result.iterrows()
    },

    "major_sources": {
        row["group"]: {
            "n": int(row["n"]),
            "delta_error_minus_correct": float(
                row["delta_error_minus_correct"]
            ),
        }
        for _, row in major_sources.iterrows()
    },

    "df50_sensitivity": {
        "delta": float(
            df50_result[
                "contrast"
            ]["delta_error_minus_correct"]
        ),
        "standardized_mean_difference": float(
            df50_result[
                "contrast"
            ]["standardized_mean_difference"]
        ),
    },

    "logistic_secondary": {
        "unadjusted_or": float(
            logistic_result[
                "unadjusted_model"
            ]["odds_ratio_per_1sd"]
        ),
        "unadjusted_or_ci": [
            float(
                logistic_result[
                    "unadjusted_model"
                ]["odds_ratio_ci_low"]
            ),
            float(
                logistic_result[
                    "unadjusted_model"
                ]["odds_ratio_ci_high"]
            ),
        ],
        "adjusted_or": float(
            logistic_result[
                "adjusted_model"
            ]["odds_ratio_per_1sd"]
        ),
        "adjusted_or_ci": [
            float(
                logistic_result[
                    "adjusted_model"
                ]["odds_ratio_ci_low"]
            ),
            float(
                logistic_result[
                    "adjusted_model"
                ]["odds_ratio_ci_high"]
            ),
        ],
    },

    "scientific_interpretation": {
        "H1_supported": False,
        "primary_gate": "GATE_C_FAILED",
        "historical_instability_explains_future_error": False,
        "lexical_topic_shift_broadly_ruled_out": False,
        "causal_claim_permitted": False,
        "secondary_results_may_override_primary": False,
        "conclusion": final_conclusion,
    },

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

save_json(
    day27_final_scientific_summary,
    DAY27_FINAL_SUMMARY_JSON,
)

with open(
    DAY27_CONCLUSION_TXT,
    "w",
    encoding="utf-8",
) as f:
    f.write(
        final_conclusion
        + "\n"
    )

# -----------------------------------------------------------------------------
# Reload / artifact checks
# -----------------------------------------------------------------------------

summary_reload = load_json(
    DAY27_FINAL_SUMMARY_JSON
)

evidence_reload = pd.read_csv(
    DAY27_EVIDENCE_TABLE_CSV
)

with open(
    DAY27_CONCLUSION_TXT,
    "r",
    encoding="utf-8",
) as f:
    conclusion_reload = f.read().strip()

checks_stage6["evidence_table_exists"] = print_check(
    "evidence_table_exists",
    DAY27_EVIDENCE_TABLE_CSV.exists()
)

checks_stage6["evidence_table_nonempty"] = print_check(
    "evidence_table_nonempty",
    DAY27_EVIDENCE_TABLE_CSV.stat().st_size > 0
)

checks_stage6["evidence_table_rows_6"] = print_check(
    "evidence_table_rows_6",
    len(evidence_reload) == 6
)

checks_stage6["final_summary_exists"] = print_check(
    "final_summary_exists",
    DAY27_FINAL_SUMMARY_JSON.exists()
)

checks_stage6["final_summary_nonempty"] = print_check(
    "final_summary_nonempty",
    DAY27_FINAL_SUMMARY_JSON.stat().st_size > 0
)

checks_stage6["final_summary_roundtrip_exact"] = print_check(
    "final_summary_roundtrip_exact",
    summary_reload
    == day27_final_scientific_summary
)

checks_stage6["conclusion_exists"] = print_check(
    "conclusion_exists",
    DAY27_CONCLUSION_TXT.exists()
)

checks_stage6["conclusion_nonempty"] = print_check(
    "conclusion_nonempty",
    DAY27_CONCLUSION_TXT.stat().st_size > 0
)

checks_stage6["conclusion_roundtrip_exact"] = print_check(
    "conclusion_roundtrip_exact",
    conclusion_reload
    == final_conclusion
)

# -----------------------------------------------------------------------------
# Artifact hashes
# -----------------------------------------------------------------------------

day27_evidence_table_sha256 = sha256_file(
    DAY27_EVIDENCE_TABLE_CSV
)

day27_final_summary_sha256 = sha256_file(
    DAY27_FINAL_SUMMARY_JSON
)

day27_conclusion_sha256 = sha256_file(
    DAY27_CONCLUSION_TXT
)

# -----------------------------------------------------------------------------
# Output
# -----------------------------------------------------------------------------

print("-" * 120)
print("DAY 27 INTEGRATED EVIDENCE")
print("-" * 120)

print(
    day27_evidence_table.to_string(
        index=False,
    )
)

print("-" * 120)
print("FINAL SCIENTIFIC INTERPRETATION")
print("-" * 120)

print(final_conclusion)

print("-" * 120)
print("STAGE 6 ARTIFACT IDENTITIES")
print("-" * 120)

print(
    f"Evidence table SHA256 : "
    f"{day27_evidence_table_sha256}"
)

print(
    f"Final summary SHA256  : "
    f"{day27_final_summary_sha256}"
)

print(
    f"Conclusion TXT SHA256 : "
    f"{day27_conclusion_sha256}"
)

# -----------------------------------------------------------------------------
# Final Stage 6 gate
# -----------------------------------------------------------------------------

all_stage6_checks_pass = all(
    checks_stage6.values()
)

print("-" * 120)

print_check(
    "DAY27_STAGE6_ALL_PASS",
    all_stage6_checks_pass
)

if not all_stage6_checks_pass:
    raise RuntimeError(
        "DAY 27 INTEGRITY STOP: evidence integration failed."
    )

print("=" * 120)
print("DAY 27 STAGE 6: PASS — SCIENTIFIC INTERPRETATION FROZEN")
print("PRIMARY H1: NOT SUPPORTED")
print("PRIMARY GATE: GATE_C_FAILED")
print("HISTORICAL LEXICAL-INSTABILITY MECHANISM: NOT VALIDATED OUT-OF-TIME")
print("READY — STAGE 7 SCIENTIFIC CLOSURE / COMPLETION CHECKPOINT")
print("=" * 120)

DAY 27 — STAGE 6 EVIDENCE INTEGRATION / INTERPRETATION GATE
PASS   — primary_gate_gate_c_failed
PASS   — primary_delta_negative
PASS   — primary_ci_entirely_below_zero
PASS   — df50_secondary_negative
PASS   — unadjusted_logistic_or_below_1
PASS   — adjusted_logistic_or_below_1
PASS   — all_three_class_deltas_negative
PASS   — three_largest_sources_negative
PASS   — evidence_table_exists
PASS   — evidence_table_nonempty
PASS   — evidence_table_rows_6
PASS   — final_summary_exists
PASS   — final_summary_nonempty
PASS   — final_summary_roundtrip_exact
PASS   — conclusion_exists
PASS   — conclusion_nonempty
PASS   — conclusion_roundtrip_exact
------------------------------------------------------------------------------------------------------------------------
DAY 27 INTEGRATED EVIDENCE
------------------------------------------------------------------------------------------------------------------------
            evidence_domain                     analysis  estimate    ci_low   ci_h

## Stage 7 — Scientific Closure and Completion Checkpoint

Day 27 scientific analysis is complete.

The pre-registered primary hypothesis was not supported:

\[
\Delta_{\mathrm{primary}}
=
-0.005388,
\qquad
95\%\,CI
=
[-0.010077,\,-0.000992].
\]

Therefore,

\[
\boxed{\text{Primary Gate = GATE C — FAILED}}.
\]

The frozen Day 26 historical lexical-instability predictor did not predict higher future classification error under the Day 27 primary exposure definition.

Pre-specified secondary analyses were broadly consistent with this result and did not provide a valid basis for rescuing the primary hypothesis.

### Final interpretation

The Day 27 result falsifies the specific mechanism:

\[
\text{historical lexical-label instability}
\rightarrow
\text{higher future classification error}
\]

under the frozen Day 26 operationalization.

It does **not** rule out lexical or topic shift more broadly.

No further subgroup analysis, exposure redefinition, model retraining, or post-hoc hypothesis rescue is permitted within Day 27.

This final stage performs only:

- artifact completeness audit;
- frozen-hash verification;
- completion checkpoint creation;
- canonical notebook save;
- canonical notebook integrity validation.

After this stage, Day 27 is closed.

In [26]:
# =============================================================================
# Day 27 — Stage 7
# Scientific closure and provenance manifest
# Canonical retained cell
# =============================================================================

import os
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

print("=" * 120)
print("DAY 27 — STAGE 7 SCIENTIFIC CLOSURE / COMPLETION CHECKPOINT")
print("=" * 120)

# =============================================================================
# PART A — REQUIRED ARTIFACT AUDIT
# =============================================================================

required_artifacts = {
    "analysis_manifest": DAY27_ANALYSIS_MANIFEST,
    "future_population_manifest": DAY27_FUTURE_POPULATION_MANIFEST,
    "future_exposure": DAY27_EXPOSURE_CSV,
    "future_exposure_qc": DAY27_EXPOSURE_QC_CSV,
    "primary_point_estimate": DAY27_PRIMARY_POINT_ESTIMATE_JSON,
    "primary_hypothesis_test": DAY27_PRIMARY_TEST_JSON,
    "primary_bootstrap_replicates": DAY27_BOOTSTRAP_CSV,
    "period_exposure_contrasts": DAY27_PERIOD_CONTRASTS_CSV,
    "f3_exposure_contrast": DAY27_F3_CONTRAST_CSV,
    "f3_positive_contrast": DAY27_F3_POSITIVE_CSV,
    "within_source_contrasts": DAY27_WITHIN_SOURCE_CSV,
    "class_conditioned_contrasts": DAY27_CLASS_CONDITIONED_CSV,
    "df50_sensitivity": DAY27_DF50_SENSITIVITY_JSON,
    "logistic_secondary": DAY27_LOGISTIC_SECONDARY_JSON,
    "evidence_table": DAY27_EVIDENCE_TABLE_CSV,
    "final_scientific_summary": DAY27_FINAL_SUMMARY_JSON,
    "scientific_conclusion": DAY27_CONCLUSION_TXT,
}

audit_rows = []

for artifact_name, artifact_path in required_artifacts.items():

    artifact_path = Path(artifact_path)

    exists = artifact_path.exists()

    nonempty = (
        exists
        and artifact_path.stat().st_size > 0
    )

    sha256_value = (
        sha256_file(artifact_path)
        if nonempty
        else None
    )

    audit_rows.append({
        "artifact": artifact_name,
        "path": str(artifact_path),
        "exists": exists,
        "nonempty": nonempty,
        "size_bytes": (
            artifact_path.stat().st_size
            if exists
            else None
        ),
        "sha256": sha256_value,
        "pass": bool(
            exists and nonempty
        ),
    })

final_artifact_audit_df = pd.DataFrame(
    audit_rows
)

final_artifact_audit_df.to_csv(
    DAY27_FINAL_AUDIT_CSV,
    index=False,
)

# =============================================================================
# PART B — CORE HASH STABILITY
# =============================================================================

print("-" * 120)
print("FINAL CORE HASH AUDIT")
print("-" * 120)

closure_checks = {}

closure_checks["day24_prediction_hash_unchanged"] = print_check(
    "day24_prediction_hash_unchanged",
    sha256_file(
        DAY24_FUTURE_PREDICTIONS
    )
    == day27_analysis_manifest[
        "future_population"
    ]["prediction_sha256"]
)

closure_checks["day26_token_rank_hash_unchanged"] = print_check(
    "day26_token_rank_hash_unchanged",
    sha256_file(
        DAY26_TOKEN_RANKING
    )
    == day27_analysis_manifest[
        "day26_predictor"
    ]["token_rank_sha256"]
)

closure_checks["day26_handoff_hash_unchanged"] = print_check(
    "day26_handoff_hash_unchanged",
    sha256_file(
        DAY26_HANDOFF_MANIFEST
    )
    == day27_analysis_manifest[
        "day26_predictor"
    ]["handoff_sha256"]
)

closure_checks["day27_exposure_hash_unchanged"] = print_check(
    "day27_exposure_hash_unchanged",
    sha256_file(
        DAY27_EXPOSURE_CSV
    )
    == day27_frozen_exposure_identity[
        "sha256"
    ]
)

closure_checks["primary_test_hash_unchanged"] = print_check(
    "primary_test_hash_unchanged",
    sha256_file(
        DAY27_PRIMARY_TEST_JSON
    )
    == day27_primary_test_sha256
)

closure_checks["bootstrap_hash_unchanged"] = print_check(
    "bootstrap_hash_unchanged",
    sha256_file(
        DAY27_BOOTSTRAP_CSV
    )
    == day27_bootstrap_sha256
)

# =============================================================================
# PART C — FINAL SCIENTIFIC STATUS
# =============================================================================

final_primary_result = load_json(
    DAY27_PRIMARY_TEST_JSON
)

final_summary_result = load_json(
    DAY27_FINAL_SUMMARY_JSON
)

closure_checks["primary_gate_frozen_gate_c_failed"] = print_check(
    "primary_gate_frozen_gate_c_failed",
    final_primary_result[
        "decision_gate"
    ]["gate"] == "GATE_C_FAILED"
)

closure_checks["H1_not_supported"] = print_check(
    "H1_not_supported",
    final_summary_result[
        "scientific_interpretation"
    ]["H1_supported"] is False
)

closure_checks["historical_instability_not_validated"] = print_check(
    "historical_instability_not_validated",
    final_summary_result[
        "scientific_interpretation"
    ][
        "historical_instability_explains_future_error"
    ] is False
)

closure_checks["lexical_topic_shift_not_broadly_ruled_out"] = print_check(
    "lexical_topic_shift_not_broadly_ruled_out",
    final_summary_result[
        "scientific_interpretation"
    ][
        "lexical_topic_shift_broadly_ruled_out"
    ] is False
)

closure_checks["no_causal_claim"] = print_check(
    "no_causal_claim",
    final_summary_result[
        "scientific_interpretation"
    ][
        "causal_claim_permitted"
    ] is False
)

# =============================================================================
# PART D — REQUIRED ARTIFACT STATUS
# =============================================================================

all_artifacts_exist = bool(
    final_artifact_audit_df[
        "exists"
    ].all()
)

all_artifacts_nonempty = bool(
    final_artifact_audit_df[
        "nonempty"
    ].all()
)

all_artifacts_pass = bool(
    final_artifact_audit_df[
        "pass"
    ].all()
)

closure_checks["all_required_artifacts_exist"] = print_check(
    "all_required_artifacts_exist",
    all_artifacts_exist
)

closure_checks["all_required_artifacts_nonempty"] = print_check(
    "all_required_artifacts_nonempty",
    all_artifacts_nonempty
)

closure_checks["all_required_artifacts_pass"] = print_check(
    "all_required_artifacts_pass",
    all_artifacts_pass
)

closure_checks["final_audit_artifact_exists"] = print_check(
    "final_audit_artifact_exists",
    DAY27_FINAL_AUDIT_CSV.exists()
)

closure_checks["final_audit_artifact_nonempty"] = print_check(
    "final_audit_artifact_nonempty",
    DAY27_FINAL_AUDIT_CSV.stat().st_size > 0
)

# =============================================================================
# PART E — CREATE COMPLETION CHECKPOINT
# =============================================================================

day27_completion_checkpoint = {
    "day": 27,

    "artifact": "day27_completion_checkpoint",

    "status": "PASS",

    "scientific_status": (
        "OUT_OF_TIME_LEXICAL_INSTABILITY_HYPOTHESIS_TEST_COMPLETE"
    ),

    "day27_complete": True,

    "primary_hypothesis": {
        "H1_supported": False,
        "gate": "GATE_C_FAILED",
        "delta_primary": float(
            final_primary_result[
                "primary_estimand"
            ]["delta_primary"]
        ),
        "ci_low": float(
            final_primary_result[
                "bootstrap"
            ]["ci_low"]
        ),
        "ci_high": float(
            final_primary_result[
                "bootstrap"
            ]["ci_high"]
        ),
    },

    "scientific_conclusion": (
        final_summary_result[
            "scientific_interpretation"
        ]["conclusion"]
    ),

    "scientific_boundary": {
        "historical_lexical_instability_validated": False,
        "lexical_topic_shift_broadly_ruled_out": False,
        "causal_claim_permitted": False,
        "posthoc_primary_rescue_permitted": False,
    },

    "core_artifact_hashes": {
        "analysis_manifest": sha256_file(
            DAY27_ANALYSIS_MANIFEST
        ),
        "future_population_manifest": sha256_file(
            DAY27_FUTURE_POPULATION_MANIFEST
        ),
        "future_exposure": sha256_file(
            DAY27_EXPOSURE_CSV
        ),
        "future_exposure_qc": sha256_file(
            DAY27_EXPOSURE_QC_CSV
        ),
        "primary_point_estimate": sha256_file(
            DAY27_PRIMARY_POINT_ESTIMATE_JSON
        ),
        "primary_test": sha256_file(
            DAY27_PRIMARY_TEST_JSON
        ),
        "bootstrap_replicates": sha256_file(
            DAY27_BOOTSTRAP_CSV
        ),
        "period_contrasts": sha256_file(
            DAY27_PERIOD_CONTRASTS_CSV
        ),
        "f3_contrast": sha256_file(
            DAY27_F3_CONTRAST_CSV
        ),
        "f3_positive_contrast": sha256_file(
            DAY27_F3_POSITIVE_CSV
        ),
        "within_source": sha256_file(
            DAY27_WITHIN_SOURCE_CSV
        ),
        "class_conditioned": sha256_file(
            DAY27_CLASS_CONDITIONED_CSV
        ),
        "df50_sensitivity": sha256_file(
            DAY27_DF50_SENSITIVITY_JSON
        ),
        "logistic_secondary": sha256_file(
            DAY27_LOGISTIC_SECONDARY_JSON
        ),
        "evidence_table": sha256_file(
            DAY27_EVIDENCE_TABLE_CSV
        ),
        "final_scientific_summary": sha256_file(
            DAY27_FINAL_SUMMARY_JSON
        ),
        "scientific_conclusion": sha256_file(
            DAY27_CONCLUSION_TXT
        ),
        "final_artifact_audit": sha256_file(
            DAY27_FINAL_AUDIT_CSV
        ),
    },

    "upstream_identity": {
        "day24_future_prediction_sha256": sha256_file(
            DAY24_FUTURE_PREDICTIONS
        ),
        "day26_token_rank_sha256": sha256_file(
            DAY26_TOKEN_RANKING
        ),
        "day26_handoff_sha256": sha256_file(
            DAY26_HANDOFF_MANIFEST
        ),
    },

    "next_action": (
        "DAY27_CLOSED_DECIDE_NEXT_RESEARCH_DIRECTION"
    ),

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

save_json(
    day27_completion_checkpoint,
    DAY27_COMPLETION_CHECKPOINT
)

completion_reload = load_json(
    DAY27_COMPLETION_CHECKPOINT
)

closure_checks["completion_checkpoint_exists"] = print_check(
    "completion_checkpoint_exists",
    DAY27_COMPLETION_CHECKPOINT.exists()
)

closure_checks["completion_checkpoint_nonempty"] = print_check(
    "completion_checkpoint_nonempty",
    DAY27_COMPLETION_CHECKPOINT.stat().st_size > 0
)

closure_checks["completion_checkpoint_roundtrip_exact"] = print_check(
    "completion_checkpoint_roundtrip_exact",
    completion_reload
    == day27_completion_checkpoint
)

closure_checks["completion_status_pass"] = print_check(
    "completion_status_pass",
    completion_reload[
        "status"
    ] == "PASS"
)

closure_checks["completion_day27_complete_true"] = print_check(
    "completion_day27_complete_true",
    completion_reload[
        "day27_complete"
    ] is True
)


DAY 27 — STAGE 7 SCIENTIFIC CLOSURE / COMPLETION CHECKPOINT
------------------------------------------------------------------------------------------------------------------------
FINAL CORE HASH AUDIT
------------------------------------------------------------------------------------------------------------------------
PASS   — day24_prediction_hash_unchanged
PASS   — day26_token_rank_hash_unchanged
PASS   — day26_handoff_hash_unchanged
PASS   — day27_exposure_hash_unchanged
PASS   — primary_test_hash_unchanged
PASS   — bootstrap_hash_unchanged
PASS   — primary_gate_frozen_gate_c_failed
PASS   — H1_not_supported
PASS   — historical_instability_not_validated
PASS   — lexical_topic_shift_not_broadly_ruled_out
PASS   — no_causal_claim
PASS   — all_required_artifacts_exist
PASS   — all_required_artifacts_nonempty
PASS   — all_required_artifacts_pass
PASS   — final_audit_artifact_exists
PASS   — final_audit_artifact_nonempty
PASS   — completion_checkpoint_exists
PASS   — completion_check

RuntimeError: DAY 27 FINAL CLOSURE FAILED. Do not declare Day27 complete until all checks pass.